In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

cleaned_path = (
    project_root
    / "data"
    / "processed"
    / "milk_yield_cleaned_v1.csv"
)

# Inspect available columns before loading the full dataset
header = pd.read_csv(cleaned_path, nrows=0)

print("Available columns:")
for column in header.columns:
    print(column)

Available columns:
source_row
cow_id_raw
cow_id
lactation_number_raw
lactation_number
days_in_milk_raw
days_in_milk
reproduction_status_raw
reproduction_status
event_date_raw
event_date
avg_milk_flow_raw
avg_milk_flow
flow_30_60_session_raw
flow_30_60_session
yield_first_2_min_session_raw
yield_first_2_min_session
milk_yield_raw
milk_yield
duration_session_sec_raw
duration_session_sec
milking_raw
milking
lactation_number_invalid
days_in_milk_invalid
avg_milk_flow_invalid
flow_30_60_session_invalid
yield_first_2_min_session_invalid
milk_yield_invalid
duration_session_sec_invalid
milking_invalid
event_date_invalid
cow_id_missing
milk_yield_missing


In [2]:
context_columns = [
    "source_row",
    "cow_id",
    "lactation_number",
    "days_in_milk",
    "reproduction_status",
    "event_date",
    "avg_milk_flow",
    "flow_30_60_session",
    "yield_first_2_min_session",
    "milk_yield",
    "duration_session_sec",
    "milking"
]

print("Loading context-modeling data...")

context_data = pd.read_csv(
    cleaned_path,
    usecols=context_columns,
    dtype={
        "cow_id": "string",
        "reproduction_status": "string",
        "milking": "string"
    },
    low_memory=False
)

# Convert the date into usable model features
context_data["event_date"] = pd.to_datetime(
    context_data["event_date"],
    errors="coerce"
)

context_data["event_year"] = (
    context_data["event_date"].dt.year
)

context_data["event_month"] = (
    context_data["event_date"].dt.month
)

context_data["event_day_of_year"] = (
    context_data["event_date"].dt.dayofyear
)

context_data["event_day_of_week"] = (
    context_data["event_date"].dt.dayofweek
)

context_targets = [
    "cow_id",
    "lactation_number",
    "days_in_milk",
    "reproduction_status"
]

# Rows with real target values for model development
known_context = context_data[
    context_data[context_targets]
    .notna()
    .all(axis=1)
].copy()

# Rows intentionally masked by the professor
context_to_predict = context_data[
    context_data[context_targets]
    .isna()
    .any(axis=1)
].copy()

print("Complete dataset rows:", len(context_data))
print("Known-context rows:", len(known_context))
print(
    "Rows requiring one or more context predictions:",
    len(context_to_predict)
)

print("\nMissing targets in prediction data:")
print(
    context_to_predict[
        context_targets
    ].isna().sum()
)

Loading context-modeling data...
Complete dataset rows: 4688514
Known-context rows: 3748763
Rows requiring one or more context predictions: 939751

Missing targets in prediction data:
cow_id                 939749
lactation_number       939749
days_in_milk           939751
reproduction_status    939749
dtype: int64


In [3]:
import gc

from sklearn.model_selection import GroupShuffleSplit

# Free the redundant full copy; known_context and
# context_to_predict already contain all required rows.
if "context_data" in globals():
    del context_data
    gc.collect()

# First split: 70% training, 30% temporary
first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_indices, temporary_indices = next(
    first_split.split(
        known_context,
        groups=known_context["cow_id"]
    )
)

temporary_data = known_context.iloc[
    temporary_indices
]

# Divide the temporary cows equally:
# 15% validation and 15% testing
second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

validation_relative, test_relative = next(
    second_split.split(
        temporary_data,
        groups=temporary_data["cow_id"]
    )
)

validation_indices = temporary_indices[
    validation_relative
]

test_indices = temporary_indices[
    test_relative
]

# Verify that no cows overlap
train_cows = set(
    known_context.iloc[train_indices]["cow_id"]
)

validation_cows = set(
    known_context.iloc[validation_indices]["cow_id"]
)

test_cows = set(
    known_context.iloc[test_indices]["cow_id"]
)

assert train_cows.isdisjoint(validation_cows)
assert train_cows.isdisjoint(test_cows)
assert validation_cows.isdisjoint(test_cows)

print("Training rows:", len(train_indices))
print("Validation rows:", len(validation_indices))
print("Testing rows:", len(test_indices))

print("\nUnique cows:")
print("Training:", len(train_cows))
print("Validation:", len(validation_cows))
print("Testing:", len(test_cows))

print("\nCow overlap checks passed.")

Training rows: 2606800
Validation rows: 568266
Testing rows: 573697

Unique cows:
Training: 6356
Validation: 1362
Testing: 1363

Cow overlap checks passed.


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

numeric_features = [
    "avg_milk_flow",
    "flow_30_60_session",
    "yield_first_2_min_session",
    "milk_yield",
    "duration_session_sec",
    "event_year",
    "event_month",
    "event_day_of_year",
    "event_day_of_week"
]

categorical_features = [
    "milking"
]

feature_columns = (
    numeric_features
    + categorical_features
)

numeric_processor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_processor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
    )
])

context_preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_processor,
        numeric_features
    ),
    (
        "categorical",
        categorical_processor,
        categorical_features
    )
])

lactation_model = Pipeline([
    (
        "preprocessor",
        context_preprocessor
    ),
    (
        "model",
        HistGradientBoostingClassifier(
            learning_rate=0.1,
            max_iter=100,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42
        )
    )
])

X_lactation_train = known_context.iloc[
    train_indices
][feature_columns]

y_lactation_train = (
    known_context.iloc[train_indices][
        "lactation_number"
    ]
    .astype("int64")
)

X_lactation_validation = known_context.iloc[
    validation_indices
][feature_columns]

y_lactation_validation = (
    known_context.iloc[validation_indices][
        "lactation_number"
    ]
    .astype("int64")
)

# Majority-class baseline
majority_lactation = (
    y_lactation_train
    .mode()
    .iloc[0]
)

baseline_predictions = np.full(
    len(y_lactation_validation),
    majority_lactation
)

baseline_accuracy = accuracy_score(
    y_lactation_validation,
    baseline_predictions
)

print(
    "Majority-class baseline accuracy:",
    round(baseline_accuracy, 4)
)

print("Fitting lactation-number model...")

lactation_model.fit(
    X_lactation_train,
    y_lactation_train
)

lactation_validation_predictions = (
    lactation_model.predict(
        X_lactation_validation
    )
)

lactation_accuracy = accuracy_score(
    y_lactation_validation,
    lactation_validation_predictions
)

lactation_balanced_accuracy = (
    balanced_accuracy_score(
        y_lactation_validation,
        lactation_validation_predictions
    )
)

lactation_macro_f1 = f1_score(
    y_lactation_validation,
    lactation_validation_predictions,
    average="macro"
)

print(
    "Validation accuracy:",
    round(lactation_accuracy, 4)
)

print(
    "Validation balanced accuracy:",
    round(lactation_balanced_accuracy, 4)
)

print(
    "Validation macro F1:",
    round(lactation_macro_f1, 4)
)

Majority-class baseline accuracy: 0.3555
Fitting lactation-number model...
Validation accuracy: 0.4674
Validation balanced accuracy: 0.1614
Validation macro F1: 0.114


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [5]:
# Create a split that simulates the professor randomly
# masking records from cows that remain elsewhere in the data.

rng = np.random.default_rng(42)
random_values = rng.random(len(known_context))

train_indices = np.flatnonzero(
    random_values < 0.70
)

validation_indices = np.flatnonzero(
    (random_values >= 0.70)
    & (random_values < 0.85)
)

test_indices = np.flatnonzero(
    random_values >= 0.85
)

train_cows = set(
    known_context.iloc[train_indices]["cow_id"]
)

validation_cows = set(
    known_context.iloc[validation_indices]["cow_id"]
)

test_cows = set(
    known_context.iloc[test_indices]["cow_id"]
)

validation_cows_missing_from_training = (
    validation_cows - train_cows
)

test_cows_missing_from_training = (
    test_cows - train_cows
)

print("Training rows:", len(train_indices))
print("Validation rows:", len(validation_indices))
print("Testing rows:", len(test_indices))

print(
    "Validation cows absent from training:",
    len(validation_cows_missing_from_training)
)

print(
    "Test cows absent from training:",
    len(test_cows_missing_from_training)
)

Training rows: 2624879
Validation rows: 562175
Testing rows: 561709
Validation cows absent from training: 3
Test cows absent from training: 8


In [6]:
# Move rows belonging to cows absent from training
# into the training set.

validation_move_mask = (
    known_context
    .iloc[validation_indices]["cow_id"]
    .isin(validation_cows_missing_from_training)
    .to_numpy()
)

test_move_mask = (
    known_context
    .iloc[test_indices]["cow_id"]
    .isin(test_cows_missing_from_training)
    .to_numpy()
)

validation_rows_to_move = validation_indices[
    validation_move_mask
]

test_rows_to_move = test_indices[
    test_move_mask
]

train_indices = np.concatenate([
    train_indices,
    validation_rows_to_move,
    test_rows_to_move
])

validation_indices = validation_indices[
    ~validation_move_mask
]

test_indices = test_indices[
    ~test_move_mask
]

# Recheck cow coverage
train_cows = set(
    known_context.iloc[train_indices]["cow_id"]
)

validation_cows = set(
    known_context.iloc[validation_indices]["cow_id"]
)

test_cows = set(
    known_context.iloc[test_indices]["cow_id"]
)

assert validation_cows.issubset(train_cows)
assert test_cows.issubset(train_cows)

print("Corrected training rows:", len(train_indices))
print("Corrected validation rows:", len(validation_indices))
print("Corrected testing rows:", len(test_indices))

print(
    "Validation cows absent from training:",
    len(validation_cows - train_cows)
)

print(
    "Test cows absent from training:",
    len(test_cows - train_cows)
)

Corrected training rows: 2624890
Corrected validation rows: 562172
Corrected testing rows: 561701
Validation cows absent from training: 0
Test cows absent from training: 0


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

numeric_features = [
    "avg_milk_flow",
    "flow_30_60_session",
    "yield_first_2_min_session",
    "milk_yield",
    "duration_session_sec",
    "event_year",
    "event_month",
    "event_day_of_year",
    "event_day_of_week"
]

categorical_features = [
    "milking"
]

feature_columns = (
    numeric_features
    + categorical_features
)

numeric_processor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_processor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
    )
])

context_preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_processor,
        numeric_features
    ),
    (
        "categorical",
        categorical_processor,
        categorical_features
    )
])

lactation_model = Pipeline([
    (
        "preprocessor",
        context_preprocessor
    ),
    (
        "model",
        HistGradientBoostingClassifier(
            learning_rate=0.1,
            max_iter=100,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42
        )
    )
])

X_lactation_train = known_context.iloc[
    train_indices
][feature_columns]

y_lactation_train = (
    known_context.iloc[train_indices][
        "lactation_number"
    ]
    .astype("int64")
)

X_lactation_validation = known_context.iloc[
    validation_indices
][feature_columns]

y_lactation_validation = (
    known_context.iloc[validation_indices][
        "lactation_number"
    ]
    .astype("int64")
)

# Majority-class baseline
majority_lactation = (
    y_lactation_train
    .mode()
    .iloc[0]
)

baseline_predictions = np.full(
    len(y_lactation_validation),
    majority_lactation
)

baseline_accuracy = accuracy_score(
    y_lactation_validation,
    baseline_predictions
)

print(
    "Majority-class baseline accuracy:",
    round(baseline_accuracy, 4)
)

print("Fitting lactation-number model...")

lactation_model.fit(
    X_lactation_train,
    y_lactation_train
)

lactation_validation_predictions = (
    lactation_model.predict(
        X_lactation_validation
    )
)

lactation_accuracy = accuracy_score(
    y_lactation_validation,
    lactation_validation_predictions
)

lactation_balanced_accuracy = (
    balanced_accuracy_score(
        y_lactation_validation,
        lactation_validation_predictions
    )
)

lactation_macro_f1 = f1_score(
    y_lactation_validation,
    lactation_validation_predictions,
    average="macro"
)

print(
    "Validation accuracy:",
    round(lactation_accuracy, 4)
)

print(
    "Validation balanced accuracy:",
    round(lactation_balanced_accuracy, 4)
)

print(
    "Validation macro F1:",
    round(lactation_macro_f1, 4)
)

Majority-class baseline accuracy: 0.3671
Fitting lactation-number model...
Validation accuracy: 0.4684
Validation balanced accuracy: 0.1313
Validation macro F1: 0.1224


In [8]:
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler

profile_features = [
    "avg_milk_flow",
    "flow_30_60_session",
    "yield_first_2_min_session",
    "milk_yield",
    "duration_session_sec"
]

# Use records from the simulated-mask training set
profile_training = known_context.iloc[
    train_indices
][
    [
        "cow_id",
        "event_date",
        "milking"
    ]
    + profile_features
].copy()

# Represent time by calendar week
profile_training["event_week"] = (
    profile_training["event_date"]
    .dt.to_period("W")
    .dt.start_time
)

# Impute the very small number of missing measurements
profile_medians = (
    profile_training[
        profile_features
    ].median()
)

profile_training[
    profile_features
] = (
    profile_training[
        profile_features
    ].fillna(profile_medians)
)

# Average each cow's measurements within each week/milking
weekly_profiles = (
    profile_training
    .groupby(
        [
            "event_week",
            "milking",
            "cow_id"
        ],
        observed=True
    )[profile_features]
    .mean()
    .reset_index()
)

# Scale measurements so large-valued variables do not dominate
profile_scaler = StandardScaler()

profile_scaler.fit(
    profile_training[
        profile_features
    ]
)

print("Weekly cow profiles created:", len(weekly_profiles))



Weekly cow profiles created: 1256281


In [9]:
# Test cow identification on 50,000 known validation records
pilot_size = min(
    50_000,
    len(validation_indices)
)

pilot_indices = np.random.default_rng(42).choice(
    validation_indices,
    size=pilot_size,
    replace=False
)

pilot = known_context.iloc[
    pilot_indices
][
    [
        "cow_id",
        "event_date",
        "milking"
    ]
    + profile_features
].copy()

pilot["event_week"] = (
    pilot["event_date"]
    .dt.to_period("W")
    .dt.start_time
)

pilot[
    profile_features
] = (
    pilot[
        profile_features
    ].fillna(profile_medians)
)

pilot["predicted_cow_id"] = pd.NA
pilot["match_distance"] = np.nan

# Create quick access to profiles for each week and milking
profile_groups = {
    key: group.reset_index(drop=True)
    for key, group in weekly_profiles.groupby(
        ["event_week", "milking"],
        observed=True
    )
}

for key, query_group in pilot.groupby(
    ["event_week", "milking"],
    observed=True
):
    candidates = profile_groups.get(key)

    if candidates is None or len(candidates) == 0:
        continue

    candidate_values = profile_scaler.transform(
        candidates[profile_features]
    )

    query_values = profile_scaler.transform(
        query_group[profile_features]
    )

    tree = cKDTree(candidate_values)

    distances, nearest_positions = tree.query(
        query_values,
        k=1
    )

    pilot.loc[
        query_group.index,
        "predicted_cow_id"
    ] = (
        candidates.iloc[
            nearest_positions
        ]["cow_id"]
        .astype("string")
        .to_numpy()
    )

    pilot.loc[
        query_group.index,
        "match_distance"
    ] = distances

matched_pilot = pilot[
    pilot["predicted_cow_id"].notna()
].copy()

cow_id_accuracy = (
    matched_pilot["predicted_cow_id"]
    .eq(matched_pilot["cow_id"])
    .mean()
)

print("Pilot records:", len(pilot))
print("Records with an available weekly profile:", len(matched_pilot))
print("Cow-ID exact-match accuracy:", round(cow_id_accuracy, 4))

Pilot records: 50000
Records with an available weekly profile: 50000
Cow-ID exact-match accuracy: 0.0234


In [10]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

X_days_train = known_context.iloc[
    train_indices
][feature_columns]

y_days_train = known_context.iloc[
    train_indices
]["days_in_milk"]

X_days_validation = known_context.iloc[
    validation_indices
][feature_columns]

y_days_validation = known_context.iloc[
    validation_indices
]["days_in_milk"]

# Median baseline
days_baseline_value = y_days_train.median()

days_baseline_predictions = np.full(
    len(y_days_validation),
    days_baseline_value
)

days_baseline_mae = mean_absolute_error(
    y_days_validation,
    days_baseline_predictions
)

print(
    "Days-in-milk baseline MAE:",
    round(days_baseline_mae, 2)
)

days_model = Pipeline([
    (
        "preprocessor",
        clone(context_preprocessor)
    ),
    (
        "model",
        HistGradientBoostingRegressor(
            learning_rate=0.1,
            max_iter=100,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42
        )
    )
])

print("Fitting days-in-milk model...")

days_model.fit(
    X_days_train,
    y_days_train
)

days_validation_predictions = days_model.predict(
    X_days_validation
)

days_mae = mean_absolute_error(
    y_days_validation,
    days_validation_predictions
)

days_rmse = np.sqrt(
    mean_squared_error(
        y_days_validation,
        days_validation_predictions
    )
)

days_r2 = r2_score(
    y_days_validation,
    days_validation_predictions
)

absolute_days_error = np.abs(
    y_days_validation.to_numpy()
    - days_validation_predictions
)

within_7_days = (
    absolute_days_error <= 7
).mean()

within_14_days = (
    absolute_days_error <= 14
).mean()

within_30_days = (
    absolute_days_error <= 30
).mean()

print("Validation MAE:", round(days_mae, 2))
print("Validation RMSE:", round(days_rmse, 2))
print("Validation R²:", round(days_r2, 4))
print("Within 7 days:", round(within_7_days * 100, 2), "%")
print("Within 14 days:", round(within_14_days * 100, 2), "%")
print("Within 30 days:", round(within_30_days * 100, 2), "%")

Days-in-milk baseline MAE: 86.37
Fitting days-in-milk model...
Validation MAE: 71.65
Validation RMSE: 92.18
Validation R²: 0.2378
Within 7 days: 6.19 %
Within 14 days: 12.33 %
Within 30 days: 26.24 %


In [11]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.pipeline import Pipeline

target = "reproduction_status"

X_repro_train = known_context.iloc[train_indices][feature_columns]
y_repro_train = known_context.iloc[train_indices][target].astype("string")

X_repro_validation = known_context.iloc[validation_indices][feature_columns]
y_repro_validation = (
    known_context.iloc[validation_indices][target].astype("string")
)

# Majority-class baseline
majority_status = y_repro_train.mode().iloc[0]
baseline_predictions = np.full(
    len(y_repro_validation),
    majority_status,
    dtype=object,
)

baseline_accuracy = accuracy_score(
    y_repro_validation,
    baseline_predictions,
)

print("Majority status:", majority_status)
print(
    "Majority-class baseline accuracy:",
    round(baseline_accuracy, 4),
)

reproduction_model = Pipeline([
    (
        "preprocessor",
        clone(context_preprocessor),
    ),
    (
        "model",
        HistGradientBoostingClassifier(
            learning_rate=0.1,
            max_iter=100,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])

print("Fitting reproduction-status model...")

reproduction_model.fit(
    X_repro_train,
    y_repro_train,
)

repro_validation_predictions = reproduction_model.predict(
    X_repro_validation
)

repro_accuracy = accuracy_score(
    y_repro_validation,
    repro_validation_predictions,
)

repro_balanced_accuracy = balanced_accuracy_score(
    y_repro_validation,
    repro_validation_predictions,
)

repro_macro_f1 = f1_score(
    y_repro_validation,
    repro_validation_predictions,
    average="macro",
    zero_division=0,
)

print("Validation accuracy:", round(repro_accuracy, 4))
print(
    "Validation balanced accuracy:",
    round(repro_balanced_accuracy, 4),
)
print("Validation macro F1:", round(repro_macro_f1, 4))

Majority status: pregnant
Majority-class baseline accuracy: 0.4948
Fitting reproduction-status model...
Validation accuracy: 0.4529
Validation balanced accuracy: 0.3637
Validation macro F1: 0.3594


In [12]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
)

# ---------------------------------------------------------
# DATE-BASED ORACLE TEST
#
# Question:
# If we knew the cow ID, could dates reconstruct the other
# missing values from that cow's nearby training records?
# ---------------------------------------------------------

reference_columns = [
    "cow_id",
    "event_date",
    "lactation_number",
    "days_in_milk",
    "reproduction_status",
]

reference = (
    known_context.iloc[train_indices][reference_columns]
    .dropna(subset=["cow_id", "event_date"])
    .copy()
)

oracle_validation = (
    known_context.iloc[validation_indices][reference_columns]
    .dropna(subset=["cow_id", "event_date"])
    .copy()
)

# Use matching data types
reference["cow_id"] = reference["cow_id"].astype("string")
oracle_validation["cow_id"] = oracle_validation["cow_id"].astype("string")

reference["event_date"] = pd.to_datetime(
    reference["event_date"],
    errors="coerce",
)

oracle_validation["event_date"] = pd.to_datetime(
    oracle_validation["event_date"],
    errors="coerce",
)

reference = reference.dropna(subset=["event_date"])
oracle_validation = oracle_validation.dropna(subset=["event_date"])

# Preserve the true validation answers
oracle_validation = oracle_validation.rename(
    columns={
        "lactation_number": "true_lactation_number",
        "days_in_milk": "true_days_in_milk",
        "reproduction_status": "true_reproduction_status",
    }
)

# Rename the training record as the temporal anchor
reference = reference.rename(
    columns={
        "event_date": "anchor_date",
        "lactation_number": "anchor_lactation_number",
        "days_in_milk": "anchor_days_in_milk",
        "reproduction_status": "anchor_reproduction_status",
    }
)

# merge_asof requires the date columns to be sorted
reference = reference.sort_values(
    ["anchor_date", "cow_id"]
).reset_index(drop=True)

oracle_validation = oracle_validation.sort_values(
    ["event_date", "cow_id"]
).reset_index(drop=True)

# Find the nearest known training record belonging to the same cow
oracle_results = pd.merge_asof(
    oracle_validation,
    reference,
    left_on="event_date",
    right_on="anchor_date",
    by="cow_id",
    direction="nearest",
    tolerance=pd.Timedelta(days=120),
)

matched = oracle_results["anchor_date"].notna()
oracle_matched = oracle_results.loc[matched].copy()

print("Validation records:", len(oracle_results))
print("Records with a nearby cow-specific anchor:", len(oracle_matched))
print(
    "Anchor coverage:",
    round(matched.mean() * 100, 2),
    "%",
)

# ---------------------------------------------------------
# Reconstruct days in milk
# DIM should increase one day for each elapsed calendar day.
# ---------------------------------------------------------

date_difference = (
    oracle_matched["event_date"]
    - oracle_matched["anchor_date"]
).dt.days

oracle_matched["predicted_days_in_milk"] = (
    oracle_matched["anchor_days_in_milk"]
    + date_difference
)

days_valid = (
    oracle_matched["true_days_in_milk"].notna()
    & oracle_matched["predicted_days_in_milk"].notna()
)

days_error = np.abs(
    oracle_matched.loc[days_valid, "true_days_in_milk"]
    - oracle_matched.loc[days_valid, "predicted_days_in_milk"]
)

print("\nDATE-BASED DAYS-IN-MILK RESULTS")
print(
    "MAE:",
    round(
        mean_absolute_error(
            oracle_matched.loc[days_valid, "true_days_in_milk"],
            oracle_matched.loc[days_valid, "predicted_days_in_milk"],
        ),
        2,
    ),
)

print("Within 7 days:", round((days_error <= 7).mean() * 100, 2), "%")
print("Within 14 days:", round((days_error <= 14).mean() * 100, 2), "%")
print("Within 30 days:", round((days_error <= 30).mean() * 100, 2), "%")

# ---------------------------------------------------------
# Reconstruct lactation number from nearest cow record
# ---------------------------------------------------------

lactation_valid = (
    oracle_matched["true_lactation_number"].notna()
    & oracle_matched["anchor_lactation_number"].notna()
)

lactation_accuracy = accuracy_score(
    oracle_matched.loc[lactation_valid, "true_lactation_number"],
    oracle_matched.loc[lactation_valid, "anchor_lactation_number"],
)

print("\nDATE-BASED LACTATION RESULTS")
print("Accuracy:", round(lactation_accuracy, 4))

# ---------------------------------------------------------
# Reconstruct reproduction status from nearest cow record
# ---------------------------------------------------------

reproduction_valid = (
    oracle_matched["true_reproduction_status"].notna()
    & oracle_matched["anchor_reproduction_status"].notna()
)

true_reproduction = (
    oracle_matched.loc[
        reproduction_valid,
        "true_reproduction_status",
    ]
    .astype("string")
)

predicted_reproduction = (
    oracle_matched.loc[
        reproduction_valid,
        "anchor_reproduction_status",
    ]
    .astype("string")
)

print("\nDATE-BASED REPRODUCTION-STATUS RESULTS")
print(
    "Accuracy:",
    round(
        accuracy_score(
            true_reproduction,
            predicted_reproduction,
        ),
        4,
    ),
)

print(
    "Balanced accuracy:",
    round(
        balanced_accuracy_score(
            true_reproduction,
            predicted_reproduction,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            true_reproduction,
            predicted_reproduction,
            average="macro",
            zero_division=0,
        ),
        4,
    ),
)

Validation records: 562172
Records with a nearby cow-specific anchor: 562172
Anchor coverage: 100.0 %

DATE-BASED DAYS-IN-MILK RESULTS
MAE: 0.06
Within 7 days: 100.0 %
Within 14 days: 100.0 %
Within 30 days: 100.0 %

DATE-BASED LACTATION RESULTS
Accuracy: 1.0

DATE-BASED REPRODUCTION-STATUS RESULTS
Accuracy: 0.992
Balanced accuracy: 0.9873
Macro F1: 0.9868


In [13]:
import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# CONSTRAINED COW-ID RECORD LINKAGE
#
# Use only training rows to create cow profiles.
# Validation cow IDs are treated as hidden.
# ---------------------------------------------------------

link_features = [
    "avg_milk_flow",
    "flow_30_60_session",
    "yield_first_2_min_session",
    "milk_yield",
    "duration_session_sec",
]

reference_link = (
    known_context.iloc[train_indices][
        [
            "cow_id",
            "event_date",
            "milking",
            *link_features,
        ]
    ]
    .dropna(subset=["cow_id", "event_date"])
    .copy()
)

validation_link = (
    known_context.iloc[validation_indices][
        [
            "cow_id",
            "event_date",
            "milking",
            *link_features,
        ]
    ]
    .dropna(subset=["cow_id", "event_date"])
    .copy()
)

reference_link["cow_id"] = (
    reference_link["cow_id"].astype("string")
)

validation_link["cow_id"] = (
    validation_link["cow_id"].astype("string")
)

reference_link["event_date"] = pd.to_datetime(
    reference_link["event_date"],
    errors="coerce",
).dt.normalize()

validation_link["event_date"] = pd.to_datetime(
    validation_link["event_date"],
    errors="coerce",
).dt.normalize()

# Ensure missing milking labels form a consistent group
reference_link["milking"] = (
    reference_link["milking"]
    .astype("string")
    .fillna("__MISSING__")
)

validation_link["milking"] = (
    validation_link["milking"]
    .astype("string")
    .fillna("__MISSING__")
)

# Create weekly date groups
reference_link["event_week"] = (
    reference_link["event_date"]
    .dt.to_period("W")
    .dt.start_time
)

validation_link["event_week"] = (
    validation_link["event_date"]
    .dt.to_period("W")
    .dt.start_time
)

# ---------------------------------------------------------
# Build each cow's weekly/milking profile
# ---------------------------------------------------------

weekly_profiles = (
    reference_link
    .groupby(
        ["event_week", "milking", "cow_id"],
        as_index=False,
    )[link_features]
    .median()
)

# Impute and standardize the matching measurements
link_imputer = SimpleImputer(strategy="median")
link_scaler = StandardScaler()

profile_values = link_imputer.fit_transform(
    weekly_profiles[link_features]
)

profile_values = link_scaler.fit_transform(
    profile_values
)

encoded_columns = [
    f"link_feature_{i}"
    for i in range(len(link_features))
]

weekly_profiles[encoded_columns] = profile_values

# Use a manageable validation pilot first
pilot_size = min(50_000, len(validation_link))

linkage_pilot = validation_link.sample(
    n=pilot_size,
    random_state=42,
).copy()

pilot_values = link_imputer.transform(
    linkage_pilot[link_features]
)

pilot_values = link_scaler.transform(pilot_values)

linkage_pilot[encoded_columns] = pilot_values

# Profiles available within each week and milking session
profile_lookup = {
    key: group.reset_index(drop=True)
    for key, group in weekly_profiles.groupby(
        ["event_week", "milking"],
        sort=False,
    )
}

# Cows already observed on each exact date/session
# cannot normally be assigned to another row from that session.
known_cows_by_session = (
    reference_link
    .groupby(["event_date", "milking"])["cow_id"]
    .agg(set)
    .to_dict()
)

predicted_cow_ids = pd.Series(
    pd.NA,
    index=linkage_pilot.index,
    dtype="string",
)

true_cow_available = pd.Series(
    False,
    index=linkage_pilot.index,
)

candidate_counts = pd.Series(
    np.nan,
    index=linkage_pilot.index,
)

# ---------------------------------------------------------
# Match each date/milking group against eligible cows
# ---------------------------------------------------------

pilot_groups = linkage_pilot.groupby(
    ["event_date", "event_week", "milking"],
    sort=False,
)

for (
    event_date,
    event_week,
    milking,
), query_group in pilot_groups:

    profile_key = (event_week, milking)

    if profile_key not in profile_lookup:
        continue

    candidates = profile_lookup[profile_key]

    cows_already_recorded = known_cows_by_session.get(
        (event_date, milking),
        set(),
    )

    # Remove cows already represented in this exact session
    candidates = candidates.loc[
        ~candidates["cow_id"].isin(cows_already_recorded)
    ].reset_index(drop=True)

    if len(candidates) == 0:
        continue

    candidate_set = set(candidates["cow_id"])

    true_cow_available.loc[query_group.index] = (
        query_group["cow_id"].isin(candidate_set)
    )

    candidate_counts.loc[query_group.index] = len(candidates)

    tree = cKDTree(
        candidates[encoded_columns].to_numpy()
    )

    query_values = query_group[
        encoded_columns
    ].to_numpy()

    distances, nearest_positions = tree.query(
        query_values,
        k=1,
    )

    predicted_cow_ids.loc[query_group.index] = (
        candidates.iloc[
            np.atleast_1d(nearest_positions)
        ]["cow_id"].to_numpy()
    )

# ---------------------------------------------------------
# Evaluate linkage
# ---------------------------------------------------------

linkage_pilot["predicted_cow_id"] = predicted_cow_ids

linkage_pilot["true_cow_in_candidates"] = (
    true_cow_available
)

matched_mask = linkage_pilot[
    "predicted_cow_id"
].notna()

exact_match = (
    linkage_pilot.loc[
        matched_mask,
        "predicted_cow_id",
    ]
    ==
    linkage_pilot.loc[
        matched_mask,
        "cow_id",
    ]
)

print("Pilot records:", len(linkage_pilot))

print(
    "Records assigned a candidate:",
    int(matched_mask.sum()),
)

print(
    "Assignment coverage:",
    round(matched_mask.mean() * 100, 2),
    "%",
)

print(
    "True cow present among candidates:",
    round(true_cow_available.mean() * 100, 2),
    "%",
)

print(
    "Median candidate cows per record:",
    round(candidate_counts.median(), 1),
)

print(
    "Cow-ID exact-match accuracy:",
    round(exact_match.mean(), 4),
)

Pilot records: 50000
Records assigned a candidate: 50000
Assignment coverage: 100.0 %
True cow present among candidates: 80.96 %
Median candidate cows per record: 2473.0
Cow-ID exact-match accuracy: 0.0258


In [14]:
# ---------------------------------------------------------
# EXACT-DATE RECORD LINKAGE
#
# Search individual known records from the same date.
# Do not exclude cows already observed that day.
# ---------------------------------------------------------

reference_exact = reference_link.copy()

reference_exact_values = link_imputer.transform(
    reference_exact[link_features]
)

reference_exact_values = link_scaler.transform(
    reference_exact_values
)

reference_exact[encoded_columns] = reference_exact_values

# Index known records by exact date
exact_date_lookup = {
    event_date: group.reset_index(drop=True)
    for event_date, group in reference_exact.groupby(
        "event_date",
        sort=False,
    )
}

exact_date_predictions = pd.Series(
    pd.NA,
    index=linkage_pilot.index,
    dtype="string",
)

exact_date_true_available = pd.Series(
    False,
    index=linkage_pilot.index,
)

top_5_hit = pd.Series(
    False,
    index=linkage_pilot.index,
)

top_20_hit = pd.Series(
    False,
    index=linkage_pilot.index,
)

exact_candidate_counts = pd.Series(
    np.nan,
    index=linkage_pilot.index,
)

for event_date, query_group in linkage_pilot.groupby(
    "event_date",
    sort=False,
):
    if event_date not in exact_date_lookup:
        continue

    candidates = exact_date_lookup[event_date]

    if len(candidates) == 0:
        continue

    candidate_cows = set(candidates["cow_id"])

    exact_date_true_available.loc[query_group.index] = (
        query_group["cow_id"].isin(candidate_cows)
    )

    exact_candidate_counts.loc[query_group.index] = len(candidates)

    tree = cKDTree(
        candidates[encoded_columns].to_numpy()
    )

    query_values = query_group[
        encoded_columns
    ].to_numpy()

    number_neighbors = min(20, len(candidates))

    distances, neighbor_positions = tree.query(
        query_values,
        k=number_neighbors,
    )

    # Make results two-dimensional even if only one neighbor exists
    neighbor_positions = np.asarray(neighbor_positions)

    if neighbor_positions.ndim == 1:
        neighbor_positions = neighbor_positions.reshape(-1, 1)

    neighbor_cow_ids = (
        candidates.iloc[
            neighbor_positions.reshape(-1)
        ]["cow_id"]
        .to_numpy()
        .reshape(
            len(query_group),
            number_neighbors,
        )
    )

    true_ids = query_group["cow_id"].to_numpy()

    # Closest record
    exact_date_predictions.loc[query_group.index] = (
        neighbor_cow_ids[:, 0]
    )

    # Whether true cow is among closest 5 records
    top_5_width = min(5, number_neighbors)

    top_5_hit.loc[query_group.index] = [
        true_id in neighbor_ids[:top_5_width]
        for true_id, neighbor_ids
        in zip(true_ids, neighbor_cow_ids)
    ]

    # Whether true cow is among closest 20 records
    top_20_hit.loc[query_group.index] = [
        true_id in neighbor_ids
        for true_id, neighbor_ids
        in zip(true_ids, neighbor_cow_ids)
    ]

linkage_pilot["exact_date_predicted_cow"] = (
    exact_date_predictions
)

linkage_pilot["exact_date_true_available"] = (
    exact_date_true_available
)

linkage_pilot["exact_date_top_5_hit"] = top_5_hit
linkage_pilot["exact_date_top_20_hit"] = top_20_hit

exact_assigned = exact_date_predictions.notna()

exact_correct = (
    exact_date_predictions.loc[exact_assigned]
    ==
    linkage_pilot.loc[exact_assigned, "cow_id"]
)

# Accuracy conditional on the correct cow actually having
# another known record on the same date
available_mask = (
    exact_assigned
    & exact_date_true_available
)

available_correct = (
    exact_date_predictions.loc[available_mask]
    ==
    linkage_pilot.loc[available_mask, "cow_id"]
)

print("Pilot records:", len(linkage_pilot))

print(
    "Exact-date assignment coverage:",
    round(exact_assigned.mean() * 100, 2),
    "%",
)

print(
    "True cow has another known record on same date:",
    round(exact_date_true_available.mean() * 100, 2),
    "%",
)

print(
    "Median exact-date candidate records:",
    round(exact_candidate_counts.median(), 1),
)

print(
    "Top-1 exact-match accuracy:",
    round(exact_correct.mean(), 4),
)

print(
    "Top-1 accuracy when true cow is available:",
    round(available_correct.mean(), 4),
)

print(
    "True cow found among 5 nearest records:",
    round(top_5_hit.mean() * 100, 2),
    "%",
)

print(
    "True cow found among 20 nearest records:",
    round(top_20_hit.mean() * 100, 2),
    "%",
)

Pilot records: 50000
Exact-date assignment coverage: 100.0 %
True cow has another known record on same date: 54.69 %
Median exact-date candidate records: 3271.0
Top-1 exact-match accuracy: 0.0204
Top-1 accuracy when true cow is available: 0.0372
True cow found among 5 nearest records: 5.49 %
True cow found among 20 nearest records: 12.15 %


In [15]:
# ---------------------------------------------------------
# SOURCE-ORDER COW IDENTIFICATION
#
# Test whether cows occur in a consistent position within
# date/milking sessions.
# ---------------------------------------------------------

order_columns = [
    "source_row",
    "event_date",
    "milking",
]

# Include every row when calculating position.
# No hidden target values are used to calculate position.
all_order_rows = pd.concat(
    [
        known_context[order_columns],
        context_to_predict[order_columns],
    ],
    ignore_index=True,
)

all_order_rows = all_order_rows.drop_duplicates(
    subset="source_row"
).copy()

all_order_rows["event_date"] = pd.to_datetime(
    all_order_rows["event_date"],
    errors="coerce",
).dt.normalize()

all_order_rows["milking"] = (
    all_order_rows["milking"]
    .astype("string")
    .fillna("__MISSING__")
)

all_order_rows = (
    all_order_rows
    .dropna(subset=["source_row", "event_date"])
    .sort_values(
        [
            "event_date",
            "milking",
            "source_row",
        ]
    )
    .reset_index(drop=True)
)

# Position of each record within a date/milking session
all_order_rows["session_position"] = (
    all_order_rows
    .groupby(
        ["event_date", "milking"],
        sort=False,
    )
    .cumcount()
)

all_order_rows["session_size"] = (
    all_order_rows
    .groupby(
        ["event_date", "milking"],
        sort=False,
    )["source_row"]
    .transform("size")
)

all_order_rows["month_key"] = (
    all_order_rows["event_date"]
    .dt.to_period("M")
    .astype("string")
)

all_order_rows["year_key"] = (
    all_order_rows["event_date"]
    .dt.year
)

position_information = all_order_rows[
    [
        "source_row",
        "event_date",
        "milking",
        "month_key",
        "year_key",
        "session_position",
        "session_size",
    ]
].copy()

# ---------------------------------------------------------
# Add session position to known training/validation records
# ---------------------------------------------------------

train_order = (
    known_context.iloc[train_indices][
        ["source_row", "cow_id"]
    ]
    .merge(
        position_information,
        on="source_row",
        how="inner",
    )
)

validation_order = (
    known_context.iloc[validation_indices][
        ["source_row", "cow_id"]
    ]
    .merge(
        position_information,
        on="source_row",
        how="inner",
    )
)

train_order["cow_id"] = (
    train_order["cow_id"].astype("string")
)

validation_order["cow_id"] = (
    validation_order["cow_id"].astype("string")
)

# ---------------------------------------------------------
# Helper: find the most frequent cow for each position
# ---------------------------------------------------------

def make_position_mapping(
    dataframe,
    grouping_columns,
    prediction_name,
):
    counts = (
        dataframe
        .groupby(
            grouping_columns + ["cow_id"],
            dropna=False,
        )
        .size()
        .rename("occurrences")
        .reset_index()
    )

    mapping = (
        counts
        .sort_values(
            "occurrences",
            ascending=False,
        )
        .drop_duplicates(
            subset=grouping_columns,
            keep="first",
        )
        [
            grouping_columns
            + ["cow_id", "occurrences"]
        ]
        .rename(
            columns={
                "cow_id": prediction_name,
                "occurrences": (
                    prediction_name
                    + "_occurrences"
                ),
            }
        )
    )

    return mapping

# Most specific mapping:
# month + milking + position
month_position_mapping = make_position_mapping(
    train_order,
    [
        "month_key",
        "milking",
        "session_position",
    ],
    "month_position_cow",
)

# Fallback:
# year + milking + position
year_position_mapping = make_position_mapping(
    train_order,
    [
        "year_key",
        "milking",
        "session_position",
    ],
    "year_position_cow",
)

# Final fallback:
# milking + position across the complete dataset
global_position_mapping = make_position_mapping(
    train_order,
    [
        "milking",
        "session_position",
    ],
    "global_position_cow",
)

# ---------------------------------------------------------
# Apply mappings to validation records
# ---------------------------------------------------------

order_results = validation_order.merge(
    month_position_mapping,
    on=[
        "month_key",
        "milking",
        "session_position",
    ],
    how="left",
)

order_results = order_results.merge(
    year_position_mapping,
    on=[
        "year_key",
        "milking",
        "session_position",
    ],
    how="left",
)

order_results = order_results.merge(
    global_position_mapping,
    on=[
        "milking",
        "session_position",
    ],
    how="left",
)

# Prefer monthly prediction, then yearly, then global
order_results["predicted_cow_id"] = (
    order_results["month_position_cow"]
    .fillna(order_results["year_position_cow"])
    .fillna(order_results["global_position_cow"])
)

assigned_order = order_results[
    "predicted_cow_id"
].notna()

order_correct = (
    order_results.loc[
        assigned_order,
        "predicted_cow_id",
    ]
    ==
    order_results.loc[
        assigned_order,
        "cow_id",
    ]
)

month_available = order_results[
    "month_position_cow"
].notna()

month_correct = (
    order_results.loc[
        month_available,
        "month_position_cow",
    ]
    ==
    order_results.loc[
        month_available,
        "cow_id",
    ]
)

print("Validation records:", len(order_results))

print(
    "Source-order assignment coverage:",
    round(assigned_order.mean() * 100, 2),
    "%",
)

print(
    "Overall source-order accuracy:",
    round(order_correct.mean(), 4),
)

print(
    "Monthly-position coverage:",
    round(month_available.mean() * 100, 2),
    "%",
)

print(
    "Monthly-position accuracy:",
    round(month_correct.mean(), 4),
)

print(
    "Median records per session:",
    round(
        all_order_rows["session_size"].median(),
        1,
    ),
)

print(
    "Maximum records per session:",
    int(all_order_rows["session_size"].max()),
)

Validation records: 562172
Source-order assignment coverage: 99.87 %
Overall source-order accuracy: 0.0002
Monthly-position coverage: 99.13 %
Monthly-position accuracy: 0.0002
Median records per session: 1951.0
Maximum records per session: 5329


In [16]:
# ---------------------------------------------------------
# CANDIDATE-RECALL TEST
#
# Do not exclude any cows.
# Find the rank of the true cow among weekly profiles.
# ---------------------------------------------------------

weekly_profile_lookup = {
    key: group.reset_index(drop=True)
    for key, group in weekly_profiles.groupby(
        ["event_week", "milking"],
        sort=False,
    )
}

recall_levels = [1, 5, 20, 50, 100, 250, 500]

recall_results = {
    level: pd.Series(
        False,
        index=linkage_pilot.index,
    )
    for level in recall_levels
}

true_candidate_available = pd.Series(
    False,
    index=linkage_pilot.index,
)

true_cow_rank = pd.Series(
    np.nan,
    index=linkage_pilot.index,
)

candidate_pool_size = pd.Series(
    np.nan,
    index=linkage_pilot.index,
)

for (
    event_week,
    milking,
), query_group in linkage_pilot.groupby(
    ["event_week", "milking"],
    sort=False,
):
    profile_key = (event_week, milking)

    if profile_key not in weekly_profile_lookup:
        continue

    candidates = weekly_profile_lookup[profile_key]

    if len(candidates) == 0:
        continue

    candidate_pool_size.loc[query_group.index] = len(candidates)

    candidate_cows = set(candidates["cow_id"])

    true_candidate_available.loc[query_group.index] = (
        query_group["cow_id"].isin(candidate_cows)
    )

    maximum_neighbors = min(
        max(recall_levels),
        len(candidates),
    )

    tree = cKDTree(
        candidates[encoded_columns].to_numpy()
    )

    distances, neighbor_positions = tree.query(
        query_group[encoded_columns].to_numpy(),
        k=maximum_neighbors,
    )

    neighbor_positions = np.asarray(neighbor_positions)

    if neighbor_positions.ndim == 1:
        neighbor_positions = neighbor_positions.reshape(-1, 1)

    neighbor_cow_ids = (
        candidates.iloc[
            neighbor_positions.reshape(-1)
        ]["cow_id"]
        .to_numpy()
        .reshape(
            len(query_group),
            maximum_neighbors,
        )
    )

    true_ids = query_group["cow_id"].to_numpy()

    cow_matches = (
        neighbor_cow_ids
        == true_ids[:, np.newaxis]
    )

    has_match = cow_matches.any(axis=1)

    first_match_position = np.where(
        has_match,
        cow_matches.argmax(axis=1) + 1,
        np.nan,
    )

    true_cow_rank.loc[query_group.index] = (
        first_match_position
    )

    for level in recall_levels:
        usable_level = min(level, maximum_neighbors)

        recall_results[level].loc[
            query_group.index
        ] = cow_matches[
            :,
            :usable_level,
        ].any(axis=1)

print("Pilot records:", len(linkage_pilot))

print(
    "True cow has a weekly profile:",
    round(true_candidate_available.mean() * 100, 2),
    "%",
)

print(
    "Median candidate cows:",
    round(candidate_pool_size.median(), 1),
)

for level in recall_levels:
    print(
        f"Recall at {level}:",
        round(
            recall_results[level].mean() * 100,
            2,
        ),
        "%",
    )

print(
    "Median true-cow rank when found:",
    round(true_cow_rank.dropna().median(), 1),
)

Pilot records: 50000
True cow has a weekly profile: 83.64 %
Median candidate cows: 3561.0
Recall at 1: 2.24 %
Recall at 5: 7.51 %
Recall at 20: 18.12 %
Recall at 50: 29.5 %
Recall at 100: 40.84 %
Recall at 250: 57.02 %
Recall at 500: 67.98 %
Median true-cow rank when found: 67.0


In [17]:
# ---------------------------------------------------------
# FIVE-WEEK LONGITUDINAL COW PROFILES
#
# Each cow's records from two weeks before through two weeks
# after the query week contribute to its candidate profile.
# ---------------------------------------------------------

base_weekly_profiles = (
    reference_link
    .groupby(
        ["event_week", "milking", "cow_id"],
        as_index=False,
    )[link_features]
    .median()
)

expanded_profile_parts = []

# Make every observed profile available to query weeks
# within plus or minus two weeks.
for week_offset in [-2, -1, 0, 1, 2]:
    profile_part = base_weekly_profiles.copy()

    profile_part["event_week"] = (
        profile_part["event_week"]
        + pd.to_timedelta(
            week_offset * 7,
            unit="D",
        )
    )

    expanded_profile_parts.append(profile_part)

expanded_profiles = pd.concat(
    expanded_profile_parts,
    ignore_index=True,
)

five_week_profiles = (
    expanded_profiles
    .groupby(
        ["event_week", "milking", "cow_id"],
        as_index=False,
    )[link_features]
    .median()
)

# Transform with the same imputer and scaler
five_week_values = link_imputer.transform(
    five_week_profiles[link_features]
)

five_week_values = link_scaler.transform(
    five_week_values
)

five_week_profiles[encoded_columns] = five_week_values

five_week_lookup = {
    key: group.reset_index(drop=True)
    for key, group in five_week_profiles.groupby(
        ["event_week", "milking"],
        sort=False,
    )
}

# ---------------------------------------------------------
# Evaluate candidate recall
# ---------------------------------------------------------

recall_levels = [1, 5, 20, 50, 100, 250, 500]

five_week_recall = {
    level: pd.Series(
        False,
        index=linkage_pilot.index,
    )
    for level in recall_levels
}

five_week_available = pd.Series(
    False,
    index=linkage_pilot.index,
)

five_week_true_rank = pd.Series(
    np.nan,
    index=linkage_pilot.index,
)

five_week_pool_size = pd.Series(
    np.nan,
    index=linkage_pilot.index,
)

for (
    event_week,
    milking,
), query_group in linkage_pilot.groupby(
    ["event_week", "milking"],
    sort=False,
):
    key = (event_week, milking)

    if key not in five_week_lookup:
        continue

    candidates = five_week_lookup[key]

    if len(candidates) == 0:
        continue

    five_week_pool_size.loc[
        query_group.index
    ] = len(candidates)

    candidate_cows = set(candidates["cow_id"])

    five_week_available.loc[
        query_group.index
    ] = query_group["cow_id"].isin(candidate_cows)

    maximum_neighbors = min(
        500,
        len(candidates),
    )

    tree = cKDTree(
        candidates[encoded_columns].to_numpy()
    )

    distances, positions = tree.query(
        query_group[encoded_columns].to_numpy(),
        k=maximum_neighbors,
    )

    positions = np.asarray(positions)

    if positions.ndim == 1:
        positions = positions.reshape(-1, 1)

    neighbor_cows = (
        candidates.iloc[
            positions.reshape(-1)
        ]["cow_id"]
        .to_numpy()
        .reshape(
            len(query_group),
            maximum_neighbors,
        )
    )

    true_cows = query_group[
        "cow_id"
    ].to_numpy()

    matches = (
        neighbor_cows
        == true_cows[:, np.newaxis]
    )

    has_match = matches.any(axis=1)

    ranks = np.where(
        has_match,
        matches.argmax(axis=1) + 1,
        np.nan,
    )

    five_week_true_rank.loc[
        query_group.index
    ] = ranks

    for level in recall_levels:
        usable_level = min(
            level,
            maximum_neighbors,
        )

        five_week_recall[level].loc[
            query_group.index
        ] = matches[
            :,
            :usable_level,
        ].any(axis=1)

print("Pilot records:", len(linkage_pilot))

print(
    "True cow has a five-week profile:",
    round(five_week_available.mean() * 100, 2),
    "%",
)

print(
    "Median candidate cows:",
    round(five_week_pool_size.median(), 1),
)

for level in recall_levels:
    print(
        f"Five-week recall at {level}:",
        round(
            five_week_recall[level].mean() * 100,
            2,
        ),
        "%",
    )

print(
    "Median true-cow rank when found:",
    round(
        five_week_true_rank.dropna().median(),
        1,
    ),
)

Pilot records: 50000
True cow has a five-week profile: 99.78 %
Median candidate cows: 4482.0
Five-week recall at 1: 2.66 %
Five-week recall at 5: 9.1 %
Five-week recall at 20: 22.38 %
Five-week recall at 50: 36.69 %
Five-week recall at 100: 50.52 %
Five-week recall at 250: 69.76 %
Five-week recall at 500: 82.57 %
Median true-cow rank when found: 64.0


In [18]:
# ---------------------------------------------------------
# SUPERVISED COW-ID RERANKER
#
# Stage 1: Retrieve 500 plausible candidate cows.
# Stage 2: Predict whether each query/candidate pair
#          represents the same cow.
# ---------------------------------------------------------

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(42)

# Keep this first experiment manageable
reranker_sample_size = min(
    30_000,
    len(linkage_pilot),
)

reranker_queries = linkage_pilot.sample(
    n=reranker_sample_size,
    random_state=42,
).copy()

reranker_train_queries = reranker_queries.sample(
    n=min(20_000, len(reranker_queries)),
    random_state=43,
).copy()

reranker_evaluation_queries = reranker_queries.drop(
    index=reranker_train_queries.index
).copy()

print(
    "Reranker training queries:",
    len(reranker_train_queries),
)

print(
    "Reranker evaluation queries:",
    len(reranker_evaluation_queries),
)

# ---------------------------------------------------------
# Pair-feature function
# ---------------------------------------------------------

def create_pair_features(query_values, candidate_values):
    """
    query_values and candidate_values must have equal shapes:
    rows = query/candidate pairs
    columns = standardized milk measurements
    """

    signed_difference = (
        query_values - candidate_values
    )

    absolute_difference = np.abs(
        signed_difference
    )

    squared_difference = (
        signed_difference ** 2
    )

    euclidean_distance = np.sqrt(
        squared_difference.sum(axis=1)
    ).reshape(-1, 1)

    maximum_difference = (
        absolute_difference.max(axis=1)
        .reshape(-1, 1)
    )

    features = np.column_stack(
        [
            query_values,
            candidate_values,
            signed_difference,
            absolute_difference,
            squared_difference,
            euclidean_distance,
            maximum_difference,
        ]
    )

    return features.astype("float32")


# ---------------------------------------------------------
# Create labeled query/candidate training pairs
#
# For each query:
# - retain the true cow as the positive example;
# - retain the closest hard negatives;
# - sample additional negatives farther down the ranking.
# ---------------------------------------------------------

training_feature_parts = []
training_label_parts = []

training_queries_with_positive = 0

for (
    event_week,
    milking,
), query_group in reranker_train_queries.groupby(
    ["event_week", "milking"],
    sort=False,
):
    key = (event_week, milking)

    if key not in five_week_lookup:
        continue

    candidates = five_week_lookup[key]

    if len(candidates) == 0:
        continue

    number_neighbors = min(
        500,
        len(candidates),
    )

    candidate_matrix = candidates[
        encoded_columns
    ].to_numpy(dtype="float32")

    tree = cKDTree(candidate_matrix)

    query_matrix = query_group[
        encoded_columns
    ].to_numpy(dtype="float32")

    distances, positions = tree.query(
        query_matrix,
        k=number_neighbors,
    )

    positions = np.asarray(positions)

    if positions.ndim == 1:
        positions = positions.reshape(-1, 1)

    candidate_cow_matrix = (
        candidates.iloc[
            positions.reshape(-1)
        ]["cow_id"]
        .to_numpy()
        .reshape(
            len(query_group),
            number_neighbors,
        )
    )

    true_cows = query_group[
        "cow_id"
    ].to_numpy()

    for row_number in range(len(query_group)):
        candidate_cows = candidate_cow_matrix[
            row_number
        ]

        positive_locations = np.flatnonzero(
            candidate_cows == true_cows[row_number]
        )

        # The reranker cannot learn this query if its true
        # cow was not retrieved among the 500 candidates.
        if len(positive_locations) == 0:
            continue

        positive_location = int(
            positive_locations[0]
        )

        # Closest candidates are difficult negatives
        hard_negative_locations = np.arange(
            min(15, number_neighbors)
        )

        hard_negative_locations = (
            hard_negative_locations[
                hard_negative_locations
                != positive_location
            ]
        )

        # Add negatives from farther down the ranking
        farther_locations = np.arange(
            min(15, number_neighbors),
            number_neighbors,
        )

        farther_locations = farther_locations[
            farther_locations != positive_location
        ]

        random_negative_count = min(
            15,
            len(farther_locations),
        )

        if random_negative_count > 0:
            random_negative_locations = rng.choice(
                farther_locations,
                size=random_negative_count,
                replace=False,
            )
        else:
            random_negative_locations = np.array(
                [],
                dtype=int,
            )

        selected_locations = np.unique(
            np.concatenate(
                [
                    np.array([positive_location]),
                    hard_negative_locations,
                    random_negative_locations,
                ]
            )
        )

        selected_candidate_positions = positions[
            row_number,
            selected_locations,
        ]

        selected_candidate_values = candidate_matrix[
            selected_candidate_positions
        ]

        query_value = query_matrix[
            row_number
        ]

        repeated_query_values = np.repeat(
            query_value.reshape(1, -1),
            len(selected_locations),
            axis=0,
        )

        pair_features = create_pair_features(
            repeated_query_values,
            selected_candidate_values,
        )

        pair_labels = (
            selected_locations
            == positive_location
        ).astype("int8")

        training_feature_parts.append(
            pair_features
        )

        training_label_parts.append(
            pair_labels
        )

        training_queries_with_positive += 1

X_linkage_train = np.vstack(
    training_feature_parts
)

y_linkage_train = np.concatenate(
    training_label_parts
)

print(
    "Training queries with true cow retrieved:",
    training_queries_with_positive,
)

print(
    "Training candidate pairs:",
    len(X_linkage_train),
)

print(
    "Positive pairs:",
    int(y_linkage_train.sum()),
)

print(
    "Negative pairs:",
    int((y_linkage_train == 0).sum()),
)

# ---------------------------------------------------------
# Train binary linkage model
# ---------------------------------------------------------

cow_reranker = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=30,
    l2_regularization=2.0,
    class_weight="balanced",
    random_state=42,
)

print("\nFitting supervised cow-ID reranker...")

cow_reranker.fit(
    X_linkage_train,
    y_linkage_train,
)

# Free the large pair matrix before evaluation if desired
# del X_linkage_train

# ---------------------------------------------------------
# Evaluate on separate pilot queries
# ---------------------------------------------------------

evaluation_records = 0
evaluation_assigned = 0
true_cow_in_top_500 = 0
nearest_neighbor_correct = 0
reranker_correct = 0

for (
    event_week,
    milking,
), query_group in reranker_evaluation_queries.groupby(
    ["event_week", "milking"],
    sort=False,
):
    key = (event_week, milking)

    evaluation_records += len(query_group)

    if key not in five_week_lookup:
        continue

    candidates = five_week_lookup[key]

    if len(candidates) == 0:
        continue

    number_neighbors = min(
        500,
        len(candidates),
    )

    candidate_matrix = candidates[
        encoded_columns
    ].to_numpy(dtype="float32")

    tree = cKDTree(candidate_matrix)

    query_matrix = query_group[
        encoded_columns
    ].to_numpy(dtype="float32")

    distances, positions = tree.query(
        query_matrix,
        k=number_neighbors,
    )

    positions = np.asarray(positions)

    if positions.ndim == 1:
        positions = positions.reshape(-1, 1)

    candidate_values = candidate_matrix[
        positions.reshape(-1)
    ].reshape(
        len(query_group),
        number_neighbors,
        len(encoded_columns),
    )

    candidate_cow_ids = (
        candidates.iloc[
            positions.reshape(-1)
        ]["cow_id"]
        .to_numpy()
        .reshape(
            len(query_group),
            number_neighbors,
        )
    )

    repeated_queries = np.repeat(
        query_matrix[:, np.newaxis, :],
        number_neighbors,
        axis=1,
    )

    evaluation_pair_features = create_pair_features(
        repeated_queries.reshape(
            -1,
            len(encoded_columns),
        ),
        candidate_values.reshape(
            -1,
            len(encoded_columns),
        ),
    )

    same_cow_probabilities = (
        cow_reranker.predict_proba(
            evaluation_pair_features
        )[:, 1]
        .reshape(
            len(query_group),
            number_neighbors,
        )
    )

    best_positions = np.argmax(
        same_cow_probabilities,
        axis=1,
    )

    reranked_predictions = candidate_cow_ids[
        np.arange(len(query_group)),
        best_positions,
    ]

    nearest_predictions = candidate_cow_ids[:, 0]

    true_ids = query_group[
        "cow_id"
    ].to_numpy()

    true_available = (
        candidate_cow_ids
        == true_ids[:, np.newaxis]
    ).any(axis=1)

    evaluation_assigned += len(query_group)

    true_cow_in_top_500 += int(
        true_available.sum()
    )

    nearest_neighbor_correct += int(
        (nearest_predictions == true_ids).sum()
    )

    reranker_correct += int(
        (reranked_predictions == true_ids).sum()
    )

print("\nRERANKER EVALUATION")

print(
    "Evaluation records:",
    evaluation_records,
)

print(
    "Assignment coverage:",
    round(
        evaluation_assigned
        / evaluation_records
        * 100,
        2,
    ),
    "%",
)

print(
    "True cow in top 500:",
    round(
        true_cow_in_top_500
        / evaluation_records
        * 100,
        2,
    ),
    "%",
)

print(
    "Nearest-profile accuracy:",
    round(
        nearest_neighbor_correct
        / evaluation_records,
        4,
    ),
)

print(
    "Reranked cow-ID accuracy:",
    round(
        reranker_correct
        / evaluation_records,
        4,
    ),
)

Reranker training queries: 20000
Reranker evaluation queries: 10000
Training queries with true cow retrieved: 16485
Training candidate pairs: 507294
Positive pairs: 16485
Negative pairs: 490809

Fitting supervised cow-ID reranker...

RERANKER EVALUATION
Evaluation records: 10000
Assignment coverage: 100.0 %
True cow in top 500: 82.5 %
Nearest-profile accuracy: 0.0281
Reranked cow-ID accuracy: 0.0156


In [19]:
 # ---------------------------------------------------------
# TEMPORAL-TRAJECTORY COW LINKAGE
# ---------------------------------------------------------

trajectory_sample = (
    reranker_evaluation_queries
    .sample(
        n=min(3_000, len(reranker_evaluation_queries)),
        random_state=44,
    )
    .copy()
)

# Create one daily profile for each cow and milking
cow_daily_history = (
    reference_link
    .groupby(
        ["cow_id", "milking", "event_date"],
        as_index=False,
    )[link_features]
    .median()
)

history_values = link_imputer.transform(
    cow_daily_history[link_features]
)

history_values = link_scaler.transform(
    history_values
)

cow_daily_history[encoded_columns] = history_values

cow_daily_history["date_number"] = (
    cow_daily_history["event_date"]
    - pd.Timestamp("1970-01-01")
).dt.days.astype("int32")

# Store each cow's history as sorted NumPy arrays
history_lookup = {}

for (
    cow_id,
    milking,
), history_group in cow_daily_history.groupby(
    ["cow_id", "milking"],
    sort=False,
):
    history_group = history_group.sort_values(
        "date_number"
    )

    history_lookup[(cow_id, milking)] = (
        history_group["date_number"].to_numpy(
            dtype="int32"
        ),
        history_group[encoded_columns].to_numpy(
            dtype="float32"
        ),
    )

print(
    "Cow/milking histories:",
    len(history_lookup),
)

# Cache the five-week candidate trees
trajectory_keys = (
    trajectory_sample[
        ["event_week", "milking"]
    ]
    .drop_duplicates()
)

candidate_tree_lookup = {}

for key_row in trajectory_keys.itertuples(index=False):
    key = (
        key_row.event_week,
        key_row.milking,
    )

    if key not in five_week_lookup:
        continue

    candidates = five_week_lookup[key]

    candidate_matrix = candidates[
        encoded_columns
    ].to_numpy(dtype="float32")

    candidate_tree_lookup[key] = (
        candidates,
        candidate_matrix,
        cKDTree(candidate_matrix),
    )

# Different weights for combining:
# trajectory distance + weight * five-week distance
weekly_distance_weights = [
    0.0,
    0.10,
    0.25,
    0.50,
    1.00,
]

correct_by_weight = {
    weight: 0
    for weight in weekly_distance_weights
}

assigned_by_weight = {
    weight: 0
    for weight in weekly_distance_weights
}

nearest_profile_correct = 0
true_cow_retrieved = 0
records_processed = 0
records_with_trajectory = 0

for row_number, query in enumerate(
    trajectory_sample.itertuples()
):
    key = (
        query.event_week,
        query.milking,
    )

    if key not in candidate_tree_lookup:
        continue

    candidates, candidate_matrix, tree = (
        candidate_tree_lookup[key]
    )

    query_values = np.array(
        [
            getattr(query, column)
            for column in encoded_columns
        ],
        dtype="float32",
    )

    number_candidates = min(
        500,
        len(candidates),
    )

    weekly_distances, candidate_positions = (
        tree.query(
            query_values,
            k=number_candidates,
        )
    )

    weekly_distances = np.atleast_1d(
        weekly_distances
    )

    candidate_positions = np.atleast_1d(
        candidate_positions
    )

    candidate_cows = (
        candidates.iloc[
            candidate_positions
        ]["cow_id"]
        .to_numpy()
    )

    true_cow = query.cow_id

    records_processed += 1

    if true_cow in candidate_cows:
        true_cow_retrieved += 1

    if candidate_cows[0] == true_cow:
        nearest_profile_correct += 1

    query_date_number = int(
        (
            query.event_date
            - pd.Timestamp("1970-01-01")
        ).days
    )

    trajectory_distances = np.full(
        number_candidates,
        np.inf,
        dtype="float32",
    )

    for candidate_number, candidate_cow in enumerate(
        candidate_cows
    ):
        history_key = (
            candidate_cow,
            query.milking,
        )

        if history_key not in history_lookup:
            continue

        history_dates, history_measurements = (
            history_lookup[history_key]
        )

        insertion_point = np.searchsorted(
            history_dates,
            query_date_number,
        )

        before_position = insertion_point - 1
        after_position = insertion_point

        before_available = before_position >= 0
        after_available = (
            after_position < len(history_dates)
        )

        estimated_measurements = None
        nearest_day_gap = np.inf

        # Interpolate between observations before and after
        if before_available and after_available:
            before_date = history_dates[
                before_position
            ]

            after_date = history_dates[
                after_position
            ]

            before_values = history_measurements[
                before_position
            ]

            after_values = history_measurements[
                after_position
            ]

            before_gap = (
                query_date_number - before_date
            )

            after_gap = (
                after_date - query_date_number
            )

            nearest_day_gap = min(
                before_gap,
                after_gap,
            )

            total_gap = before_gap + after_gap

            if total_gap == 0:
                estimated_measurements = (
                    before_values
                )
            else:
                estimated_measurements = (
                    before_values * after_gap
                    + after_values * before_gap
                ) / total_gap

        elif before_available:
            before_date = history_dates[
                before_position
            ]

            nearest_day_gap = (
                query_date_number - before_date
            )

            estimated_measurements = (
                history_measurements[
                    before_position
                ]
            )

        elif after_available:
            after_date = history_dates[
                after_position
            ]

            nearest_day_gap = (
                after_date - query_date_number
            )

            estimated_measurements = (
                history_measurements[
                    after_position
                ]
            )

        if estimated_measurements is None:
            continue

        measurement_distance = np.linalg.norm(
            query_values
            - estimated_measurements
        )

        # Small penalty for histories farther from query date
        date_gap_penalty = (
            0.02 * nearest_day_gap
        )

        trajectory_distances[
            candidate_number
        ] = (
            measurement_distance
            + date_gap_penalty
        )

    finite_trajectory = np.isfinite(
        trajectory_distances
    )

    if not finite_trajectory.any():
        continue

    records_with_trajectory += 1

    for weight in weekly_distance_weights:
        combined_score = (
            trajectory_distances
            + weight * weekly_distances
        )

        best_candidate_position = np.argmin(
            combined_score
        )

        predicted_cow = candidate_cows[
            best_candidate_position
        ]

        assigned_by_weight[weight] += 1

        if predicted_cow == true_cow:
            correct_by_weight[weight] += 1

    if (row_number + 1) % 500 == 0:
        print(
            "Processed:",
            row_number + 1,
            "of",
            len(trajectory_sample),
        )

print("\nTEMPORAL TRAJECTORY RESULTS")

print(
    "Records processed:",
    records_processed,
)

print(
    "Records with trajectory predictions:",
    records_with_trajectory,
)

print(
    "True cow retrieved among 500:",
    round(
        true_cow_retrieved
        / records_processed
        * 100,
        2,
    ),
    "%",
)

print(
    "Original nearest-profile accuracy:",
    round(
        nearest_profile_correct
        / records_processed,
        4,
    ),
)

for weight in weekly_distance_weights:
    if assigned_by_weight[weight] == 0:
        continue

    accuracy = (
        correct_by_weight[weight]
        / assigned_by_weight[weight]
    )

    print(
        f"Trajectory accuracy "
        f"(weekly weight={weight}):",
        round(accuracy, 4),
    )

Cow/milking histories: 27055
Processed: 500 of 3000
Processed: 1000 of 3000
Processed: 1500 of 3000
Processed: 2000 of 3000
Processed: 2500 of 3000
Processed: 3000 of 3000

TEMPORAL TRAJECTORY RESULTS
Records processed: 3000
Records with trajectory predictions: 3000
True cow retrieved among 500: 82.3 %
Original nearest-profile accuracy: 0.0247
Trajectory accuracy (weekly weight=0.0): 0.025
Trajectory accuracy (weekly weight=0.1): 0.0273
Trajectory accuracy (weekly weight=0.25): 0.03
Trajectory accuracy (weekly weight=0.5): 0.025
Trajectory accuracy (weekly weight=1.0): 0.0263


In [20]:
# ---------------------------------------------------------
# SESSION-STRUCTURE DIAGNOSTIC
# ---------------------------------------------------------

session_diagnostic = known_context[
    [
        "cow_id",
        "event_date",
        "milking",
        "source_row",
    ]
].copy()

session_diagnostic["cow_id"] = (
    session_diagnostic["cow_id"].astype("string")
)

session_diagnostic["event_date"] = pd.to_datetime(
    session_diagnostic["event_date"],
    errors="coerce",
).dt.normalize()

session_diagnostic["milking"] = (
    session_diagnostic["milking"]
    .astype("string")
    .fillna("__MISSING__")
)

print("Unique cows:", session_diagnostic["cow_id"].nunique())

print(
    "Unique dates:",
    session_diagnostic["event_date"].nunique(),
)

print(
    "Unique milking labels:",
    session_diagnostic["milking"].nunique(),
)

print(
    "Milking labels:",
    session_diagnostic["milking"]
    .value_counts()
    .head(20)
    .to_dict(),
)

# Number of records for each cow in an exact date/milking group
cow_session_counts = (
    session_diagnostic
    .groupby(
        [
            "cow_id",
            "event_date",
            "milking",
        ]
    )
    .size()
)

print("\nRECORDS PER COW PER DATE/MILKING")

print(
    cow_session_counts.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

print(
    "Percent of cow-sessions containing more than one record:",
    round(
        (cow_session_counts > 1).mean() * 100,
        2,
    ),
    "%",
)

print(
    "Maximum records for one cow in one session:",
    int(cow_session_counts.max()),
)

# Number of records and distinct cows in each overall session
overall_sessions = (
    session_diagnostic
    .groupby(
        [
            "event_date",
            "milking",
        ]
    )
    .agg(
        records=("source_row", "size"),
        unique_cows=("cow_id", "nunique"),
    )
)

overall_sessions["records_per_cow"] = (
    overall_sessions["records"]
    / overall_sessions["unique_cows"]
)

print("\nOVERALL SESSION STRUCTURE")

print(
    overall_sessions[
        [
            "records",
            "unique_cows",
            "records_per_cow",
        ]
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

# Examine how known and missing records are distributed per session
known_session_rows = known_context[
    [
        "source_row",
        "event_date",
        "milking",
    ]
].copy()

known_session_rows["context_missing"] = 0

missing_session_rows = context_to_predict[
    [
        "source_row",
        "event_date",
        "milking",
    ]
].copy()

missing_session_rows["context_missing"] = 1

all_session_rows = pd.concat(
    [
        known_session_rows,
        missing_session_rows,
    ],
    ignore_index=True,
)

all_session_rows["event_date"] = pd.to_datetime(
    all_session_rows["event_date"],
    errors="coerce",
).dt.normalize()

all_session_rows["milking"] = (
    all_session_rows["milking"]
    .astype("string")
    .fillna("__MISSING__")
)

missing_session_summary = (
    all_session_rows
    .groupby(
        [
            "event_date",
            "milking",
        ]
    )
    .agg(
        total_records=("source_row", "size"),
        missing_records=("context_missing", "sum"),
    )
)

missing_session_summary["missing_percent"] = (
    missing_session_summary["missing_records"]
    / missing_session_summary["total_records"]
    * 100
)

print("\nMISSINGNESS WITHIN SESSIONS")

print(
    missing_session_summary.describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

KeyboardInterrupt: 

## High-confidence known-neighbor imputation
### Part 1: Calibrate a cow-ID confidence threshold

In [21]:
# ---------------------------------------------------------
# PART 1: CALIBRATE NEIGHBOR CONFIDENCE
#
# For every validation record:
# 1. Find the two closest five-week cow profiles.
# 2. Predict the closest cow.
# 3. Measure how clearly it beats the second-best cow.
# ---------------------------------------------------------

neighbor_prediction = pd.Series(
    pd.NA,
    index=linkage_pilot.index,
    dtype="string",
)

best_distance = pd.Series(
    np.nan,
    index=linkage_pilot.index,
)

second_distance = pd.Series(
    np.nan,
    index=linkage_pilot.index,
)

for (
    event_week,
    milking,
), query_group in linkage_pilot.groupby(
    ["event_week", "milking"],
    sort=False,
):
    key = (event_week, milking)

    if key not in five_week_lookup:
        continue

    candidates = five_week_lookup[key]

    if len(candidates) < 2:
        continue

    candidate_values = candidates[
        encoded_columns
    ].to_numpy(dtype="float32")

    query_values = query_group[
        encoded_columns
    ].to_numpy(dtype="float32")

    tree = cKDTree(candidate_values)

    distances, positions = tree.query(
        query_values,
        k=2,
    )

    predicted_ids = (
        candidates.iloc[positions[:, 0]]["cow_id"]
        .to_numpy()
    )

    neighbor_prediction.loc[
        query_group.index
    ] = predicted_ids

    best_distance.loc[
        query_group.index
    ] = distances[:, 0]

    second_distance.loc[
        query_group.index
    ] = distances[:, 1]

# Create validation results
neighbor_confidence_results = pd.DataFrame(
    {
        "true_cow_id": linkage_pilot["cow_id"],
        "predicted_cow_id": neighbor_prediction,
        "best_distance": best_distance,
        "second_distance": second_distance,
    }
).dropna()

neighbor_confidence_results["correct"] = (
    neighbor_confidence_results["true_cow_id"]
    ==
    neighbor_confidence_results["predicted_cow_id"]
)

# A larger margin means the best neighbor is more clearly
# separated from the second-best neighbor.
neighbor_confidence_results["confidence"] = (
    (
        neighbor_confidence_results["second_distance"]
        - neighbor_confidence_results["best_distance"]
    )
    /
    (
        neighbor_confidence_results["second_distance"]
        + 1e-9
    )
)

print(
    "All-neighbor accuracy:",
    round(
        neighbor_confidence_results["correct"].mean(),
        4,
    ),
)

print(
    "Predictions evaluated:",
    len(neighbor_confidence_results),
)

# ---------------------------------------------------------
# Accuracy at different confidence-coverage levels
# ---------------------------------------------------------

ranked_confidence = (
    neighbor_confidence_results
    .sort_values(
        "confidence",
        ascending=False,
    )
    .reset_index(drop=True)
)

coverage_levels = [
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
    0.10,
    0.20,
    0.50,
    1.00,
]

confidence_summary = []

for coverage in coverage_levels:
    number_accepted = max(
        1,
        int(
            len(ranked_confidence)
            * coverage
        ),
    )

    accepted = ranked_confidence.iloc[
        :number_accepted
    ]

    confidence_summary.append(
        {
            "coverage_percent": coverage * 100,
            "accepted_predictions": number_accepted,
            "cow_id_accuracy": accepted[
                "correct"
            ].mean(),
            "minimum_confidence": accepted[
                "confidence"
            ].min(),
        }
    )

confidence_summary = pd.DataFrame(
    confidence_summary
)

display(confidence_summary)

# ---------------------------------------------------------
# Find the largest accepted subset with at least 60% accuracy
# ---------------------------------------------------------

ranked_confidence["accepted_count"] = (
    np.arange(len(ranked_confidence)) + 1
)

ranked_confidence["cumulative_accuracy"] = (
    ranked_confidence["correct"]
    .cumsum()
    / ranked_confidence["accepted_count"]
)

ranked_confidence["coverage_percent"] = (
    ranked_confidence["accepted_count"]
    / len(ranked_confidence)
    * 100
)

possible_thresholds = ranked_confidence.loc[
    (ranked_confidence["accepted_count"] >= 100)
    & (
        ranked_confidence["cumulative_accuracy"]
        >= 0.60
    )
]

if len(possible_thresholds) > 0:
    selected_threshold_row = (
        possible_thresholds.iloc[-1]
    )

    cow_id_confidence_threshold = (
        selected_threshold_row["confidence"]
    )

    print("\n60% accuracy threshold found.")

    print(
        "Confidence threshold:",
        round(cow_id_confidence_threshold, 6),
    )

    print(
        "Accepted predictions:",
        int(
            selected_threshold_row[
                "accepted_count"
            ]
        ),
    )

    print(
        "Coverage:",
        round(
            selected_threshold_row[
                "coverage_percent"
            ],
            2,
        ),
        "%",
    )

    print(
        "Validation accuracy:",
        round(
            selected_threshold_row[
                "cumulative_accuracy"
            ],
            4,
        ),
    )

else:
    cow_id_confidence_threshold = None

    print(
        "\nNo confidence threshold produced "
        "60% accuracy for at least 100 records."
    )

All-neighbor accuracy: 0.0266
Predictions evaluated: 50000


,coverage_percent,accepted_predictions,cow_id_accuracy,minimum_confidence
0,0.1,50,0.08000,0.826082
1,0.5,250,0.06000,0.732633
2,1.0,500,0.05600,0.681103
3,2.0,1000,0.05700,0.615829
4,5.0,2500,0.05160,0.512862
5,10.0,5000,0.04380,0.425041
6,20.0,10000,0.03570,0.321903
7,50.0,25000,0.03280,0.154828
8,100.0,50000,0.02662,0.000000



No confidence threshold produced 60% accuracy for at least 100 records.


### Part 2: Validate surrounding-neighbor agreement

In [22]:
# ---------------------------------------------------------
# PART 2: SURROUNDING KNOWN-NEIGHBOR AGREEMENT
# ---------------------------------------------------------

neighbor_reference = (
    known_context.iloc[train_indices][
        ["source_row", "cow_id"]
    ]
    .dropna()
    .copy()
)

neighbor_validation = (
    known_context.iloc[validation_indices][
        ["source_row", "cow_id"]
    ]
    .dropna()
    .copy()
    .rename(
        columns={
            "cow_id": "true_cow_id"
        }
    )
)

neighbor_reference["cow_id"] = (
    neighbor_reference["cow_id"]
    .astype("string")
)

neighbor_validation["true_cow_id"] = (
    neighbor_validation["true_cow_id"]
    .astype("string")
)

neighbor_reference["source_row"] = pd.to_numeric(
    neighbor_reference["source_row"],
    errors="coerce",
)

neighbor_validation["source_row"] = pd.to_numeric(
    neighbor_validation["source_row"],
    errors="coerce",
)

neighbor_reference = (
    neighbor_reference
    .dropna(subset=["source_row"])
    .drop_duplicates(subset=["source_row"])
)

neighbor_validation = (
    neighbor_validation
    .dropna(subset=["source_row"])
    .drop_duplicates(subset=["source_row"])
)

# Previous known record
previous_reference = (
    neighbor_reference
    .rename(
        columns={
            "source_row": "previous_source_row",
            "cow_id": "previous_cow_id",
        }
    )
    .sort_values("previous_source_row")
)

neighbor_results = pd.merge_asof(
    neighbor_validation.sort_values("source_row"),
    previous_reference,
    left_on="source_row",
    right_on="previous_source_row",
    direction="backward",
    allow_exact_matches=False,
)

# Next known record
next_reference = (
    neighbor_reference
    .rename(
        columns={
            "source_row": "next_source_row",
            "cow_id": "next_cow_id",
        }
    )
    .sort_values("next_source_row")
)

neighbor_results = pd.merge_asof(
    neighbor_results.sort_values("source_row"),
    next_reference,
    left_on="source_row",
    right_on="next_source_row",
    direction="forward",
    allow_exact_matches=False,
)

# Distances to surrounding known records
neighbor_results["previous_gap"] = (
    neighbor_results["source_row"]
    - neighbor_results["previous_source_row"]
)

neighbor_results["next_gap"] = (
    neighbor_results["next_source_row"]
    - neighbor_results["source_row"]
)

neighbor_results["maximum_gap"] = neighbor_results[
    ["previous_gap", "next_gap"]
].max(axis=1)

# Predict only when both surrounding IDs agree
neighbor_results["neighbors_agree"] = (
    neighbor_results["previous_cow_id"].notna()
    & neighbor_results["next_cow_id"].notna()
    & (
        neighbor_results["previous_cow_id"]
        == neighbor_results["next_cow_id"]
    )
)

neighbor_results["predicted_cow_id"] = (
    neighbor_results["previous_cow_id"]
    .where(
        neighbor_results["neighbors_agree"]
    )
)

neighbor_results["correct"] = (
    neighbor_results["predicted_cow_id"]
    == neighbor_results["true_cow_id"]
)

agreed_predictions = neighbor_results.loc[
    neighbor_results["neighbors_agree"]
].copy()

print("Validation records:", len(neighbor_results))

print(
    "Records with agreeing neighbors:",
    len(agreed_predictions),
)

print(
    "Coverage:",
    round(
        len(agreed_predictions)
        / len(neighbor_results)
        * 100,
        4,
    ),
    "%",
)

if len(agreed_predictions) > 0:
    print(
        "Accuracy when neighbors agree:",
        round(
            agreed_predictions["correct"].mean(),
            4,
        ),
    )

# ---------------------------------------------------------
# Determine whether limiting neighbor distance improves accuracy
# ---------------------------------------------------------

gap_limits = [
    1,
    2,
    3,
    5,
    10,
    25,
    50,
    100,
]

gap_summary = []

for gap_limit in gap_limits:
    accepted = neighbor_results.loc[
        neighbor_results["neighbors_agree"]
        & (
            neighbor_results["maximum_gap"]
            <= gap_limit
        )
    ]

    gap_summary.append(
        {
            "maximum_neighbor_gap": gap_limit,
            "accepted_predictions": len(accepted),
            "coverage_percent": (
                len(accepted)
                / len(neighbor_results)
                * 100
            ),
            "cow_id_accuracy": (
                accepted["correct"].mean()
                if len(accepted) > 0
                else np.nan
            ),
        }
    )

gap_summary = pd.DataFrame(gap_summary)

display(gap_summary)

Validation records: 562172
Records with agreeing neighbors: 84
Coverage: 0.0149 %
Accuracy when neighbors agree: 0.0


,maximum_neighbor_gap,accepted_predictions,coverage_percent,cow_id_accuracy
0,1,29,0.005159,0.0
1,2,48,0.008538,0.0
2,3,64,0.011384,0.0
3,5,79,0.014053,0.0
4,10,84,0.014942,0.0
5,25,84,0.014942,0.0
6,50,84,0.014942,0.0
7,100,84,0.014942,0.0


### Part 3: K-nearest-neighbor lactation-number model

In [23]:
# ---------------------------------------------------------
# PART 3: KNN LACTATION-NUMBER MODEL
#
# This predicts lactation number directly from date,
# milking session, and production measurements.
# Cow ID is deliberately excluded.
# ---------------------------------------------------------

from scipy.spatial import cKDTree
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

knn_features = [
    "avg_milk_flow",
    "flow_30_60_session",
    "yield_first_2_min_session",
    "milk_yield",
    "duration_session_sec",
]

knn_train = known_context.iloc[train_indices][
    [
        *knn_features,
        "event_date",
        "milking",
        "lactation_number",
    ]
].dropna(
    subset=["lactation_number", "event_date"]
).copy()

knn_validation = known_context.iloc[validation_indices][
    [
        *knn_features,
        "event_date",
        "milking",
        "lactation_number",
    ]
].dropna(
    subset=["lactation_number", "event_date"]
).copy()

# Use a large but manageable reference sample initially
maximum_reference_rows = 750_000

if len(knn_train) > maximum_reference_rows:
    knn_train = knn_train.sample(
        n=maximum_reference_rows,
        random_state=42,
    )

maximum_validation_rows = 50_000

if len(knn_validation) > maximum_validation_rows:
    knn_validation = knn_validation.sample(
        n=maximum_validation_rows,
        random_state=42,
    )

# Date as a continuous numerical feature
origin_date = pd.Timestamp("1970-01-01")

knn_train["event_date_number"] = (
    pd.to_datetime(
        knn_train["event_date"],
        errors="coerce",
    )
    - origin_date
).dt.days

knn_validation["event_date_number"] = (
    pd.to_datetime(
        knn_validation["event_date"],
        errors="coerce",
    )
    - origin_date
).dt.days

# Encode milking consistently across both datasets
all_milking_values = pd.concat(
    [
        knn_train["milking"],
        knn_validation["milking"],
    ],
    ignore_index=True,
).astype("string").fillna("__MISSING__")

milking_categories = pd.Categorical(
    all_milking_values
).categories

knn_train["milking_code"] = pd.Categorical(
    knn_train["milking"]
    .astype("string")
    .fillna("__MISSING__"),
    categories=milking_categories,
).codes

knn_validation["milking_code"] = pd.Categorical(
    knn_validation["milking"]
    .astype("string")
    .fillna("__MISSING__"),
    categories=milking_categories,
).codes

model_features = [
    *knn_features,
    "event_date_number",
    "milking_code",
]

# Impute and standardize
knn_imputer = SimpleImputer(strategy="median")
knn_scaler = StandardScaler()

X_knn_train = knn_imputer.fit_transform(
    knn_train[model_features]
)

X_knn_validation = knn_imputer.transform(
    knn_validation[model_features]
)

X_knn_train = knn_scaler.fit_transform(
    X_knn_train
).astype("float32")

X_knn_validation = knn_scaler.transform(
    X_knn_validation
).astype("float32")

y_knn_train = (
    knn_train["lactation_number"]
    .round()
    .astype("int16")
    .to_numpy()
)

y_knn_validation = (
    knn_validation["lactation_number"]
    .round()
    .astype("int16")
    .to_numpy()
)

print("Reference records:", len(X_knn_train))
print("Validation records:", len(X_knn_validation))

# Baseline
majority_lactation = pd.Series(
    y_knn_train
).mode().iloc[0]

baseline_predictions = np.full(
    len(y_knn_validation),
    majority_lactation,
)

print(
    "Majority baseline accuracy:",
    round(
        accuracy_score(
            y_knn_validation,
            baseline_predictions,
        ),
        4,
    ),
)

# Retrieve the 50 closest known records once
print("Building neighbor tree...")

lactation_tree = cKDTree(X_knn_train)

maximum_k = 50

neighbor_distances, neighbor_positions = (
    lactation_tree.query(
        X_knn_validation,
        k=maximum_k,
        workers=-1,
    )
)

neighbor_labels = y_knn_train[
    neighbor_positions
]

# ---------------------------------------------------------
# Distance-weighted vote
# ---------------------------------------------------------

def weighted_neighbor_vote(
    labels,
    distances,
):
    predictions = []

    weights = 1 / (distances + 1e-6)

    for row_labels, row_weights in zip(
        labels,
        weights,
    ):
        class_scores = {}

        for label, weight in zip(
            row_labels,
            row_weights,
        ):
            class_scores[label] = (
                class_scores.get(label, 0)
                + weight
            )

        prediction = max(
            class_scores,
            key=class_scores.get,
        )

        predictions.append(prediction)

    return np.asarray(predictions)

k_values = [1, 5, 15, 25, 50]

lactation_knn_results = []

for k in k_values:
    predictions = weighted_neighbor_vote(
        neighbor_labels[:, :k],
        neighbor_distances[:, :k],
    )

    lactation_knn_results.append(
        {
            "neighbors": k,
            "accuracy": accuracy_score(
                y_knn_validation,
                predictions,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_knn_validation,
                predictions,
            ),
            "macro_f1": f1_score(
                y_knn_validation,
                predictions,
                average="macro",
                zero_division=0,
            ),
        }
    )

lactation_knn_results = pd.DataFrame(
    lactation_knn_results
)

display(lactation_knn_results)

best_lactation_result = (
    lactation_knn_results
    .sort_values(
        [
            "balanced_accuracy",
            "macro_f1",
        ],
        ascending=False,
    )
    .iloc[0]
)

print(
    "Best number of neighbors:",
    int(best_lactation_result["neighbors"]),
)

print(
    "Best validation accuracy:",
    round(
        best_lactation_result["accuracy"],
        4,
    ),
)

print(
    "Best balanced accuracy:",
    round(
        best_lactation_result[
            "balanced_accuracy"
        ],
        4,
    ),
)

print(
    "Best macro F1:",
    round(
        best_lactation_result["macro_f1"],
        4,
    ),
)

Reference records: 750000
Validation records: 50000
Majority baseline accuracy: 0.3653
Building neighbor tree...


,neighbors,accuracy,balanced_accuracy,macro_f1
0,1,0.34342,0.131374,0.128321
1,5,0.39888,0.134949,0.137547
2,15,0.44042,0.136846,0.138082
3,25,0.45232,0.137247,0.136032
4,50,0.46066,0.137007,0.136351


Best number of neighbors: 25
Best validation accuracy: 0.4523
Best balanced accuracy: 0.1372
Best macro F1: 0.136


### Part 4: Ordinal-regression lactation model

In [24]:
# ---------------------------------------------------------
# PART 4: LACTATION AS ORDINAL REGRESSION
# ---------------------------------------------------------

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
)

X_ordinal_train = known_context.iloc[
    train_indices
][feature_columns]

y_ordinal_train = (
    known_context.iloc[train_indices][
        "lactation_number"
    ]
    .astype(float)
)

X_ordinal_validation = known_context.iloc[
    validation_indices
][feature_columns]

y_ordinal_validation = (
    known_context.iloc[validation_indices][
        "lactation_number"
    ]
    .astype(float)
)

# Remove any rows where the target itself is missing
train_target_known = y_ordinal_train.notna()
validation_target_known = y_ordinal_validation.notna()

X_ordinal_train = X_ordinal_train.loc[
    train_target_known
]

y_ordinal_train = y_ordinal_train.loc[
    train_target_known
]

X_ordinal_validation = X_ordinal_validation.loc[
    validation_target_known
]

y_ordinal_validation = y_ordinal_validation.loc[
    validation_target_known
]

minimum_lactation = int(
    y_ordinal_train.min()
)

maximum_lactation = int(
    y_ordinal_train.max()
)

ordinal_lactation_model = Pipeline([
    (
        "preprocessor",
        clone(context_preprocessor),
    ),
    (
        "model",
        HistGradientBoostingRegressor(
            learning_rate=0.08,
            max_iter=150,
            max_leaf_nodes=31,
            min_samples_leaf=30,
            l2_regularization=2.0,
            random_state=42,
        ),
    ),
])

print("Fitting ordinal lactation model...")

ordinal_lactation_model.fit(
    X_ordinal_train,
    y_ordinal_train,
)

raw_lactation_predictions = (
    ordinal_lactation_model.predict(
        X_ordinal_validation
    )
)

print(
    "Raw regression MAE:",
    round(
        mean_absolute_error(
            y_ordinal_validation,
            raw_lactation_predictions,
        ),
        4,
    ),
)

# ---------------------------------------------------------
# Test small rounding adjustments on validation data
# ---------------------------------------------------------

rounding_offsets = np.arange(
    -0.5,
    0.51,
    0.1,
)

ordinal_results = []

true_lactation_classes = (
    y_ordinal_validation
    .round()
    .astype(int)
    .to_numpy()
)

for offset in rounding_offsets:
    rounded_predictions = np.rint(
        raw_lactation_predictions + offset
    )

    rounded_predictions = np.clip(
        rounded_predictions,
        minimum_lactation,
        maximum_lactation,
    ).astype(int)

    exact_accuracy = accuracy_score(
        true_lactation_classes,
        rounded_predictions,
    )

    balanced_accuracy = balanced_accuracy_score(
        true_lactation_classes,
        rounded_predictions,
    )

    macro_f1 = f1_score(
        true_lactation_classes,
        rounded_predictions,
        average="macro",
        zero_division=0,
    )

    within_one = (
        np.abs(
            true_lactation_classes
            - rounded_predictions
        )
        <= 1
    ).mean()

    ordinal_results.append(
        {
            "rounding_offset": round(
                float(offset),
                2,
            ),
            "exact_accuracy": exact_accuracy,
            "balanced_accuracy": balanced_accuracy,
            "macro_f1": macro_f1,
            "within_one_lactation": within_one,
        }
    )

ordinal_results = pd.DataFrame(
    ordinal_results
)

display(ordinal_results)

best_ordinal_result = (
    ordinal_results
    .sort_values(
        [
            "exact_accuracy",
            "balanced_accuracy",
        ],
        ascending=False,
    )
    .iloc[0]
)

best_lactation_offset = float(
    best_ordinal_result[
        "rounding_offset"
    ]
)

print(
    "Best rounding offset:",
    best_lactation_offset,
)

print(
    "Best exact accuracy:",
    round(
        best_ordinal_result[
            "exact_accuracy"
        ],
        4,
    ),
)

print(
    "Best balanced accuracy:",
    round(
        best_ordinal_result[
            "balanced_accuracy"
        ],
        4,
    ),
)

print(
    "Best macro F1:",
    round(
        best_ordinal_result[
            "macro_f1"
        ],
        4,
    ),
)

print(
    "Within one lactation:",
    round(
        best_ordinal_result[
            "within_one_lactation"
        ]
        * 100,
        2,
    ),
    "%",
)

Fitting ordinal lactation model...
Raw regression MAE: 0.9928


,rounding_offset,exact_accuracy,balanced_accuracy,macro_f1,within_one_lactation
0,-0.5,0.415223,0.115550,0.104291,0.804330
1,-0.4,0.389448,0.112323,0.101132,0.816762
2,-0.3,0.356697,0.107473,0.094563,0.826727
3,-0.2,0.321352,0.102023,0.085162,0.833051
4,-0.1,0.288365,0.097004,0.074602,0.834645
5,-0.0,0.263976,0.094013,0.067990,0.830623
6,0.1,0.247985,0.093106,0.066050,0.820923
7,0.2,0.235615,0.092992,0.065855,0.805042
8,0.3,0.223051,0.092881,0.065444,0.782600
9,0.4,0.209788,0.092455,0.064063,0.752946


Best rounding offset: -0.5
Best exact accuracy: 0.4152
Best balanced accuracy: 0.1156
Best macro F1: 0.1043
Within one lactation: 80.43 %


In [25]:
import numpy as np
import pandas as pd

# Use columns that remain known when the context variables are missing.
signature_candidates = [
    "event_date",
    "milk_yield",
    "duration_session_sec",
    "milking",
    "flow_30_60_session",
    "yield_first_2_min_session",
    "avg_milk_flow",
]

signature_columns = [
    column
    for column in signature_candidates
    if column in known_context.columns
]

print("Signature columns found:", signature_columns)

signature_data = known_context.copy()

# Standardize the date representation.
if "event_date" in signature_columns:
    signature_data["event_date"] = pd.to_datetime(
        signature_data["event_date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

# Round continuous measurements so tiny floating-point differences
# do not prevent otherwise matching records.
continuous_signature_columns = [
    column
    for column in [
        "milk_yield",
        "duration_session_sec",
        "flow_30_60_session",
        "yield_first_2_min_session",
        "avg_milk_flow",
    ]
    if column in signature_columns
]

for column in continuous_signature_columns:
    signature_data[column] = pd.to_numeric(
        signature_data[column],
        errors="coerce"
    ).round(3)

train_signature = signature_data.iloc[train_indices].copy()
validation_signature = signature_data.iloc[validation_indices].copy()

# Only reference rows with a known lactation number.
train_signature = train_signature[
    train_signature["lactation_number"].notna()
].copy()

# Determine whether each signature maps consistently to one lactation.
signature_lookup = (
    train_signature
    .dropna(subset=signature_columns)
    .groupby(signature_columns, dropna=False)["lactation_number"]
    .agg(
        predicted_lactation=lambda values: values.mode().iloc[0],
        reference_records="size",
        unique_lactations="nunique",
    )
    .reset_index()
)

# Keep unambiguous signatures only.
signature_lookup = signature_lookup[
    signature_lookup["unique_lactations"] == 1
].copy()

matched_validation = (
    validation_signature
    .reset_index(names="original_index")
    .merge(
        signature_lookup,
        on=signature_columns,
        how="left",
    )
)

accepted = matched_validation[
    matched_validation["predicted_lactation"].notna()
].copy()

coverage = len(accepted) / len(validation_signature)

if len(accepted) > 0:
    exact_accuracy = (
        accepted["predicted_lactation"]
        .round()
        .astype(int)
        .eq(
            accepted["lactation_number"]
            .round()
            .astype(int)
        )
        .mean()
    )
else:
    exact_accuracy = np.nan

print("Validation records:", len(validation_signature))
print("Records with an unambiguous signature:", len(accepted))
print("Signature coverage:", round(coverage * 100, 2), "%")
print("Exact lactation accuracy:", round(exact_accuracy, 4))
print(
    "Unambiguous signatures available:",
    len(signature_lookup)
)

Signature columns found: ['event_date', 'milk_yield', 'duration_session_sec', 'milking', 'flow_30_60_session', 'yield_first_2_min_session', 'avg_milk_flow']
Validation records: 562172
Records with an unambiguous signature: 3808
Signature coverage: 0.68 %
Exact lactation accuracy: 0.9968
Unambiguous signatures available: 2615325


In [26]:
# Part 6: Test a less-strict signature by rounding measurements
# to one decimal place instead of three.

relaxed_signature_data = known_context.copy()

relaxed_signature_columns = signature_columns.copy()

if "event_date" in relaxed_signature_columns:
    relaxed_signature_data["event_date"] = pd.to_datetime(
        relaxed_signature_data["event_date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

# Round measurements to one decimal place.
for column in continuous_signature_columns:
    relaxed_signature_data[column] = pd.to_numeric(
        relaxed_signature_data[column],
        errors="coerce"
    ).round(1)

relaxed_train = relaxed_signature_data.iloc[train_indices].copy()
relaxed_validation = relaxed_signature_data.iloc[validation_indices].copy()

relaxed_train = relaxed_train[
    relaxed_train["lactation_number"].notna()
].copy()

print("Building relaxed signature lookup...")

relaxed_lookup = (
    relaxed_train
    .dropna(subset=relaxed_signature_columns)
    .groupby(relaxed_signature_columns, dropna=False)["lactation_number"]
    .agg(
        predicted_lactation=lambda values: values.mode().iloc[0],
        reference_records="size",
        unique_lactations="nunique",
    )
    .reset_index()
)

# Only accept signatures that always corresponded to one lactation
# in the training data.
relaxed_lookup = relaxed_lookup[
    relaxed_lookup["unique_lactations"] == 1
].copy()

relaxed_matches = (
    relaxed_validation
    .reset_index(names="original_index")
    .merge(
        relaxed_lookup,
        on=relaxed_signature_columns,
        how="left",
    )
)

relaxed_accepted = relaxed_matches[
    relaxed_matches["predicted_lactation"].notna()
].copy()

relaxed_coverage = (
    len(relaxed_accepted) / len(relaxed_validation)
)

if len(relaxed_accepted) > 0:
    relaxed_accuracy = (
        relaxed_accepted["predicted_lactation"]
        .round()
        .astype(int)
        .eq(
            relaxed_accepted["lactation_number"]
            .round()
            .astype(int)
        )
        .mean()
    )
else:
    relaxed_accuracy = np.nan

print("\nRELAXED SIGNATURE RESULTS")
print("Validation records:", len(relaxed_validation))
print(
    "Records with an unambiguous signature:",
    len(relaxed_accepted)
)
print(
    "Signature coverage:",
    round(relaxed_coverage * 100, 2),
    "%"
)
print(
    "Exact lactation accuracy:",
    round(relaxed_accuracy, 4)
)
print(
    "Unambiguous signatures available:",
    len(relaxed_lookup)
)

Building relaxed signature lookup...

RELAXED SIGNATURE RESULTS
Validation records: 562172
Records with an unambiguous signature: 3979
Signature coverage: 0.71 %
Exact lactation accuracy: 0.9719
Unambiguous signatures available: 2614660


In [27]:
# Part 7: Fast known-neighbor pilot
# Uses samples while we select the best settings.

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Sample positions from the existing splits.
fast_train_indices = rng.choice(
    np.asarray(train_indices),
    size=min(300_000, len(train_indices)),
    replace=False
)

fast_validation_indices = rng.choice(
    np.asarray(validation_indices),
    size=min(50_000, len(validation_indices)),
    replace=False
)

required_columns = [
    "event_date",
    "milking",
    "milk_yield",
    "duration_session_sec",
    "flow_30_60_session",
    "yield_first_2_min_session",
    "lactation_number",
]

# Load only the columns needed for this experiment.
fast_train = known_context.iloc[
    fast_train_indices
][required_columns].copy()

fast_validation = known_context.iloc[
    fast_validation_indices
][required_columns].copy()


def create_neighbor_bins(data):
    result = pd.DataFrame(index=data.index)

    result["event_date"] = pd.to_datetime(
        data["event_date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    result["milking"] = data["milking"].astype("string")

    result["milk_yield_bin"] = (
        pd.to_numeric(data["milk_yield"], errors="coerce")
        .div(2)
        .round()
        .mul(2)
    )

    result["duration_bin"] = (
        pd.to_numeric(
            data["duration_session_sec"],
            errors="coerce"
        )
        .div(60)
        .round()
        .mul(60)
    )

    result["flow_bin"] = (
        pd.to_numeric(
            data["flow_30_60_session"],
            errors="coerce"
        )
        .div(0.5)
        .round()
        .mul(0.5)
    )

    result["first_2_min_bin"] = (
        pd.to_numeric(
            data["yield_first_2_min_session"],
            errors="coerce"
        ).round()
    )

    result["lactation_number"] = pd.to_numeric(
        data["lactation_number"],
        errors="coerce"
    )

    return result


fast_train = create_neighbor_bins(fast_train)
fast_validation = create_neighbor_bins(fast_validation)

neighbor_keys = [
    "event_date",
    "milking",
    "milk_yield_bin",
    "duration_bin",
    "flow_bin",
    "first_2_min_bin",
]

fast_train = fast_train.dropna(
    subset=neighbor_keys + ["lactation_number"]
)

print("Training sample:", len(fast_train))
print("Validation sample:", len(fast_validation))

# Count each lactation in each neighborhood.
counts = (
    fast_train
    .groupby(
        neighbor_keys + ["lactation_number"],
        observed=True,
        sort=False
    )
    .size()
    .rename("class_count")
    .reset_index()
)

# Calculate neighborhood size.
counts["neighbor_records"] = (
    counts.groupby(
        neighbor_keys,
        observed=True,
        sort=False
    )["class_count"].transform("sum")
)

# Keep the most common lactation in each neighborhood.
lookup = (
    counts
    .sort_values("class_count", ascending=False)
    .drop_duplicates(neighbor_keys)
    .rename(
        columns={
            "lactation_number": "predicted_lactation"
        }
    )
)

lookup["confidence"] = (
    lookup["class_count"] / lookup["neighbor_records"]
)

matches = (
    fast_validation
    .reset_index(drop=True)
    .merge(
        lookup[
            neighbor_keys + [
                "predicted_lactation",
                "neighbor_records",
                "confidence",
            ]
        ],
        on=neighbor_keys,
        how="left"
    )
)

results = []

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    accepted = matches[
        matches["predicted_lactation"].notna()
        & (matches["neighbor_records"] >= 3)
        & (matches["confidence"] >= threshold)
    ]

    accuracy = (
        accepted["predicted_lactation"]
        .round()
        .eq(accepted["lactation_number"].round())
        .mean()
        if len(accepted) > 0
        else np.nan
    )

    results.append({
        "minimum_confidence": threshold,
        "accepted_predictions": len(accepted),
        "coverage_percent": round(
            len(accepted) / len(fast_validation) * 100,
            2
        ),
        "exact_accuracy": round(accuracy, 4),
    })

display(pd.DataFrame(results))

Training sample: 300000
Validation sample: 50000


,minimum_confidence,accepted_predictions,coverage_percent,exact_accuracy
0,0.5,1964,3.93,0.4145
1,0.6,1622,3.24,0.4303
2,0.7,565,1.13,0.5168
3,0.8,362,0.72,0.5414
4,0.9,292,0.58,0.5137


In [31]:
fast_train_indices = rng.choice(
    np.asarray(train_indices),
    size=min(1_000_000, len(train_indices)),
    replace=False
)

fast_validation_indices = rng.choice(
    np.asarray(validation_indices),
    size=min(100_000, len(validation_indices)),
    replace=False
)

In [33]:
# Part 8: Larger fast known-neighbor validation

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Use 1,000,000 training rows and 100,000 validation rows.
fast_train_indices = rng.choice(
    np.asarray(train_indices),
    size=min(1_000_000, len(train_indices)),
    replace=False
)

fast_validation_indices = rng.choice(
    np.asarray(validation_indices),
    size=min(100_000, len(validation_indices)),
    replace=False
)

required_columns = [
    "event_date",
    "milking",
    "milk_yield",
    "duration_session_sec",
    "flow_30_60_session",
    "yield_first_2_min_session",
    "lactation_number",
]

# Select only the necessary columns.
fast_train = known_context.iloc[
    fast_train_indices
][required_columns].copy()

fast_validation = known_context.iloc[
    fast_validation_indices
][required_columns].copy()


def create_neighbor_bins(data):
    result = pd.DataFrame(index=data.index)

    result["event_date"] = pd.to_datetime(
        data["event_date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    result["milking"] = data["milking"].astype("string")

    result["milk_yield_bin"] = (
        pd.to_numeric(
            data["milk_yield"],
            errors="coerce"
        )
        .div(2)
        .round()
        .mul(2)
    )

    result["duration_bin"] = (
        pd.to_numeric(
            data["duration_session_sec"],
            errors="coerce"
        )
        .div(60)
        .round()
        .mul(60)
    )

    result["flow_bin"] = (
        pd.to_numeric(
            data["flow_30_60_session"],
            errors="coerce"
        )
        .div(0.5)
        .round()
        .mul(0.5)
    )

    result["first_2_min_bin"] = (
        pd.to_numeric(
            data["yield_first_2_min_session"],
            errors="coerce"
        ).round()
    )

    result["lactation_number"] = pd.to_numeric(
        data["lactation_number"],
        errors="coerce"
    )

    return result


fast_train = create_neighbor_bins(fast_train)
fast_validation = create_neighbor_bins(fast_validation)

neighbor_keys = [
    "event_date",
    "milking",
    "milk_yield_bin",
    "duration_bin",
    "flow_bin",
    "first_2_min_bin",
]

fast_train = fast_train.dropna(
    subset=neighbor_keys + ["lactation_number"]
)

print("Training sample:", len(fast_train))
print("Validation sample:", len(fast_validation))

# Count each lactation number within each neighborhood.
counts = (
    fast_train
    .groupby(
        neighbor_keys + ["lactation_number"],
        observed=True,
        sort=False
    )
    .size()
    .rename("class_count")
    .reset_index()
)

# Count all training records in each neighborhood.
counts["neighbor_records"] = (
    counts
    .groupby(
        neighbor_keys,
        observed=True,
        sort=False
    )["class_count"]
    .transform("sum")
)

# Select the most common lactation in each neighborhood.
lookup = (
    counts
    .sort_values(
        "class_count",
        ascending=False
    )
    .drop_duplicates(neighbor_keys)
    .rename(
        columns={
            "lactation_number": "predicted_lactation"
        }
    )
)

lookup["confidence"] = (
    lookup["class_count"]
    / lookup["neighbor_records"]
)

# Match validation records to known neighborhoods.
matches = (
    fast_validation
    .reset_index(drop=True)
    .merge(
        lookup[
            neighbor_keys + [
                "predicted_lactation",
                "neighbor_records",
                "confidence",
            ]
        ],
        on=neighbor_keys,
        how="left"
    )
)

# Test narrower confidence thresholds.
results = []

for threshold in [0.70, 0.75, 0.80, 0.85, 0.90]:
    accepted = matches[
        matches["predicted_lactation"].notna()
        & (matches["neighbor_records"] >= 3)
        & (matches["confidence"] >= threshold)
    ].copy()

    if len(accepted) > 0:
        accuracy = (
            accepted["predicted_lactation"]
            .round()
            .eq(
                accepted["lactation_number"].round()
            )
            .mean()
        )
    else:
        accuracy = np.nan

    results.append({
        "minimum_confidence": threshold,
        "accepted_predictions": len(accepted),
        "coverage_percent": round(
            len(accepted) / len(fast_validation) * 100,
            2
        ),
        "exact_accuracy": round(accuracy, 4),
    })

results_table = pd.DataFrame(results)

display(results_table)

Training sample: 1000000
Validation sample: 100000


,minimum_confidence,accepted_predictions,coverage_percent,exact_accuracy
0,0.70,6238,6.24,0.5218
1,0.75,5751,5.75,0.5190
2,0.80,3607,3.61,0.5403
3,0.85,2344,2.34,0.5478
4,0.90,2087,2.09,0.5443


In [34]:
# Part 9: Final unbiased test of the lactation neighbor method

# Training and validation may now be combined because
# the confidence threshold was already selected.
reference_indices = np.concatenate([
    np.asarray(train_indices),
    np.asarray(validation_indices)
])

final_reference = known_context.iloc[
    reference_indices
][required_columns].copy()

final_test = known_context.iloc[
    np.asarray(test_indices)
][required_columns].copy()

final_reference = create_neighbor_bins(final_reference)
final_test = create_neighbor_bins(final_test)

final_reference = final_reference.dropna(
    subset=neighbor_keys + ["lactation_number"]
)

print("Reference records:", len(final_reference))
print("Test records:", len(final_test))

# Count lactation values within each neighborhood.
final_counts = (
    final_reference
    .groupby(
        neighbor_keys + ["lactation_number"],
        observed=True,
        sort=False
    )
    .size()
    .rename("class_count")
    .reset_index()
)

final_counts["neighbor_records"] = (
    final_counts
    .groupby(
        neighbor_keys,
        observed=True,
        sort=False
    )["class_count"]
    .transform("sum")
)

# Select the most common lactation for each neighborhood.
final_lookup = (
    final_counts
    .sort_values(
        "class_count",
        ascending=False
    )
    .drop_duplicates(neighbor_keys)
    .rename(
        columns={
            "lactation_number": "predicted_lactation"
        }
    )
)

final_lookup["confidence"] = (
    final_lookup["class_count"]
    / final_lookup["neighbor_records"]
)

# Match the untouched test data.
final_matches = (
    final_test
    .reset_index(drop=True)
    .merge(
        final_lookup[
            neighbor_keys + [
                "predicted_lactation",
                "neighbor_records",
                "confidence",
            ]
        ],
        on=neighbor_keys,
        how="left"
    )
)

# Apply the selected validation threshold.
final_accepted = final_matches[
    final_matches["predicted_lactation"].notna()
    & (final_matches["neighbor_records"] >= 3)
    & (final_matches["confidence"] >= 0.85)
].copy()

final_accuracy = (
    final_accepted["predicted_lactation"]
    .round()
    .eq(
        final_accepted["lactation_number"].round()
    )
    .mean()
)

final_coverage = (
    len(final_accepted) / len(final_test) * 100
)

print("\nFINAL LACTATION TEST RESULTS")
print("Accepted predictions:", len(final_accepted))
print("Test coverage:", round(final_coverage, 2), "%")
print("Exact test accuracy:", round(final_accuracy, 4))

Reference records: 3187062
Test records: 561701

FINAL LACTATION TEST RESULTS
Accepted predictions: 15260
Test coverage: 2.72 %
Exact test accuracy: 0.5547


In [36]:
# Part 10: Weekly-neighbor lactation pilot
# Uses training and validation only—not the test set.

weekly_columns = [
    "event_date",
    "milking",
    "milk_yield",
    "duration_session_sec",
    "flow_30_60_session",
    "yield_first_2_min_session",
    "lactation_number",
]

weekly_train = known_context.iloc[
    fast_train_indices
][weekly_columns].copy()

weekly_validation = known_context.iloc[
    fast_validation_indices
][weekly_columns].copy()


def create_weekly_bins(data):
    result = pd.DataFrame(index=data.index)

    event_dates = pd.to_datetime(
        data["event_date"],
        errors="coerce"
    )

    # Match records occurring within the same calendar week.
    result["event_week"] = (
        event_dates.dt.to_period("W").astype("string")
    )

    result["milking"] = data["milking"].astype("string")

    # Use narrower measurement bins because the date window
    # is now wider.
    result["milk_yield_bin"] = (
        pd.to_numeric(data["milk_yield"], errors="coerce")
        .round()
    )

    result["duration_bin"] = (
        pd.to_numeric(
            data["duration_session_sec"],
            errors="coerce"
        )
        .div(30)
        .round()
        .mul(30)
    )

    result["flow_bin"] = (
        pd.to_numeric(
            data["flow_30_60_session"],
            errors="coerce"
        )
        .div(0.25)
        .round()
        .mul(0.25)
    )

    result["first_2_min_bin"] = (
        pd.to_numeric(
            data["yield_first_2_min_session"],
            errors="coerce"
        )
        .div(0.5)
        .round()
        .mul(0.5)
    )

    result["lactation_number"] = pd.to_numeric(
        data["lactation_number"],
        errors="coerce"
    )

    return result


weekly_train = create_weekly_bins(weekly_train)
weekly_validation = create_weekly_bins(weekly_validation)

weekly_keys = [
    "event_week",
    "milking",
    "milk_yield_bin",
    "duration_bin",
    "flow_bin",
    "first_2_min_bin",
]

weekly_train = weekly_train.dropna(
    subset=weekly_keys + ["lactation_number"]
)

weekly_counts = (
    weekly_train
    .groupby(
        weekly_keys + ["lactation_number"],
        observed=True,
        sort=False
    )
    .size()
    .rename("class_count")
    .reset_index()
)

weekly_counts["neighbor_records"] = (
    weekly_counts
    .groupby(
        weekly_keys,
        observed=True,
        sort=False
    )["class_count"]
    .transform("sum")
)

weekly_lookup = (
    weekly_counts
    .sort_values("class_count", ascending=False)
    .drop_duplicates(weekly_keys)
    .rename(
        columns={
            "lactation_number": "predicted_lactation"
        }
    )
)

weekly_lookup["confidence"] = (
    weekly_lookup["class_count"]
    / weekly_lookup["neighbor_records"]
)

weekly_matches = (
    weekly_validation
    .reset_index(drop=True)
    .merge(
        weekly_lookup[
            weekly_keys + [
                "predicted_lactation",
                "neighbor_records",
                "confidence",
            ]
        ],
        on=weekly_keys,
        how="left"
    )
)

weekly_results = []

for threshold in [0.60, 0.70, 0.75, 0.80, 0.85, 0.90]:
    accepted = weekly_matches[
        weekly_matches["predicted_lactation"].notna()
        & (weekly_matches["neighbor_records"] >= 3)
        & (weekly_matches["confidence"] >= threshold)
    ]

    accuracy = (
        accepted["predicted_lactation"]
        .round()
        .eq(accepted["lactation_number"].round())
        .mean()
        if len(accepted) > 0
        else np.nan
    )

    weekly_results.append({
        "minimum_confidence": threshold,
        "accepted_predictions": len(accepted),
        "coverage_percent": round(
            len(accepted) / len(weekly_validation) * 100,
            2
        ),
        "exact_accuracy": round(accuracy, 4),
    })

display(pd.DataFrame(weekly_results))

,minimum_confidence,accepted_predictions,coverage_percent,exact_accuracy
0,0.60,14436,14.44,0.4975
1,0.70,6806,6.81,0.5614
2,0.75,6139,6.14,0.5590
3,0.80,3941,3.94,0.5849
4,0.85,2654,2.65,0.5746
5,0.90,2296,2.30,0.5753


In [37]:
# Part 11: Final weekly-neighbor lactation test

weekly_reference_indices = np.concatenate([
    np.asarray(train_indices),
    np.asarray(validation_indices)
])

weekly_reference = known_context.iloc[
    weekly_reference_indices
][weekly_columns].copy()

weekly_test = known_context.iloc[
    np.asarray(test_indices)
][weekly_columns].copy()

weekly_reference = create_weekly_bins(weekly_reference)
weekly_test = create_weekly_bins(weekly_test)

weekly_reference = weekly_reference.dropna(
    subset=weekly_keys + ["lactation_number"]
)

print("Reference records:", len(weekly_reference))
print("Test records:", len(weekly_test))

# Build the weekly lookup using training and validation data.
test_weekly_counts = (
    weekly_reference
    .groupby(
        weekly_keys + ["lactation_number"],
        observed=True,
        sort=False
    )
    .size()
    .rename("class_count")
    .reset_index()
)

test_weekly_counts["neighbor_records"] = (
    test_weekly_counts
    .groupby(
        weekly_keys,
        observed=True,
        sort=False
    )["class_count"]
    .transform("sum")
)

test_weekly_lookup = (
    test_weekly_counts
    .sort_values("class_count", ascending=False)
    .drop_duplicates(weekly_keys)
    .rename(
        columns={
            "lactation_number": "predicted_lactation"
        }
    )
)

test_weekly_lookup["confidence"] = (
    test_weekly_lookup["class_count"]
    / test_weekly_lookup["neighbor_records"]
)

# Match the untouched test records.
test_weekly_matches = (
    weekly_test
    .reset_index(drop=True)
    .merge(
        test_weekly_lookup[
            weekly_keys + [
                "predicted_lactation",
                "neighbor_records",
                "confidence",
            ]
        ],
        on=weekly_keys,
        how="left"
    )
)

# Apply the threshold selected using validation data.
weekly_test_accepted = test_weekly_matches[
    test_weekly_matches["predicted_lactation"].notna()
    & (test_weekly_matches["neighbor_records"] >= 3)
    & (test_weekly_matches["confidence"] >= 0.70)
].copy()

weekly_test_accuracy = (
    weekly_test_accepted["predicted_lactation"]
    .round()
    .eq(
        weekly_test_accepted["lactation_number"].round()
    )
    .mean()
)

weekly_test_coverage = (
    len(weekly_test_accepted)
    / len(weekly_test)
    * 100
)

print("\nFINAL WEEKLY LACTATION TEST RESULTS")
print(
    "Accepted predictions:",
    len(weekly_test_accepted)
)
print(
    "Test coverage:",
    round(weekly_test_coverage, 2),
    "%"
)
print(
    "Exact test accuracy:",
    round(weekly_test_accuracy, 4)
)

Reference records: 3187062
Test records: 561701

FINAL WEEKLY LACTATION TEST RESULTS
Accepted predictions: 56003
Test coverage: 9.97 %
Exact test accuracy: 0.5764


In [38]:
# Part 12: Find the broadest reliable weekly threshold
# Uses full training and validation sets, but not the test set.

full_weekly_train = known_context.iloc[
    np.asarray(train_indices)
][weekly_columns].copy()

full_weekly_validation = known_context.iloc[
    np.asarray(validation_indices)
][weekly_columns].copy()

full_weekly_train = create_weekly_bins(full_weekly_train)
full_weekly_validation = create_weekly_bins(
    full_weekly_validation
)

full_weekly_train = full_weekly_train.dropna(
    subset=weekly_keys + ["lactation_number"]
)

# Build lookup from the complete training set.
full_counts = (
    full_weekly_train
    .groupby(
        weekly_keys + ["lactation_number"],
        observed=True,
        sort=False
    )
    .size()
    .rename("class_count")
    .reset_index()
)

full_counts["neighbor_records"] = (
    full_counts
    .groupby(
        weekly_keys,
        observed=True,
        sort=False
    )["class_count"]
    .transform("sum")
)

full_lookup = (
    full_counts
    .sort_values("class_count", ascending=False)
    .drop_duplicates(weekly_keys)
    .rename(
        columns={
            "lactation_number": "predicted_lactation"
        }
    )
)

full_lookup["confidence"] = (
    full_lookup["class_count"]
    / full_lookup["neighbor_records"]
)

full_validation_matches = (
    full_weekly_validation
    .reset_index(drop=True)
    .merge(
        full_lookup[
            weekly_keys + [
                "predicted_lactation",
                "neighbor_records",
                "confidence",
            ]
        ],
        on=weekly_keys,
        how="left"
    )
)

threshold_results = []

for threshold in [0.50, 0.55, 0.60, 0.65, 0.70]:
    accepted = full_validation_matches[
        full_validation_matches["predicted_lactation"].notna()
        & (full_validation_matches["neighbor_records"] >= 3)
        & (
            full_validation_matches["confidence"]
            >= threshold
        )
    ]

    accuracy = (
        accepted["predicted_lactation"]
        .round()
        .eq(accepted["lactation_number"].round())
        .mean()
        if len(accepted) > 0
        else np.nan
    )

    threshold_results.append({
        "minimum_confidence": threshold,
        "accepted_predictions": len(accepted),
        "coverage_percent": round(
            len(accepted)
            / len(full_weekly_validation)
            * 100,
            2
        ),
        "exact_accuracy": round(accuracy, 4),
    })

display(pd.DataFrame(threshold_results))

,minimum_confidence,accepted_predictions,coverage_percent,exact_accuracy
0,0.50,188469,33.53,0.4797
1,0.55,136740,24.32,0.5117
2,0.60,119219,21.21,0.5136
3,0.65,92605,16.47,0.5187
4,0.70,54252,9.65,0.5708


In [39]:
# Part 13: Test the validation-selected 0.55 threshold

broad_test_accepted = test_weekly_matches[
    test_weekly_matches["predicted_lactation"].notna()
    & (test_weekly_matches["neighbor_records"] >= 3)
    & (test_weekly_matches["confidence"] >= 0.55)
].copy()

broad_test_accuracy = (
    broad_test_accepted["predicted_lactation"]
    .round()
    .eq(
        broad_test_accepted["lactation_number"].round()
    )
    .mean()
)

broad_test_coverage = (
    len(broad_test_accepted)
    / len(test_weekly_matches)
    * 100
)

print("BROAD WEEKLY LACTATION TEST RESULTS")
print(
    "Accepted predictions:",
    len(broad_test_accepted)
)
print(
    "Test coverage:",
    round(broad_test_coverage, 2),
    "%"
)
print(
    "Exact test accuracy:",
    round(broad_test_accuracy, 4)
)

BROAD WEEKLY LACTATION TEST RESULTS
Accepted predictions: 145173
Test coverage: 25.85 %
Exact test accuracy: 0.5135


In [40]:
# Part 14: Days-in-milk weekly-neighbor pilot

dim_columns = weekly_columns + ["days_in_milk"]

dim_train_raw = known_context.iloc[
    fast_train_indices
][dim_columns].copy()

dim_validation_raw = known_context.iloc[
    fast_validation_indices
][dim_columns].copy()

# Create the same weekly measurement neighborhoods.
dim_train = create_weekly_bins(dim_train_raw)
dim_validation = create_weekly_bins(dim_validation_raw)

# Add the days-in-milk target back to the binned data.
dim_train["days_in_milk"] = pd.to_numeric(
    dim_train_raw["days_in_milk"],
    errors="coerce"
)

dim_validation["days_in_milk"] = pd.to_numeric(
    dim_validation_raw["days_in_milk"],
    errors="coerce"
)

dim_train = dim_train.dropna(
    subset=weekly_keys + ["days_in_milk"]
)

# Calculate the typical days in milk and its variability
# within every known neighborhood.
dim_lookup = (
    dim_train
    .groupby(
        weekly_keys,
        observed=True,
        sort=False
    )["days_in_milk"]
    .agg(
        predicted_days="median",
        neighbor_records="size",
        neighbor_std="std",
    )
    .reset_index()
)

dim_matches = (
    dim_validation
    .reset_index(drop=True)
    .merge(
        dim_lookup,
        on=weekly_keys,
        how="left"
    )
)

dim_results = []

# Smaller standard deviation means the neighborhood
# has more consistent days-in-milk values.
for maximum_std in [7, 14, 30, 60]:
    accepted = dim_matches[
        dim_matches["predicted_days"].notna()
        & (dim_matches["neighbor_records"] >= 3)
        & (dim_matches["neighbor_std"] <= maximum_std)
    ].copy()

    if len(accepted) > 0:
        absolute_error = (
            accepted["days_in_milk"]
            - accepted["predicted_days"]
        ).abs()

        mae = absolute_error.mean()
        within_7 = (absolute_error <= 7).mean()
        within_14 = (absolute_error <= 14).mean()
        within_30 = (absolute_error <= 30).mean()
    else:
        mae = np.nan
        within_7 = np.nan
        within_14 = np.nan
        within_30 = np.nan

    dim_results.append({
        "maximum_neighbor_std": maximum_std,
        "accepted_predictions": len(accepted),
        "coverage_percent": round(
            len(accepted) / len(dim_validation) * 100,
            2
        ),
        "MAE_days": round(mae, 2),
        "within_7_days": round(within_7, 4),
        "within_14_days": round(within_14, 4),
        "within_30_days": round(within_30, 4),
    })

display(pd.DataFrame(dim_results))

,maximum_neighbor_std,accepted_predictions,coverage_percent,MAE_days,within_7_days,within_14_days,within_30_days
0,7,100,0.10,91.96,0.1100,0.1600,0.3000
1,14,376,0.38,82.61,0.0824,0.1543,0.2926
2,30,1662,1.66,75.76,0.0788,0.1534,0.3045
3,60,7712,7.71,75.11,0.0727,0.1343,0.2737


In [41]:
# Part 15: Two-stage weekly-neighbor model
# Stage 1: predict lactation number
# Stage 2: predict days in milk within that predicted lactation

import numpy as np
import pandas as pd

# Prevent missing variables from becoming lookup keys
base_keys = [
    column for column in weekly_keys
    if column not in {
        "cow_id",
        "lactation_number",
        "days_in_milk",
        "reproduction_status"
    }
]

# ---------------------------------------------------------
# 1. Build lactation-number lookup from training data
# ---------------------------------------------------------

lactation_counts = (
    dim_train
    .dropna(subset=base_keys + ["lactation_number"])
    .groupby(
        base_keys + ["lactation_number"],
        observed=True,
        sort=False
    )
    .size()
    .rename("class_count")
    .reset_index()
)

lactation_counts["total_count"] = (
    lactation_counts
    .groupby(base_keys, observed=True)["class_count"]
    .transform("sum")
)

lactation_counts["lactation_confidence"] = (
    lactation_counts["class_count"]
    / lactation_counts["total_count"]
)

# Keep the most common lactation for each signature
lactation_lookup = (
    lactation_counts
    .sort_values(
        ["class_count", "lactation_number"],
        ascending=[False, True]
    )
    .drop_duplicates(base_keys)
    .rename(
        columns={
            "lactation_number": "predicted_lactation"
        }
    )
)

lactation_lookup = lactation_lookup[
    base_keys
    + [
        "predicted_lactation",
        "lactation_confidence",
        "total_count"
    ]
]

# ---------------------------------------------------------
# 2. Build days-in-milk lookup within each lactation
# ---------------------------------------------------------

dim_lookup_stage2 = (
    dim_train
    .dropna(
        subset=base_keys
        + ["lactation_number", "days_in_milk"]
    )
    .groupby(
        base_keys + ["lactation_number"],
        observed=True,
        sort=False
    )["days_in_milk"]
    .agg(
        predicted_days="median",
        neighbor_records="size",
        neighbor_std="std"
    )
    .reset_index()
    .rename(
        columns={
            "lactation_number": "predicted_lactation"
        }
    )
)

# ---------------------------------------------------------
# 3. Apply both stages to validation data
# ---------------------------------------------------------

two_stage_matches = (
    dim_validation
    .reset_index(drop=True)
    .merge(
        lactation_lookup,
        on=base_keys,
        how="left"
    )
    .merge(
        dim_lookup_stage2,
        on=base_keys + ["predicted_lactation"],
        how="left"
    )
)

# ---------------------------------------------------------
# 4. Test confidence and variability thresholds
# ---------------------------------------------------------

two_stage_results = []

for minimum_confidence in [0.50, 0.60, 0.70]:
    for maximum_std in [30, 60, 90]:

        accepted = two_stage_matches[
            two_stage_matches["predicted_days"].notna()
            & two_stage_matches["days_in_milk"].notna()
            & (
                two_stage_matches["lactation_confidence"]
                >= minimum_confidence
            )
            & (
                two_stage_matches["neighbor_std"].isna()
                | (
                    two_stage_matches["neighbor_std"]
                    <= maximum_std
                )
            )
        ].copy()

        if len(accepted) == 0:
            continue

        errors = (
            accepted["predicted_days"]
            - accepted["days_in_milk"]
        ).abs()

        two_stage_results.append({
            "minimum_confidence": minimum_confidence,
            "maximum_neighbor_std": maximum_std,
            "accepted_predictions": len(accepted),
            "coverage_percent": round(
                len(accepted) / len(two_stage_matches) * 100,
                2
            ),
            "MAE_days": round(errors.mean(), 2),
            "within_7_days": round((errors <= 7).mean(), 4),
            "within_14_days": round((errors <= 14).mean(), 4),
            "within_30_days": round((errors <= 30).mean(), 4)
        })

two_stage_results = pd.DataFrame(two_stage_results)

display(
    two_stage_results.sort_values(
        ["within_30_days", "coverage_percent"],
        ascending=[False, False]
    )
)

,minimum_confidence,maximum_neighbor_std,accepted_predictions,coverage_percent,MAE_days,within_7_days,within_14_days,within_30_days
4,0.6,60,27752,27.75,93.03,0.0747,0.1291,0.2493
7,0.7,60,24293,24.29,94.56,0.0759,0.1292,0.2484
3,0.6,30,23445,23.45,95.49,0.0775,0.1308,0.2481
6,0.7,30,21743,21.74,96.26,0.0776,0.1298,0.2469
5,0.6,90,32446,32.45,91.46,0.0719,0.1260,0.2461
8,0.7,90,27170,27.17,93.28,0.0739,0.1269,0.2458
1,0.5,60,38524,38.52,93.53,0.0728,0.1270,0.2457
0,0.5,30,32779,32.78,95.96,0.0749,0.1281,0.2437
2,0.5,90,44773,44.77,91.90,0.0705,0.1247,0.2434


In [42]:
# Part 16: Date-aware days-in-milk regression pilot

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Retrieve the original rows used in the previous pilot
dim_model_train = known_context.loc[dim_train_raw.index].copy()
dim_model_validation = known_context.loc[dim_validation_raw.index].copy()

target = "days_in_milk"

# Only use variables that remain available when context is missing
candidate_features = [
    "source_row",
    "event_date",
    "avg_milk_flow",
    "flow_30_60_session",
    "yield_first_2_min_session",
    "milk_yield",
    "duration_session_sec",
    "milking"
]

available_features = [
    column
    for column in candidate_features
    if column in dim_model_train.columns
]

print("Features used:", available_features)


def make_dim_features(data):
    features = pd.DataFrame(index=data.index)

    # Date features
    dates = pd.to_datetime(
        data["event_date"],
        errors="coerce"
    )

    features["event_year"] = dates.dt.year
    features["event_month"] = dates.dt.month
    features["event_day"] = dates.dt.day
    features["day_of_year"] = dates.dt.dayofyear
    features["day_of_week"] = dates.dt.dayofweek

    # Cyclical date features
    features["year_sin"] = np.sin(
        2 * np.pi * features["day_of_year"] / 365.25
    )
    features["year_cos"] = np.cos(
        2 * np.pi * features["day_of_year"] / 365.25
    )

    # Convert production variables to numeric
    numeric_columns = [
        "source_row",
        "avg_milk_flow",
        "flow_30_60_session",
        "yield_first_2_min_session",
        "milk_yield",
        "duration_session_sec",
        "milking"
    ]

    for column in numeric_columns:
        if column in data.columns:
            features[column] = pd.to_numeric(
                data[column],
                errors="coerce"
            )

    # Additional biologically relevant relationships
    if {
        "milk_yield",
        "duration_session_sec"
    }.issubset(features.columns):
        features["yield_per_second"] = (
            features["milk_yield"]
            / features["duration_session_sec"].replace(0, np.nan)
        )

    if {
        "yield_first_2_min_session",
        "milk_yield"
    }.issubset(features.columns):
        features["first_2_min_fraction"] = (
            features["yield_first_2_min_session"]
            / features["milk_yield"].replace(0, np.nan)
        )

    if {
        "flow_30_60_session",
        "avg_milk_flow"
    }.issubset(features.columns):
        features["flow_ratio"] = (
            features["flow_30_60_session"]
            / features["avg_milk_flow"].replace(0, np.nan)
        )

    return features.replace([np.inf, -np.inf], np.nan)


# Construct feature matrices
X_dim_train = make_dim_features(dim_model_train)
X_dim_validation = make_dim_features(dim_model_validation)

y_dim_train = pd.to_numeric(
    dim_model_train[target],
    errors="coerce"
)

y_dim_validation = pd.to_numeric(
    dim_model_validation[target],
    errors="coerce"
)

# Remove rows with unavailable targets
train_mask = y_dim_train.notna()
validation_mask = y_dim_validation.notna()

X_dim_train = X_dim_train.loc[train_mask]
y_dim_train = y_dim_train.loc[train_mask]

X_dim_validation = X_dim_validation.loc[validation_mask]
y_dim_validation = y_dim_validation.loc[validation_mask]

# Limit training size so it runs quickly
maximum_training_rows = 400_000

if len(X_dim_train) > maximum_training_rows:
    sampled_indices = X_dim_train.sample(
        n=maximum_training_rows,
        random_state=42
    ).index

    X_dim_train = X_dim_train.loc[sampled_indices]
    y_dim_train = y_dim_train.loc[sampled_indices]

# Median imputation calculated from training only
training_medians = X_dim_train.median(numeric_only=True)

X_dim_train = X_dim_train.fillna(training_medians)
X_dim_validation = X_dim_validation.fillna(training_medians)

print("Training records:", len(X_dim_train))
print("Validation records:", len(X_dim_validation))
print("Fitting date-aware model...")

dim_model = HistGradientBoostingRegressor(
    loss="absolute_error",
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=40,
    l2_regularization=1.0,
    random_state=42
)

dim_model.fit(X_dim_train, y_dim_train)

dim_predictions = dim_model.predict(X_dim_validation)

# Prevent biologically impossible negative predictions
dim_predictions = np.maximum(dim_predictions, 0)

dim_errors = np.abs(
    y_dim_validation.to_numpy() - dim_predictions
)

dim_mae = mean_absolute_error(
    y_dim_validation,
    dim_predictions
)

dim_rmse = np.sqrt(
    mean_squared_error(
        y_dim_validation,
        dim_predictions
    )
)

dim_r2 = r2_score(
    y_dim_validation,
    dim_predictions
)

print("\nDATE-AWARE DAYS-IN-MILK RESULTS")
print("MAE:", round(dim_mae, 2), "days")
print("RMSE:", round(dim_rmse, 2), "days")
print("R²:", round(dim_r2, 4))
print("Within 7 days:", round((dim_errors <= 7).mean() * 100, 2), "%")
print("Within 14 days:", round((dim_errors <= 14).mean() * 100, 2), "%")
print("Within 30 days:", round((dim_errors <= 30).mean() * 100, 2), "%")

Features used: ['source_row', 'event_date', 'avg_milk_flow', 'flow_30_60_session', 'yield_first_2_min_session', 'milk_yield', 'duration_session_sec', 'milking']
Training records: 400000
Validation records: 100000
Fitting date-aware model...

DATE-AWARE DAYS-IN-MILK RESULTS
MAE: 71.09 days
RMSE: 92.95 days
R²: 0.2278
Within 7 days: 6.65 %
Within 14 days: 13.25 %
Within 30 days: 27.88 %


In [43]:
# Part 17: Predict 60-day days-in-milk stages

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

bin_width = 60

# Convert days in milk into 60-day stages
y_train_stage = (
    y_dim_train // bin_width
).astype(int)

y_validation_stage = (
    y_dim_validation // bin_width
).astype(int)

# Use a smaller training sample to keep runtime reasonable
maximum_stage_training_rows = 250_000

if len(X_dim_train) > maximum_stage_training_rows:
    stage_indices = X_dim_train.sample(
        n=maximum_stage_training_rows,
        random_state=42
    ).index

    X_stage_train = X_dim_train.loc[stage_indices]
    y_stage_train = y_train_stage.loc[stage_indices]
    y_days_stage_train = y_dim_train.loc[stage_indices]

else:
    X_stage_train = X_dim_train
    y_stage_train = y_train_stage
    y_days_stage_train = y_dim_train

print("Stage-model training records:", len(X_stage_train))
print("Fitting days-in-milk stage classifier...")

stage_model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=120,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    l2_regularization=1.0,
    random_state=42
)

stage_model.fit(
    X_stage_train,
    y_stage_train
)

predicted_stages = stage_model.predict(
    X_dim_validation
)

# Calculate the median days-in-milk for each training stage
stage_medians = (
    pd.DataFrame({
        "stage": y_stage_train,
        "days_in_milk": y_days_stage_train
    })
    .groupby("stage")["days_in_milk"]
    .median()
)

# Convert each predicted stage back to a numeric estimate
stage_predictions = (
    pd.Series(predicted_stages)
    .map(stage_medians)
    .to_numpy()
)

stage_errors = np.abs(
    y_dim_validation.to_numpy()
    - stage_predictions
)

stage_accuracy = accuracy_score(
    y_validation_stage,
    predicted_stages
)

stage_mae = stage_errors.mean()

# Majority-stage baseline
majority_stage = y_stage_train.mode().iloc[0]

baseline_accuracy = (
    y_validation_stage == majority_stage
).mean()

print("\nDAYS-IN-MILK STAGE RESULTS")
print("Majority-stage baseline:", round(baseline_accuracy, 4))
print("Stage classification accuracy:", round(stage_accuracy, 4))
print("Numeric MAE:", round(stage_mae, 2), "days")
print(
    "Within 7 days:",
    round((stage_errors <= 7).mean() * 100, 2),
    "%"
)
print(
    "Within 14 days:",
    round((stage_errors <= 14).mean() * 100, 2),
    "%"
)
print(
    "Within 30 days:",
    round((stage_errors <= 30).mean() * 100, 2),
    "%"
)

Stage-model training records: 250000
Fitting days-in-milk stage classifier...

DAYS-IN-MILK STAGE RESULTS
Majority-stage baseline: 0.1969
Stage classification accuracy: 0.2913
Numeric MAE: 77.83 days
Within 7 days: 7.75 %
Within 14 days: 14.64 %
Within 30 days: 29.79 %


In [44]:
# Part 18: Days-in-milk confidence and coverage test

import numpy as np
import pandas as pd

# Probability assigned to every possible stage
stage_probabilities = stage_model.predict_proba(
    X_dim_validation
)

# Confidence of the selected stage
stage_confidence = stage_probabilities.max(axis=1)

confidence_results = []

for minimum_confidence in [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]:
    accepted_mask = (
        stage_confidence >= minimum_confidence
    )

    accepted_count = accepted_mask.sum()

    if accepted_count == 0:
        continue

    accepted_true_stages = (
        y_validation_stage.to_numpy()[accepted_mask]
    )

    accepted_predicted_stages = (
        predicted_stages[accepted_mask]
    )

    accepted_errors = (
        stage_errors[accepted_mask]
    )

    confidence_results.append({
        "minimum_confidence": minimum_confidence,
        "accepted_predictions": accepted_count,
        "coverage_percent": round(
            accepted_count
            / len(y_dim_validation)
            * 100,
            2
        ),
        "stage_accuracy": round(
            (
                accepted_true_stages
                == accepted_predicted_stages
            ).mean(),
            4
        ),
        "MAE_days": round(
            accepted_errors.mean(),
            2
        ),
        "within_7_days": round(
            (accepted_errors <= 7).mean(),
            4
        ),
        "within_14_days": round(
            (accepted_errors <= 14).mean(),
            4
        ),
        "within_30_days": round(
            (accepted_errors <= 30).mean(),
            4
        )
    })

confidence_results = pd.DataFrame(
    confidence_results
)

display(confidence_results)

,minimum_confidence,accepted_predictions,coverage_percent,stage_accuracy,MAE_days,within_7_days,within_14_days,within_30_days
0,0.3,32212,32.21,0.3630,62.87,0.0970,0.1860,0.3703
1,0.4,7413,7.41,0.4396,47.80,0.1224,0.2287,0.4523
2,0.5,820,0.82,0.5354,51.90,0.1366,0.2780,0.5427
3,0.6,148,0.15,0.6216,48.92,0.1554,0.3311,0.6351
4,0.7,10,0.01,0.9000,18.70,0.3000,0.7000,0.9000


In [45]:
# Part 19: Refine days-in-milk confidence threshold

refined_results = []

for minimum_confidence in np.arange(
    0.40,
    0.501,
    0.01
):
    accepted_mask = (
        stage_confidence >= minimum_confidence
    )

    accepted_count = accepted_mask.sum()

    if accepted_count == 0:
        continue

    accepted_errors = stage_errors[
        accepted_mask
    ]

    accepted_true_stages = (
        y_validation_stage
        .to_numpy()[accepted_mask]
    )

    accepted_predicted_stages = (
        predicted_stages[accepted_mask]
    )

    refined_results.append({
        "minimum_confidence": round(
            minimum_confidence,
            2
        ),
        "accepted_predictions": accepted_count,
        "coverage_percent": round(
            accepted_count
            / len(y_dim_validation)
            * 100,
            2
        ),
        "stage_accuracy": round(
            (
                accepted_true_stages
                == accepted_predicted_stages
            ).mean(),
            4
        ),
        "MAE_days": round(
            accepted_errors.mean(),
            2
        ),
        "within_30_days": round(
            (accepted_errors <= 30).mean(),
            4
        )
    })

refined_results = pd.DataFrame(
    refined_results
)

display(refined_results)

eligible_thresholds = refined_results[
    refined_results["within_30_days"] >= 0.50
].sort_values(
    "coverage_percent",
    ascending=False
)

print("\nTHRESHOLDS ACHIEVING AT LEAST 50%:")
display(eligible_thresholds)

,minimum_confidence,accepted_predictions,coverage_percent,stage_accuracy,MAE_days,within_30_days
0,0.40,7413,7.41,0.4396,47.80,0.4523
1,0.41,6277,6.28,0.4461,47.34,0.4587
2,0.42,5154,5.15,0.4532,47.28,0.4657
3,0.43,3995,4.00,0.4626,47.23,0.4746
4,0.44,3067,3.07,0.4803,47.48,0.4910
5,0.45,2390,2.39,0.4979,47.58,0.5079
6,0.46,1917,1.92,0.5065,48.13,0.5159
7,0.47,1586,1.59,0.5101,49.27,0.5195
8,0.48,1305,1.31,0.5157,50.16,0.5249
9,0.49,1045,1.04,0.5225,50.76,0.5330



THRESHOLDS ACHIEVING AT LEAST 50%:


,minimum_confidence,accepted_predictions,coverage_percent,stage_accuracy,MAE_days,within_30_days
5,0.45,2390,2.39,0.4979,47.58,0.5079
6,0.46,1917,1.92,0.5065,48.13,0.5159
7,0.47,1586,1.59,0.5101,49.27,0.5195
8,0.48,1305,1.31,0.5157,50.16,0.5249
9,0.49,1045,1.04,0.5225,50.76,0.5330
10,0.50,820,0.82,0.5354,51.90,0.5427


In [46]:
# Part 20: Final days-in-milk test
# Threshold selected using validation results only

selected_dim_threshold = 0.45

# Keep test rows with known days-in-milk values
dim_test_data = test.copy()

dim_test_target = pd.to_numeric(
    dim_test_data["days_in_milk"],
    errors="coerce"
)

known_test_target = dim_test_target.notna()

dim_test_data = dim_test_data.loc[
    known_test_target
].copy()

dim_test_target = dim_test_target.loc[
    known_test_target
]

# Create test features using the same function as training
X_dim_test = make_dim_features(
    dim_test_data
)

# Guarantee identical feature order
X_dim_test = X_dim_test.reindex(
    columns=X_dim_train.columns
)

# Use training medians only
X_dim_test = X_dim_test.fillna(
    training_medians
)

# Make stage predictions
test_stage_probabilities = (
    stage_model.predict_proba(X_dim_test)
)

test_stage_confidence = (
    test_stage_probabilities.max(axis=1)
)

test_predicted_stages = (
    stage_model.predict(X_dim_test)
)

# Convert predicted stages back into numeric days
test_stage_predictions = (
    pd.Series(test_predicted_stages)
    .map(stage_medians)
    .to_numpy()
)

# Accept only predictions meeting the fixed threshold
test_accepted_mask = (
    test_stage_confidence
    >= selected_dim_threshold
)

accepted_test_count = (
    test_accepted_mask.sum()
)

test_coverage = (
    accepted_test_count
    / len(dim_test_target)
)

accepted_true_days = (
    dim_test_target
    .to_numpy()[test_accepted_mask]
)

accepted_predicted_days = (
    test_stage_predictions[
        test_accepted_mask
    ]
)

accepted_test_errors = np.abs(
    accepted_true_days
    - accepted_predicted_days
)

true_test_stages = (
    dim_test_target.to_numpy()
    // bin_width
).astype(int)

accepted_stage_accuracy = (
    true_test_stages[test_accepted_mask]
    == test_predicted_stages[test_accepted_mask]
).mean()

print("\nFINAL DAYS-IN-MILK TEST RESULTS")
print("Selected confidence:", selected_dim_threshold)
print("Test records:", len(dim_test_target))
print("Accepted predictions:", accepted_test_count)
print(
    "Test coverage:",
    round(test_coverage * 100, 2),
    "%"
)
print(
    "Stage accuracy:",
    round(accepted_stage_accuracy, 4)
)
print(
    "MAE:",
    round(accepted_test_errors.mean(), 2),
    "days"
)
print(
    "Within 7 days:",
    round(
        (accepted_test_errors <= 7).mean()
        * 100,
        2
    ),
    "%"
)
print(
    "Within 14 days:",
    round(
        (accepted_test_errors <= 14).mean()
        * 100,
        2
    ),
    "%"
)
print(
    "Within 30 days:",
    round(
        (accepted_test_errors <= 30).mean()
        * 100,
        2
    ),
    "%"
)

NameError: name 'test' is not defined

In [47]:
# Part 20: Final days-in-milk test
# Threshold selected using validation data only

selected_dim_threshold = 0.45

# Retrieve the untouched test rows from known_context
dim_test_data = known_context.iloc[
    test_indices
].copy()

# Extract the known test targets
dim_test_target = pd.to_numeric(
    dim_test_data["days_in_milk"],
    errors="coerce"
)

# Keep only records with a known target
known_test_target = dim_test_target.notna()

dim_test_data = dim_test_data.loc[
    known_test_target
].copy()

dim_test_target = dim_test_target.loc[
    known_test_target
].copy()

# Create the same features used during training
X_dim_test = make_dim_features(
    dim_test_data
)

# Guarantee identical columns and column order
X_dim_test = X_dim_test.reindex(
    columns=X_dim_train.columns
)

# Use medians calculated from training data only
X_dim_test = X_dim_test.fillna(
    training_medians
)

print("Test records:", len(X_dim_test))
print("Making final test predictions...")

# Predict stage probabilities
test_stage_probabilities = (
    stage_model.predict_proba(
        X_dim_test
    )
)

# Confidence is the probability of the selected stage
test_stage_confidence = (
    test_stage_probabilities.max(axis=1)
)

# Predict the 60-day stage
test_predicted_stages = (
    stage_model.predict(
        X_dim_test
    )
)

# Convert predicted stages into numerical days-in-milk
test_stage_predictions = (
    pd.Series(test_predicted_stages)
    .map(stage_medians)
    .to_numpy()
)

# Accept predictions using the threshold chosen on validation
test_accepted_mask = (
    test_stage_confidence
    >= selected_dim_threshold
)

accepted_test_count = int(
    test_accepted_mask.sum()
)

test_coverage = (
    accepted_test_count
    / len(dim_test_target)
)

# Retrieve accepted true and predicted values
accepted_true_days = (
    dim_test_target
    .to_numpy()[test_accepted_mask]
)

accepted_predicted_days = (
    test_stage_predictions[
        test_accepted_mask
    ]
)

accepted_test_errors = np.abs(
    accepted_true_days
    - accepted_predicted_days
)

# Convert true test values into 60-day stages
true_test_stages = (
    dim_test_target.to_numpy()
    // bin_width
).astype(int)

accepted_stage_accuracy = (
    true_test_stages[test_accepted_mask]
    == test_predicted_stages[test_accepted_mask]
).mean()

print("\nFINAL DAYS-IN-MILK TEST RESULTS")
print(
    "Selected confidence:",
    selected_dim_threshold
)
print(
    "Test records:",
    len(dim_test_target)
)
print(
    "Accepted predictions:",
    accepted_test_count
)
print(
    "Test coverage:",
    round(test_coverage * 100, 2),
    "%"
)
print(
    "Stage accuracy:",
    round(accepted_stage_accuracy, 4)
)
print(
    "MAE:",
    round(
        accepted_test_errors.mean(),
        2
    ),
    "days"
)
print(
    "Within 7 days:",
    round(
        (accepted_test_errors <= 7).mean()
        * 100,
        2
    ),
    "%"
)
print(
    "Within 14 days:",
    round(
        (accepted_test_errors <= 14).mean()
        * 100,
        2
    ),
    "%"
)
print(
    "Within 30 days:",
    round(
        (accepted_test_errors <= 30).mean()
        * 100,
        2
    ),
    "%"
)

Test records: 561701
Making final test predictions...

FINAL DAYS-IN-MILK TEST RESULTS
Selected confidence: 0.45
Test records: 561701
Accepted predictions: 13516
Test coverage: 2.41 %
Stage accuracy: 0.5071
MAE: 45.2 days
Within 7 days: 14.72 %
Within 14 days: 28.18 %
Within 30 days: 52.17 %


In [48]:
# Part 21: Reproduction-status classification pilot

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# Targets aligned with the existing feature matrices
repro_train_target = dim_model_train.loc[
    X_dim_train.index,
    "reproduction_status"
]

repro_validation_target = dim_model_validation.loc[
    X_dim_validation.index,
    "reproduction_status"
]

# Remove missing targets
repro_train_mask = repro_train_target.notna()
repro_validation_mask = repro_validation_target.notna()

X_repro_train = X_dim_train.loc[
    repro_train_mask
].copy()

y_repro_train_text = (
    repro_train_target.loc[repro_train_mask]
    .astype(str)
)

X_repro_validation = X_dim_validation.loc[
    repro_validation_mask
].copy()

y_repro_validation_text = (
    repro_validation_target.loc[
        repro_validation_mask
    ]
    .astype(str)
)

# Encode status labels as numbers
repro_encoder = LabelEncoder()

y_repro_train = repro_encoder.fit_transform(
    y_repro_train_text
)

# Keep validation statuses found in training
known_validation_class = (
    y_repro_validation_text
    .isin(repro_encoder.classes_)
)

X_repro_validation = X_repro_validation.loc[
    known_validation_class
]

y_repro_validation_text = (
    y_repro_validation_text.loc[
        known_validation_class
    ]
)

y_repro_validation = repro_encoder.transform(
    y_repro_validation_text
)

# Limit training size
maximum_repro_training_rows = 250_000

if len(X_repro_train) > maximum_repro_training_rows:
    sampled_positions = np.random.default_rng(42).choice(
        len(X_repro_train),
        size=maximum_repro_training_rows,
        replace=False
    )

    X_repro_train = X_repro_train.iloc[
        sampled_positions
    ]

    y_repro_train = y_repro_train[
        sampled_positions
    ]

# Majority-class baseline
majority_class = pd.Series(
    y_repro_train
).mode().iloc[0]

baseline_predictions = np.full(
    len(y_repro_validation),
    majority_class
)

baseline_accuracy = accuracy_score(
    y_repro_validation,
    baseline_predictions
)

print("Classes:", list(repro_encoder.classes_))
print("Training records:", len(X_repro_train))
print("Validation records:", len(X_repro_validation))
print("Majority baseline:", round(baseline_accuracy, 4))
print("Fitting reproduction-status model...")

repro_model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    l2_regularization=1.0,
    random_state=42
)

repro_model.fit(
    X_repro_train,
    y_repro_train
)

repro_validation_predictions = (
    repro_model.predict(
        X_repro_validation
    )
)

repro_accuracy = accuracy_score(
    y_repro_validation,
    repro_validation_predictions
)

repro_balanced_accuracy = balanced_accuracy_score(
    y_repro_validation,
    repro_validation_predictions
)

repro_macro_f1 = f1_score(
    y_repro_validation,
    repro_validation_predictions,
    average="macro"
)

print("\nREPRODUCTION-STATUS RESULTS")
print("Accuracy:", round(repro_accuracy, 4))
print(
    "Balanced accuracy:",
    round(repro_balanced_accuracy, 4)
)
print("Macro F1:", round(repro_macro_f1, 4))

Classes: ['bred', 'fresh', 'open', 'pregnant']
Training records: 250000
Validation records: 100000
Majority baseline: 0.4948
Fitting reproduction-status model...

REPRODUCTION-STATUS RESULTS
Accuracy: 0.5323
Balanced accuracy: 0.3142
Macro F1: 0.2825


In [49]:
# Part 22: Reproduction-status confidence threshold test

repro_validation_probabilities = (
    repro_model.predict_proba(
        X_repro_validation
    )
)

repro_validation_confidence = (
    repro_validation_probabilities.max(axis=1)
)

repro_confidence_results = []

for minimum_confidence in [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.70,
    0.80,
    0.90
]:
    accepted_mask = (
        repro_validation_confidence
        >= minimum_confidence
    )

    accepted_count = int(
        accepted_mask.sum()
    )

    if accepted_count == 0:
        continue

    accepted_true = y_repro_validation[
        accepted_mask
    ]

    accepted_predictions = (
        repro_validation_predictions[
            accepted_mask
        ]
    )

    accepted_accuracy = accuracy_score(
        accepted_true,
        accepted_predictions
    )

    accepted_balanced_accuracy = (
        balanced_accuracy_score(
            accepted_true,
            accepted_predictions
        )
    )

    accepted_macro_f1 = f1_score(
        accepted_true,
        accepted_predictions,
        average="macro",
        zero_division=0
    )

    repro_confidence_results.append({
        "minimum_confidence": minimum_confidence,
        "accepted_predictions": accepted_count,
        "coverage_percent": round(
            accepted_count
            / len(y_repro_validation)
            * 100,
            2
        ),
        "accuracy": round(
            accepted_accuracy,
            4
        ),
        "balanced_accuracy": round(
            accepted_balanced_accuracy,
            4
        ),
        "macro_f1": round(
            accepted_macro_f1,
            4
        )
    })

repro_confidence_results = pd.DataFrame(
    repro_confidence_results
)

display(repro_confidence_results)

print("\nTHRESHOLDS WITH AT LEAST 50% ACCURACY:")

display(
    repro_confidence_results[
        repro_confidence_results["accuracy"]
        >= 0.50
    ].sort_values(
        "coverage_percent",
        ascending=False
    )
)

,minimum_confidence,accepted_predictions,coverage_percent,accuracy,balanced_accuracy,macro_f1
0,0.30,99900,99.90,0.5324,0.3141,0.2823
1,0.35,97821,97.82,0.5365,0.3125,0.2799
2,0.40,86774,86.77,0.5572,0.3077,0.2756
3,0.45,69030,69.03,0.5921,0.2855,0.2521
4,0.50,55006,55.01,0.6245,0.2603,0.2140
5,0.55,43874,43.87,0.6517,0.2526,0.2027
6,0.60,32939,32.94,0.6774,0.2510,0.2040
7,0.70,9273,9.27,0.7431,0.2503,0.2138
8,0.80,300,0.30,0.8100,0.2500,0.2238



THRESHOLDS WITH AT LEAST 50% ACCURACY:


,minimum_confidence,accepted_predictions,coverage_percent,accuracy,balanced_accuracy,macro_f1
0,0.30,99900,99.90,0.5324,0.3141,0.2823
1,0.35,97821,97.82,0.5365,0.3125,0.2799
2,0.40,86774,86.77,0.5572,0.3077,0.2756
3,0.45,69030,69.03,0.5921,0.2855,0.2521
4,0.50,55006,55.01,0.6245,0.2603,0.2140
5,0.55,43874,43.87,0.6517,0.2526,0.2027
6,0.60,32939,32.94,0.6774,0.2510,0.2040
7,0.70,9273,9.27,0.7431,0.2503,0.2138
8,0.80,300,0.30,0.8100,0.2500,0.2238


In [50]:
# Part 23: Examine reproduction-status performance by class

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

selected_repro_threshold = 0.45

accepted_repro_mask = (
    repro_validation_confidence
    >= selected_repro_threshold
)

accepted_repro_true = (
    y_repro_validation[
        accepted_repro_mask
    ]
)

accepted_repro_predictions = (
    repro_validation_predictions[
        accepted_repro_mask
    ]
)

# Convert encoded numbers back into status names
accepted_true_names = (
    repro_encoder.inverse_transform(
        accepted_repro_true
    )
)

accepted_prediction_names = (
    repro_encoder.inverse_transform(
        accepted_repro_predictions
    )
)

print(
    "Accepted predictions:",
    accepted_repro_mask.sum()
)

print(
    "Coverage:",
    round(
        accepted_repro_mask.mean() * 100,
        2
    ),
    "%"
)

# Precision, recall and F1 for every status
class_report = classification_report(
    accepted_true_names,
    accepted_prediction_names,
    labels=repro_encoder.classes_,
    output_dict=True,
    zero_division=0
)

class_report_table = (
    pd.DataFrame(class_report)
    .transpose()
)

display(class_report_table)

# Row-normalized confusion matrix
# Each row shows what percentage of a true class
# was assigned to each predicted class
normalized_confusion = confusion_matrix(
    accepted_true_names,
    accepted_prediction_names,
    labels=repro_encoder.classes_,
    normalize="true"
)

normalized_confusion_table = pd.DataFrame(
    normalized_confusion,
    index=[
        f"True: {status}"
        for status in repro_encoder.classes_
    ],
    columns=[
        f"Predicted: {status}"
        for status in repro_encoder.classes_
    ]
)

display(
    normalized_confusion_table.round(3)
)

Accepted predictions: 69030
Coverage: 69.03 %


,precision,recall,f1-score,support
bred,0.456209,0.155082,0.231476,15753.000000
fresh,0.522388,0.016663,0.032295,8402.000000
open,1.000000,0.000185,0.000369,5415.000000
pregnant,0.603823,0.970248,0.744386,39460.000000
accuracy,0.592061,0.592061,0.592061,0.592061
macro avg,0.645605,0.285544,0.252132,69030.000000
weighted avg,0.591303,0.592061,0.482301,69030.000000


,Predicted: bred,Predicted: fresh,Predicted: open,Predicted: pregnant
True: bred,0.155,0.003,0.0,0.842
True: fresh,0.120,0.017,0.0,0.863
True: open,0.146,0.004,0.0,0.850
True: pregnant,0.028,0.001,0.0,0.970


In [51]:
# Part 24: Class-balanced reproduction-status model

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

print("Fitting class-balanced reproduction model...")

repro_balanced_model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    l2_regularization=1.0,
    class_weight="balanced",
    random_state=42
)

repro_balanced_model.fit(
    X_repro_train,
    y_repro_train
)

balanced_repro_predictions = (
    repro_balanced_model.predict(
        X_repro_validation
    )
)

balanced_repro_accuracy = accuracy_score(
    y_repro_validation,
    balanced_repro_predictions
)

balanced_repro_balanced_accuracy = (
    balanced_accuracy_score(
        y_repro_validation,
        balanced_repro_predictions
    )
)

balanced_repro_macro_f1 = f1_score(
    y_repro_validation,
    balanced_repro_predictions,
    average="macro",
    zero_division=0
)

print("\nCLASS-BALANCED REPRODUCTION RESULTS")
print(
    "Overall accuracy:",
    round(balanced_repro_accuracy, 4)
)
print(
    "Balanced accuracy:",
    round(
        balanced_repro_balanced_accuracy,
        4
    )
)
print(
    "Macro F1:",
    round(balanced_repro_macro_f1, 4)
)

balanced_prediction_names = (
    repro_encoder.inverse_transform(
        balanced_repro_predictions
    )
)

balanced_report = classification_report(
    y_repro_validation_text,
    balanced_prediction_names,
    labels=repro_encoder.classes_,
    output_dict=True,
    zero_division=0
)

display(
    pd.DataFrame(
        balanced_report
    ).transpose()
)

Fitting class-balanced reproduction model...

CLASS-BALANCED REPRODUCTION RESULTS
Overall accuracy: 0.4498
Balanced accuracy: 0.357
Macro F1: 0.3539


,precision,recall,f1-score,support
bred,0.368301,0.435413,0.399055,27165.00000
fresh,0.239186,0.215929,0.226963,14213.00000
open,0.166169,0.207066,0.184377,9142.00000
pregnant,0.645710,0.569786,0.605377,49480.00000
accuracy,0.449830,0.449830,0.449830,0.44983
macro avg,0.354842,0.357049,0.353943,100000.00000
weighted avg,0.468733,0.449830,0.457058,100000.00000


In [53]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# Find the class-balanced model from the previous cell
possible_balanced_names = [
    "balanced_repro_model",
    "balanced_reproduction_model",
    "repro_balanced_model",
    "balanced_model"
]

balanced_model_found = None
balanced_model_name = None

for model_name in possible_balanced_names:
    candidate = globals().get(model_name)

    if candidate is not None and hasattr(
        candidate,
        "predict_proba"
    ):
        balanced_model_found = candidate
        balanced_model_name = model_name
        break

if balanced_model_found is None:
    raise NameError(
        "Balanced model not found. Rerun the class-balanced "
        "reproduction model cell immediately above this cell."
    )

print("Balanced model found:", balanced_model_name)

# Generate validation probabilities
original_probabilities = repro_model.predict_proba(
    X_repro_validation
)

balanced_probabilities = balanced_model_found.predict_proba(
    X_repro_validation
)

# Confirm that the class orders match
print("Original classes:", repro_model.classes_)
print("Balanced classes:", balanced_model_found.classes_)

if not np.array_equal(
    repro_model.classes_,
    balanced_model_found.classes_
):
    raise ValueError(
        "The original and balanced models use different class orders."
    )

classes = repro_model.classes_
hybrid_results = []

# Test different combinations of the two models
for alpha in np.arange(0.0, 1.01, 0.1):

    hybrid_probabilities = (
        (1 - alpha) * original_probabilities
        + alpha * balanced_probabilities
    )

    hybrid_predictions = classes[
        np.argmax(hybrid_probabilities, axis=1)
    ]

    hybrid_results.append(
        {
            "balanced_model_weight": round(alpha, 1),
            "overall_accuracy": round(
                accuracy_score(
                    y_repro_validation,
                    hybrid_predictions
                ),
                4
            ),
            "balanced_accuracy": round(
                balanced_accuracy_score(
                    y_repro_validation,
                    hybrid_predictions
                ),
                4
            ),
            "macro_f1": round(
                f1_score(
                    y_repro_validation,
                    hybrid_predictions,
                    average="macro"
                ),
                4
            )
        }
    )

hybrid_results = pd.DataFrame(hybrid_results)

display(hybrid_results)

print("\nHYBRIDS WITH AT LEAST 50% OVERALL ACCURACY")

display(
    hybrid_results[
        hybrid_results["overall_accuracy"] >= 0.50
    ].sort_values(
        ["macro_f1", "balanced_accuracy"],
        ascending=False
    )
)

Balanced model found: repro_balanced_model
Original classes: [0 1 2 3]
Balanced classes: [0 1 2 3]


,balanced_model_weight,overall_accuracy,balanced_accuracy,macro_f1
0,0.0,0.5323,0.3142,0.2825
1,0.1,0.5319,0.3171,0.2873
2,0.2,0.5315,0.3205,0.2929
3,0.3,0.5305,0.3243,0.2998
4,0.4,0.5284,0.3288,0.3082
5,0.5,0.5239,0.3335,0.3184
6,0.6,0.5180,0.3390,0.3294
7,0.7,0.5080,0.3439,0.3399
8,0.8,0.4940,0.3498,0.3498
9,0.9,0.4762,0.3551,0.3556



HYBRIDS WITH AT LEAST 50% OVERALL ACCURACY


,balanced_model_weight,overall_accuracy,balanced_accuracy,macro_f1
7,0.7,0.5080,0.3439,0.3399
6,0.6,0.5180,0.3390,0.3294
5,0.5,0.5239,0.3335,0.3184
4,0.4,0.5284,0.3288,0.3082
3,0.3,0.5305,0.3243,0.2998
2,0.2,0.5315,0.3205,0.2929
1,0.1,0.5319,0.3171,0.2873
0,0.0,0.5323,0.3142,0.2825


In [54]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Lock the validation-selected mixture
selected_repro_weight = 0.70

# Predict test-set probabilities
original_test_probabilities = repro_model.predict_proba(
    X_repro_test
)

balanced_test_probabilities = (
    balanced_model_found.predict_proba(
        X_repro_test
    )
)

# Combine the two models
hybrid_test_probabilities = (
    (1 - selected_repro_weight)
    * original_test_probabilities
    + selected_repro_weight
    * balanced_test_probabilities
)

repro_test_predictions = repro_model.classes_[
    np.argmax(hybrid_test_probabilities, axis=1)
]

# Final metrics
repro_test_accuracy = accuracy_score(
    y_repro_test,
    repro_test_predictions
)

repro_test_balanced_accuracy = balanced_accuracy_score(
    y_repro_test,
    repro_test_predictions
)

repro_test_macro_f1 = f1_score(
    y_repro_test,
    repro_test_predictions,
    average="macro"
)

print("\nFINAL HYBRID REPRODUCTION-STATUS TEST RESULTS")
print("Balanced-model weight:", selected_repro_weight)
print("Test records:", len(y_repro_test))
print("Overall accuracy:", round(repro_test_accuracy, 4))
print(
    "Balanced accuracy:",
    round(repro_test_balanced_accuracy, 4)
)
print("Macro F1:", round(repro_test_macro_f1, 4))

status_names = [
    "bred",
    "fresh",
    "open",
    "pregnant"
]

report = classification_report(
    y_repro_test,
    repro_test_predictions,
    labels=repro_model.classes_,
    target_names=status_names,
    output_dict=True,
    zero_division=0
)

display(pd.DataFrame(report).transpose())

confusion_table = pd.DataFrame(
    confusion_matrix(
        y_repro_test,
        repro_test_predictions,
        labels=repro_model.classes_,
        normalize="true"
    ),
    index=[f"True: {status}" for status in status_names],
    columns=[
        f"Predicted: {status}"
        for status in status_names
    ]
)

display(confusion_table.round(3))

NameError: name 'X_repro_test' is not defined

In [55]:
import numpy as np
import pandas as pd

print("TEST-RELATED VARIABLES")
for name, value in list(globals().items()):
    if "test" in name.lower():
        if hasattr(value, "shape"):
            print(name, "shape =", value.shape)
        elif hasattr(value, "__len__") and not isinstance(value, str):
            print(name, "length =", len(value))
        else:
            print(name, "type =", type(value).__name__)

print("\nLARGE DATAFRAMES")
for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame) and len(value) > 500_000:
        print(
            name,
            "shape =", value.shape,
            "has reproduction_status =",
            "reproduction_status" in value.columns
        )

print("\nVALIDATION FEATURES")
print("X_repro_validation type:", type(X_repro_validation).__name__)
print("X_repro_validation shape:", X_repro_validation.shape)

if isinstance(X_repro_validation, pd.DataFrame):
    print(
        "Feature columns:",
        X_repro_validation.columns.tolist()
    )

TEST-RELATED VARIABLES
test_relative shape = (573697,)
test_indices shape = (561701,)
test_cows length = 8935
test_cows_missing_from_training length = 8
test_move_mask shape = (561709,)
test_rows_to_move shape = (8,)
final_test shape = (561701, 7)
weekly_test shape = (561701, 7)
test_weekly_counts shape = (1969963, 9)
test_weekly_lookup shape = (1185265, 10)
test_weekly_matches shape = (561701, 10)
weekly_test_accepted shape = (56003, 10)
weekly_test_accuracy shape = ()
weekly_test_coverage type = float
broad_test_accepted shape = (145173, 10)
broad_test_accuracy shape = ()
broad_test_coverage type = float
dim_test_data shape = (561701, 16)
dim_test_target shape = (561701,)
known_test_target shape = (561701,)
X_dim_test shape = (561701, 17)
test_stage_probabilities shape = (561701, 15)
test_stage_confidence shape = (561701,)
test_predicted_stages shape = (561701,)
test_stage_predictions shape = (561701,)
test_accepted_mask shape = (561701,)
accepted_test_count type = int
test_coverage 

In [56]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# --------------------------------------------------
# 1. Create reproduction test features
# --------------------------------------------------

if not isinstance(X_dim_test, pd.DataFrame):
    raise TypeError(
        "X_dim_test must be a DataFrame so its columns can be checked."
    )

repro_feature_columns = X_repro_validation.columns.tolist()

missing_features = [
    column
    for column in repro_feature_columns
    if column not in X_dim_test.columns
]

if missing_features:
    raise ValueError(
        "X_dim_test is missing these reproduction features: "
        + str(missing_features)
    )

# Put test features in exactly the same order as training
X_repro_test = X_dim_test[
    repro_feature_columns
].copy()

# --------------------------------------------------
# 2. Create reproduction-status test target
# --------------------------------------------------

status_mapping = {
    "bred": 0,
    "fresh": 1,
    "open": 2,
    "pregnant": 3
}

y_repro_test = (
    dim_test_data["reproduction_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(status_mapping)
)

# Retain only rows with known test answers
known_status_mask = y_repro_test.notna()

X_repro_test = X_repro_test.loc[
    known_status_mask
].copy()

y_repro_test = (
    y_repro_test.loc[known_status_mask]
    .astype(int)
    .to_numpy()
)

print("Reproduction test records:", len(y_repro_test))
print("Test feature shape:", X_repro_test.shape)

# --------------------------------------------------
# 3. Apply the validation-selected hybrid
# --------------------------------------------------

selected_repro_weight = 0.70

original_test_probabilities = repro_model.predict_proba(
    X_repro_test
)

balanced_test_probabilities = (
    balanced_model_found.predict_proba(
        X_repro_test
    )
)

hybrid_test_probabilities = (
    (1 - selected_repro_weight)
    * original_test_probabilities
    + selected_repro_weight
    * balanced_test_probabilities
)

repro_test_predictions = repro_model.classes_[
    np.argmax(hybrid_test_probabilities, axis=1)
]

# --------------------------------------------------
# 4. Calculate final untouched-test results
# --------------------------------------------------

test_accuracy = accuracy_score(
    y_repro_test,
    repro_test_predictions
)

test_balanced_accuracy = balanced_accuracy_score(
    y_repro_test,
    repro_test_predictions
)

test_macro_f1 = f1_score(
    y_repro_test,
    repro_test_predictions,
    average="macro"
)

print("\nFINAL HYBRID REPRODUCTION-STATUS TEST RESULTS")
print("Balanced-model weight:", selected_repro_weight)
print("Overall accuracy:", round(test_accuracy, 4))
print(
    "Balanced accuracy:",
    round(test_balanced_accuracy, 4)
)
print("Macro F1:", round(test_macro_f1, 4))

status_names = [
    "bred",
    "fresh",
    "open",
    "pregnant"
]

report = classification_report(
    y_repro_test,
    repro_test_predictions,
    labels=[0, 1, 2, 3],
    target_names=status_names,
    output_dict=True,
    zero_division=0
)

display(pd.DataFrame(report).transpose())

confusion_table = pd.DataFrame(
    confusion_matrix(
        y_repro_test,
        repro_test_predictions,
        labels=[0, 1, 2, 3],
        normalize="true"
    ),
    index=[
        f"True: {status}"
        for status in status_names
    ],
    columns=[
        f"Predicted: {status}"
        for status in status_names
    ]
)

display(confusion_table.round(3))

Reproduction test records: 561701
Test feature shape: (561701, 17)

FINAL HYBRID REPRODUCTION-STATUS TEST RESULTS
Balanced-model weight: 0.7
Overall accuracy: 0.5103
Balanced accuracy: 0.3479
Macro F1: 0.3448


,precision,recall,f1-score,support
bred,0.391088,0.456942,0.421458,152540.000000
fresh,0.296229,0.128997,0.179729,79901.000000
open,0.213575,0.075345,0.111393,51576.000000
pregnant,0.613469,0.730121,0.666731,277684.000000
accuracy,0.510304,0.510304,0.510304,0.510304
macro avg,0.378590,0.347851,0.344828,561701.000000
weighted avg,0.471232,0.510304,0.479856,561701.000000


,Predicted: bred,Predicted: fresh,Predicted: open,Predicted: pregnant
True: bred,0.457,0.061,0.045,0.437
True: fresh,0.359,0.129,0.047,0.465
True: open,0.377,0.085,0.075,0.462
True: pregnant,0.218,0.039,0.014,0.730


In [57]:
import pandas as pd

print("DATAFRAMES CONTAINING REPRODUCTION STATUS\n")

for name, value in list(globals().items()):
    if (
        isinstance(value, pd.DataFrame)
        and "reproduction_status" in value.columns
    ):
        missing_count = (
            value["reproduction_status"]
            .isna()
            .sum()
        )

        print(
            name,
            "| shape:", value.shape,
            "| missing reproduction status:",
            missing_count
        )

DATAFRAMES CONTAINING REPRODUCTION STATUS

header | shape: (0, 34) | missing reproduction status: 0
known_context | shape: (3748763, 16) | missing reproduction status: 0
context_to_predict | shape: (939751, 16) | missing reproduction status: 939749
temporary_data | shape: (1141963, 16) | missing reproduction status: 0
signature_data | shape: (3748763, 16) | missing reproduction status: 0
train_signature | shape: (2624890, 16) | missing reproduction status: 0
validation_signature | shape: (562172, 16) | missing reproduction status: 0
matched_validation | shape: (562172, 20) | missing reproduction status: 0
relaxed_signature_data | shape: (3748763, 16) | missing reproduction status: 0
relaxed_train | shape: (2624890, 16) | missing reproduction status: 0
relaxed_validation | shape: (562172, 16) | missing reproduction status: 0
relaxed_matches | shape: (562172, 20) | missing reproduction status: 0
relaxed_accepted | shape: (3979, 20) | missing reproduction status: 0
dim_model_train | shape

In [58]:
import numpy as np
import pandas as pd

# Work on a copy so the original data remains unchanged
context_predictions = context_to_predict.copy()

# Rows requiring reproduction-status predictions
missing_repro_mask = (
    context_predictions["reproduction_status"].isna()
)

missing_repro_data = context_predictions.loc[
    missing_repro_mask
].copy()

print(
    "Missing reproduction-status rows:",
    len(missing_repro_data)
)

# --------------------------------------------------
# Build the same 17 features used by the models
# --------------------------------------------------

def build_reproduction_features(data, feature_columns):

    working_data = data.copy()

    if "event_date" in working_data.columns:
        event_dates = pd.to_datetime(
            working_data["event_date"],
            errors="coerce"
        )

        date_features = {
            "event_year": event_dates.dt.year,
            "event_month": event_dates.dt.month,
            "event_day": event_dates.dt.day,
            "day_of_year": event_dates.dt.dayofyear,
            "day_of_week": event_dates.dt.dayofweek,
            "event_dayofweek": event_dates.dt.dayofweek,
            "event_week": event_dates.dt.isocalendar().week,
            "event_quarter": event_dates.dt.quarter,
            "event_ordinal": event_dates.map(
                lambda date: (
                    date.toordinal()
                    if pd.notna(date)
                    else np.nan
                )
            )
        }

        for column, values in date_features.items():
            if column not in working_data.columns:
                working_data[column] = values

    missing_features = [
        column
        for column in feature_columns
        if column not in working_data.columns
    ]

    if missing_features:
        raise ValueError(
            "Unable to create these features: "
            + str(missing_features)
        )

    feature_data = working_data[
        feature_columns
    ].copy()

    # Ensure every model feature is numeric
    for column in feature_data.columns:
        feature_data[column] = pd.to_numeric(
            feature_data[column],
            errors="coerce"
        )

    return feature_data


repro_feature_columns = (
    X_repro_validation.columns.tolist()
)

X_missing_repro = build_reproduction_features(
    missing_repro_data,
    repro_feature_columns
)

print("Prediction feature shape:", X_missing_repro.shape)

# --------------------------------------------------
# Apply the locked 70/30 hybrid
# --------------------------------------------------

selected_repro_weight = 0.70

original_missing_probabilities = (
    repro_model.predict_proba(X_missing_repro)
)

balanced_missing_probabilities = (
    balanced_model_found.predict_proba(
        X_missing_repro
    )
)

hybrid_missing_probabilities = (
    (1 - selected_repro_weight)
    * original_missing_probabilities
    + selected_repro_weight
    * balanced_missing_probabilities
)

predicted_codes = repro_model.classes_[
    np.argmax(hybrid_missing_probabilities, axis=1)
]

prediction_confidence = np.max(
    hybrid_missing_probabilities,
    axis=1
)

status_lookup = {
    0: "bred",
    1: "fresh",
    2: "open",
    3: "pregnant"
}

predicted_statuses = pd.Series(
    predicted_codes
).map(status_lookup).to_numpy()

# --------------------------------------------------
# Store predictions without overwriting known values
# --------------------------------------------------

context_predictions[
    "reproduction_status_predicted"
] = context_predictions["reproduction_status"]

context_predictions.loc[
    missing_repro_mask,
    "reproduction_status_predicted"
] = predicted_statuses

context_predictions[
    "reproduction_status_confidence"
] = np.nan

context_predictions.loc[
    missing_repro_mask,
    "reproduction_status_confidence"
] = prediction_confidence

print("\nREPRODUCTION-STATUS IMPUTATION RESULTS")
print("Predictions created:", len(predicted_statuses))
print(
    "Values still missing:",
    context_predictions[
        "reproduction_status_predicted"
    ].isna().sum()
)

print("\nPredicted-status distribution:")
display(
    context_predictions.loc[
        missing_repro_mask,
        "reproduction_status_predicted"
    ].value_counts(
        dropna=False
    ).to_frame("predictions")
)

display(
    context_predictions[
        [
            "source_row",
            "reproduction_status",
            "reproduction_status_predicted",
            "reproduction_status_confidence"
        ]
    ].head(10)
)

Missing reproduction-status rows: 939749


ValueError: Unable to create these features: ['year_sin', 'year_cos', 'yield_per_second', 'first_2_min_fraction', 'flow_ratio']

In [59]:
import numpy as np
import pandas as pd

# Preserve the original data
context_predictions = context_to_predict.copy()

missing_repro_mask = (
    context_predictions["reproduction_status"].isna()
)

missing_repro_data = context_predictions.loc[
    missing_repro_mask
].copy()

print(
    "Missing reproduction-status rows:",
    len(missing_repro_data)
)

# --------------------------------------------------
# Create the exact engineered features
# --------------------------------------------------

event_dates = pd.to_datetime(
    missing_repro_data["event_date"],
    errors="coerce"
)

missing_repro_data["event_year"] = event_dates.dt.year
missing_repro_data["event_month"] = event_dates.dt.month
missing_repro_data["event_day"] = event_dates.dt.day
missing_repro_data["day_of_year"] = event_dates.dt.dayofyear
missing_repro_data["day_of_week"] = event_dates.dt.dayofweek

# Seasonal position within the year
missing_repro_data["year_sin"] = np.sin(
    2
    * np.pi
    * missing_repro_data["day_of_year"]
    / 365.25
)

missing_repro_data["year_cos"] = np.cos(
    2
    * np.pi
    * missing_repro_data["day_of_year"]
    / 365.25
)

# Milk yield per second
duration = pd.to_numeric(
    missing_repro_data["duration_session_sec"],
    errors="coerce"
)

milk_yield = pd.to_numeric(
    missing_repro_data["milk_yield"],
    errors="coerce"
)

first_two_minute_yield = pd.to_numeric(
    missing_repro_data["yield_first_2_min_session"],
    errors="coerce"
)

missing_repro_data["yield_per_second"] = np.where(
    duration > 0,
    milk_yield / duration,
    np.nan
)

# Fraction of total milk collected in first two minutes
missing_repro_data["first_2_min_fraction"] = np.where(
    milk_yield > 0,
    first_two_minute_yield / milk_yield,
    np.nan
)

# Remove infinite values created by division
missing_repro_data.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

# --------------------------------------------------
# Match the model's feature columns and order
# --------------------------------------------------

repro_feature_columns = (
    X_repro_validation.columns.tolist()
)

still_missing_features = [
    column
    for column in repro_feature_columns
    if column not in missing_repro_data.columns
]

if still_missing_features:
    raise ValueError(
        "Features still missing: "
        + str(still_missing_features)
    )

X_missing_repro = missing_repro_data[
    repro_feature_columns
].copy()

for column in X_missing_repro.columns:
    X_missing_repro[column] = pd.to_numeric(
        X_missing_repro[column],
        errors="coerce"
    )

print("Prediction feature shape:", X_missing_repro.shape)

# --------------------------------------------------
# Apply the locked hybrid reproduction model
# --------------------------------------------------

selected_repro_weight = 0.70

original_probabilities = repro_model.predict_proba(
    X_missing_repro
)

balanced_probabilities = (
    balanced_model_found.predict_proba(
        X_missing_repro
    )
)

hybrid_probabilities = (
    (1 - selected_repro_weight)
    * original_probabilities
    + selected_repro_weight
    * balanced_probabilities
)

predicted_codes = repro_model.classes_[
    np.argmax(hybrid_probabilities, axis=1)
]

prediction_confidence = np.max(
    hybrid_probabilities,
    axis=1
)

status_lookup = {
    0: "bred",
    1: "fresh",
    2: "open",
    3: "pregnant"
}

predicted_statuses = pd.Series(
    predicted_codes
).map(status_lookup).to_numpy()

# --------------------------------------------------
# Store predictions without changing known values
# --------------------------------------------------

context_predictions[
    "reproduction_status_predicted"
] = context_predictions["reproduction_status"]

context_predictions.loc[
    missing_repro_mask,
    "reproduction_status_predicted"
] = predicted_statuses

context_predictions[
    "reproduction_status_confidence"
] = np.nan

context_predictions.loc[
    missing_repro_mask,
    "reproduction_status_confidence"
] = prediction_confidence

print("\nREPRODUCTION-STATUS IMPUTATION RESULTS")
print("Predictions created:", len(predicted_statuses))
print(
    "Predicted values still missing:",
    context_predictions[
        "reproduction_status_predicted"
    ].isna().sum()
)

print("\nPrediction distribution:")
display(
    context_predictions.loc[
        missing_repro_mask,
        "reproduction_status_predicted"
    ].value_counts().to_frame("predictions")
)

display(
    context_predictions[
        [
            "source_row",
            "reproduction_status",
            "reproduction_status_predicted",
            "reproduction_status_confidence"
        ]
    ].head(10)
)

Missing reproduction-status rows: 939749


ValueError: Features still missing: ['flow_ratio']

In [60]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# Determine the original flow-ratio formula
# --------------------------------------------------

reference_average_flow = pd.to_numeric(
    dim_test_data["avg_milk_flow"],
    errors="coerce"
).to_numpy()

reference_30_60_flow = pd.to_numeric(
    dim_test_data["flow_30_60_session"],
    errors="coerce"
).to_numpy()

actual_reference_ratio = pd.to_numeric(
    X_dim_test["flow_ratio"],
    errors="coerce"
).to_numpy()

# Test both possible ratio directions
candidate_ratio_a = np.divide(
    reference_30_60_flow,
    reference_average_flow,
    out=np.full(
        reference_30_60_flow.shape,
        np.nan,
        dtype=float
    ),
    where=reference_average_flow != 0
)

candidate_ratio_b = np.divide(
    reference_average_flow,
    reference_30_60_flow,
    out=np.full(
        reference_average_flow.shape,
        np.nan,
        dtype=float
    ),
    where=reference_30_60_flow != 0
)

error_a = np.nanmedian(
    np.abs(candidate_ratio_a - actual_reference_ratio)
)

error_b = np.nanmedian(
    np.abs(candidate_ratio_b - actual_reference_ratio)
)

print("Formula A error:", error_a)
print("Formula B error:", error_b)

# --------------------------------------------------
# Apply the matching formula to missing records
# --------------------------------------------------

missing_average_flow = pd.to_numeric(
    missing_repro_data["avg_milk_flow"],
    errors="coerce"
).to_numpy()

missing_30_60_flow = pd.to_numeric(
    missing_repro_data["flow_30_60_session"],
    errors="coerce"
).to_numpy()

if error_a <= error_b:
    print(
        "Using: flow_30_60_session / avg_milk_flow"
    )

    missing_repro_data["flow_ratio"] = np.divide(
        missing_30_60_flow,
        missing_average_flow,
        out=np.full(
            missing_30_60_flow.shape,
            np.nan,
            dtype=float
        ),
        where=missing_average_flow != 0
    )

else:
    print(
        "Using: avg_milk_flow / flow_30_60_session"
    )

    missing_repro_data["flow_ratio"] = np.divide(
        missing_average_flow,
        missing_30_60_flow,
        out=np.full(
            missing_average_flow.shape,
            np.nan,
            dtype=float
        ),
        where=missing_30_60_flow != 0
    )

# --------------------------------------------------
# Build final feature table
# --------------------------------------------------

repro_feature_columns = (
    X_repro_validation.columns.tolist()
)

X_missing_repro = missing_repro_data[
    repro_feature_columns
].copy()

for column in X_missing_repro.columns:
    X_missing_repro[column] = pd.to_numeric(
        X_missing_repro[column],
        errors="coerce"
    )

X_missing_repro.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

print("Prediction feature shape:", X_missing_repro.shape)

# --------------------------------------------------
# Make hybrid predictions
# --------------------------------------------------

selected_repro_weight = 0.70

original_probabilities = repro_model.predict_proba(
    X_missing_repro
)

balanced_probabilities = (
    balanced_model_found.predict_proba(
        X_missing_repro
    )
)

hybrid_probabilities = (
    (1 - selected_repro_weight)
    * original_probabilities
    + selected_repro_weight
    * balanced_probabilities
)

predicted_codes = repro_model.classes_[
    np.argmax(hybrid_probabilities, axis=1)
]

prediction_confidence = np.max(
    hybrid_probabilities,
    axis=1
)

status_lookup = {
    0: "bred",
    1: "fresh",
    2: "open",
    3: "pregnant"
}

predicted_statuses = pd.Series(
    predicted_codes
).map(status_lookup).to_numpy()

# --------------------------------------------------
# Store predictions
# --------------------------------------------------

context_predictions[
    "reproduction_status_predicted"
] = context_predictions["reproduction_status"]

context_predictions.loc[
    missing_repro_mask,
    "reproduction_status_predicted"
] = predicted_statuses

context_predictions[
    "reproduction_status_confidence"
] = np.nan

context_predictions.loc[
    missing_repro_mask,
    "reproduction_status_confidence"
] = prediction_confidence

print("\nREPRODUCTION-STATUS IMPUTATION RESULTS")
print("Predictions created:", len(predicted_statuses))
print(
    "Predicted values still missing:",
    context_predictions[
        "reproduction_status_predicted"
    ].isna().sum()
)

display(
    context_predictions.loc[
        missing_repro_mask,
        "reproduction_status_predicted"
    ].value_counts().to_frame("predictions")
)

display(
    context_predictions[
        [
            "source_row",
            "reproduction_status",
            "reproduction_status_predicted",
            "reproduction_status_confidence"
        ]
    ].head(10)
)

Formula A error: 0.0
Formula B error: 0.3778312950860786
Using: flow_30_60_session / avg_milk_flow
Prediction feature shape: (939749, 17)

REPRODUCTION-STATUS IMPUTATION RESULTS
Predictions created: 939749
Predicted values still missing: 0


,predictions
reproduction_status_predicted,
pregnant,554020
bred,297551
fresh,57467
open,30711


,source_row,reproduction_status,reproduction_status_predicted,reproduction_status_confidence
2,3,<NA>,bred,0.323464
4,5,<NA>,pregnant,0.422883
5,6,<NA>,pregnant,0.343145
10,11,<NA>,pregnant,0.381782
12,13,<NA>,pregnant,0.473542
24,25,<NA>,pregnant,0.357774
26,27,<NA>,pregnant,0.571092
28,29,<NA>,pregnant,0.444099
29,30,<NA>,pregnant,0.487285
31,32,<NA>,bred,0.322194


In [61]:
from pathlib import Path

# Locate the processed-data folder
if Path.cwd().name == "notebooks":
    output_directory = Path("../data/processed")
else:
    output_directory = Path("data/processed")

output_directory.mkdir(
    parents=True,
    exist_ok=True
)

reproduction_output_path = (
    output_directory
    / "reproduction_status_predictions.csv"
)

reproduction_checkpoint = context_predictions[
    [
        "source_row",
        "reproduction_status",
        "reproduction_status_predicted",
        "reproduction_status_confidence"
    ]
].copy()

reproduction_checkpoint.to_csv(
    reproduction_output_path,
    index=False
)

print("Rows saved:", len(reproduction_checkpoint))
print("Saved to:", reproduction_output_path.resolve())
print(
    "Missing predictions:",
    reproduction_checkpoint[
        "reproduction_status_predicted"
    ].isna().sum()
)

Rows saved: 939751
Saved to: /Users/finmaroney/Desktop/project-1-dairy-missing-data/data/processed/reproduction_status_predictions.csv
Missing predictions: 0


In [62]:
# Part 30: Identify the fitted days-in-milk model and its variables

import numpy as np
import pandas as pd

print("DAYS-IN-MILK MODEL CANDIDATES")

for name, value in list(globals().items()):
    lower_name = name.lower()

    if (
        ("dim" in lower_name or "stage" in lower_name)
        and hasattr(value, "predict")
    ):
        print(
            f"{name}:",
            type(value).__name__,
            "| features:",
            getattr(value, "n_features_in_", "unknown"),
            "| classes:",
            getattr(value, "classes_", "not applicable"),
        )

print("\nFEATURE-DATA CANDIDATES")

for name, value in list(globals().items()):
    lower_name = name.lower()

    if (
        ("dim" in lower_name or "stage" in lower_name)
        and hasattr(value, "shape")
    ):
        print(f"{name}: shape={value.shape}")

print("\nSTAGE DEFINITIONS")

for name, value in list(globals().items()):
    lower_name = name.lower()

    if (
        ("stage" in lower_name or "dim" in lower_name)
        and any(
            word in lower_name
            for word in ["edge", "bound", "midpoint", "center", "label"]
        )
    ):
        try:
            array = np.asarray(value)

            if array.size <= 50:
                print(f"{name}: {array}")
        except Exception:
            pass

DAYS-IN-MILK MODEL CANDIDATES
dim_model: HistGradientBoostingRegressor | features: 17 | classes: not applicable
stage_model: HistGradientBoostingClassifier | features: 17 | classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]

FEATURE-DATA CANDIDATES
dim_train_raw: shape=(1000000, 8)
dim_validation_raw: shape=(100000, 8)
dim_train: shape=(1000000, 8)
dim_validation: shape=(100000, 8)
dim_lookup: shape=(586464, 9)
dim_matches: shape=(100000, 11)
dim_lookup_stage2: shape=(796874, 10)
two_stage_matches: shape=(100000, 14)
two_stage_results: shape=(9, 8)
dim_model_train: shape=(1000000, 16)
dim_model_validation: shape=(100000, 16)
X_dim_train: shape=(400000, 17)
X_dim_validation: shape=(100000, 17)
y_dim_train: shape=(400000,)
y_dim_validation: shape=(100000,)
dim_predictions: shape=(100000,)
dim_errors: shape=(100000,)
dim_rmse: shape=()
y_train_stage: shape=(400000,)
y_validation_stage: shape=(100000,)
stage_indices: shape=(250000,)
X_stage_train: shape=(250000, 17)
y_stage_train: sha

In [63]:
# Part 31: Apply the high-confidence days-in-milk model

import numpy as np
import pandas as pd

selected_dim_threshold = 0.45

# Include every row missing days_in_milk
dim_missing_data = context_to_predict.loc[
    pd.to_numeric(
        context_to_predict["days_in_milk"],
        errors="coerce",
    ).isna()
].copy()

print("Missing days-in-milk rows:", len(dim_missing_data))


def build_dim_features(data, feature_columns, training_features):
    working = data.copy()

    # Date-based features
    event_dates = pd.to_datetime(
        working["event_date"],
        errors="coerce",
    )

    working["event_year"] = event_dates.dt.year
    working["event_month"] = event_dates.dt.month
    working["event_day"] = event_dates.dt.day
    working["day_of_year"] = event_dates.dt.dayofyear
    working["day_of_week"] = event_dates.dt.dayofweek

    working["year_sin"] = np.sin(
        2 * np.pi * working["day_of_year"] / 365.25
    )
    working["year_cos"] = np.cos(
        2 * np.pi * working["day_of_year"] / 365.25
    )

    # Milk-session features
    milk_yield = pd.to_numeric(
        working.get("milk_yield"),
        errors="coerce",
    )

    duration = pd.to_numeric(
        working.get("duration_session_sec"),
        errors="coerce",
    )

    first_two_minutes = pd.to_numeric(
        working.get("yield_first_2_min_session"),
        errors="coerce",
    )

    flow_30_60 = pd.to_numeric(
        working.get("flow_30_60_session"),
        errors="coerce",
    )

    average_flow = pd.to_numeric(
        working.get("avg_milk_flow"),
        errors="coerce",
    )

    working["yield_per_second"] = (
        milk_yield / duration.replace(0, np.nan)
    )

    working["first_2_min_fraction"] = (
        first_two_minutes / milk_yield.replace(0, np.nan)
    )

    working["flow_ratio"] = (
        flow_30_60 / average_flow.replace(0, np.nan)
    )

    # Construct columns in exactly the same order as training
    features = pd.DataFrame(
        index=working.index,
        columns=feature_columns,
        dtype=float,
    )

    for column in feature_columns:
        if column in working.columns:
            features[column] = pd.to_numeric(
                working[column],
                errors="coerce",
            )

    features = features.replace([np.inf, -np.inf], np.nan)

    # Use training medians for unavailable values
    training_medians = training_features[
        feature_columns
    ].median(numeric_only=True)

    features = features.fillna(training_medians)
    features = features.fillna(0)

    return features


dim_feature_columns = X_dim_validation.columns.tolist()

X_missing_dim = build_dim_features(
    dim_missing_data,
    dim_feature_columns,
    X_dim_train,
)

print("Prediction feature shape:", X_missing_dim.shape)

# Predict stages and confidence
dim_stage_probabilities = stage_model.predict_proba(X_missing_dim)

dim_stage_confidence = dim_stage_probabilities.max(axis=1)

dim_predicted_stages = stage_model.classes_[
    dim_stage_probabilities.argmax(axis=1)
].astype(int)

# Each class represents a 60-day interval.
# Use its midpoint as the numeric prediction.
dim_numeric_predictions = (
    dim_predicted_stages * 60 + 30
).astype(float)

# Keep predictions inside the observed training range
maximum_training_dim = float(
    pd.to_numeric(
        pd.Series(y_dim_train),
        errors="coerce",
    ).max()
)

dim_numeric_predictions = np.clip(
    dim_numeric_predictions,
    1,
    maximum_training_dim,
)

dim_accepted_mask = (
    dim_stage_confidence >= selected_dim_threshold
)

# Build the result table
dim_predictions = dim_missing_data[
    ["source_row", "days_in_milk"]
].copy()

dim_predictions["days_in_milk_stage_predicted"] = (
    dim_predicted_stages
)

dim_predictions["days_in_milk_confidence"] = (
    dim_stage_confidence
)

dim_predictions["days_in_milk_predicted"] = np.where(
    dim_accepted_mask,
    dim_numeric_predictions,
    np.nan,
)

dim_predictions["days_in_milk_prediction_accepted"] = (
    dim_accepted_mask
)

print("\nDAYS-IN-MILK IMPUTATION RESULTS")
print("Selected confidence:", selected_dim_threshold)
print("Rows evaluated:", len(dim_predictions))
print("Accepted predictions:", int(dim_accepted_mask.sum()))
print(
    "Coverage:",
    round(dim_accepted_mask.mean() * 100, 2),
    "%",
)
print(
    "Rows intentionally left unfilled:",
    int((~dim_accepted_mask).sum()),
)

display(
    dim_predictions.loc[
        dim_predictions["days_in_milk_prediction_accepted"]
    ].head(10)
)

Missing days-in-milk rows: 939751
Prediction feature shape: (939751, 17)

DAYS-IN-MILK IMPUTATION RESULTS
Selected confidence: 0.45
Rows evaluated: 939751
Accepted predictions: 22417
Coverage: 2.39 %
Rows intentionally left unfilled: 917334


,source_row,days_in_milk,days_in_milk_stage_predicted,days_in_milk_confidence,days_in_milk_predicted,days_in_milk_prediction_accepted
192,193,NaN,1,0.487737,90.0,True
209,210,NaN,1,0.508163,90.0,True
311,312,NaN,0,0.532877,30.0,True
380,381,NaN,1,0.450298,90.0,True
585,586,NaN,0,0.466143,30.0,True
1133,1134,NaN,1,0.450008,90.0,True
1289,1290,NaN,0,0.503943,30.0,True
2016,2017,NaN,1,0.451908,90.0,True
2198,2199,NaN,0,0.656288,30.0,True
2545,2546,NaN,0,0.465111,30.0,True


In [64]:
# Part 32: Save the days-in-milk prediction checkpoint

from pathlib import Path

if "output_directory" not in globals():
    if Path.cwd().name == "notebooks":
        output_directory = Path("../data/processed")
    else:
        output_directory = Path("data/processed")

output_directory.mkdir(parents=True, exist_ok=True)

dim_output_path = (
    output_directory / "days_in_milk_predictions.csv"
)

dim_predictions.to_csv(
    dim_output_path,
    index=False,
)

print("Rows saved:", len(dim_predictions))
print(
    "Accepted predictions:",
    int(
        dim_predictions[
            "days_in_milk_prediction_accepted"
        ].sum()
    ),
)
print(
    "Unfilled predictions:",
    dim_predictions[
        "days_in_milk_predicted"
    ].isna().sum(),
)
print("Saved to:", dim_output_path.resolve())

Rows saved: 939751
Accepted predictions: 22417
Unfilled predictions: 917334
Saved to: /Users/finmaroney/Desktop/project-1-dairy-missing-data/data/processed/days_in_milk_predictions.csv


In [65]:
# Part 33: Combine predictions with the complete dataset

import pandas as pd

# Reconstruct the complete original dataset
final_cleaned_data = pd.concat(
    [
        known_context.copy(),
        context_to_predict.copy(),
    ],
    ignore_index=True,
)

# Verify that source_row uniquely identifies every row
if final_cleaned_data["source_row"].duplicated().any():
    raise ValueError(
        "Duplicate source_row values found. "
        "Stop before combining predictions."
    )

final_cleaned_data = (
    final_cleaned_data
    .sort_values("source_row")
    .reset_index(drop=True)
)

print("Complete dataset rows:", len(final_cleaned_data))


# ---------------------------------------------------------
# 1. Add reproduction-status predictions
# ---------------------------------------------------------

reproduction_lookup = (
    context_predictions
    .drop_duplicates("source_row")
    .set_index("source_row")
)

final_cleaned_data["reproduction_status_predicted"] = (
    final_cleaned_data["source_row"].map(
        reproduction_lookup[
            "reproduction_status_predicted"
        ]
    )
)

final_cleaned_data["reproduction_status_confidence"] = (
    final_cleaned_data["source_row"].map(
        reproduction_lookup[
            "reproduction_status_confidence"
        ]
    )
)

reproduction_fill_mask = (
    final_cleaned_data["reproduction_status"].isna()
    & final_cleaned_data[
        "reproduction_status_predicted"
    ].notna()
)

final_cleaned_data[
    "reproduction_status_was_imputed"
] = reproduction_fill_mask

# Convert to object so categorical values can be inserted safely
final_cleaned_data["reproduction_status"] = (
    final_cleaned_data["reproduction_status"].astype("object")
)

final_cleaned_data.loc[
    reproduction_fill_mask,
    "reproduction_status",
] = final_cleaned_data.loc[
    reproduction_fill_mask,
    "reproduction_status_predicted",
]


# ---------------------------------------------------------
# 2. Add only accepted days-in-milk predictions
# ---------------------------------------------------------

dim_lookup = (
    dim_predictions
    .drop_duplicates("source_row")
    .set_index("source_row")
)

final_cleaned_data["days_in_milk_predicted"] = (
    final_cleaned_data["source_row"].map(
        dim_lookup["days_in_milk_predicted"]
    )
)

final_cleaned_data["days_in_milk_confidence"] = (
    final_cleaned_data["source_row"].map(
        dim_lookup["days_in_milk_confidence"]
    )
)

final_cleaned_data[
    "days_in_milk_prediction_accepted"
] = (
    final_cleaned_data["source_row"]
    .map(
        dim_lookup[
            "days_in_milk_prediction_accepted"
        ]
    )
    .fillna(False)
    .astype(bool)
)

dim_fill_mask = (
    pd.to_numeric(
        final_cleaned_data["days_in_milk"],
        errors="coerce",
    ).isna()
    & final_cleaned_data[
        "days_in_milk_prediction_accepted"
    ]
    & final_cleaned_data[
        "days_in_milk_predicted"
    ].notna()
)

final_cleaned_data[
    "days_in_milk_was_imputed"
] = dim_fill_mask

final_cleaned_data["days_in_milk"] = pd.to_numeric(
    final_cleaned_data["days_in_milk"],
    errors="coerce",
)

final_cleaned_data.loc[
    dim_fill_mask,
    "days_in_milk",
] = final_cleaned_data.loc[
    dim_fill_mask,
    "days_in_milk_predicted",
]


# ---------------------------------------------------------
# 3. Verify the combined results
# ---------------------------------------------------------

print("\nCOMBINED IMPUTATION RESULTS")
print(
    "Reproduction-status values filled:",
    int(reproduction_fill_mask.sum()),
)
print(
    "Days-in-milk values filled:",
    int(dim_fill_mask.sum()),
)
print(
    "Reproduction status still missing:",
    final_cleaned_data[
        "reproduction_status"
    ].isna().sum(),
)
print(
    "Days in milk still missing:",
    final_cleaned_data[
        "days_in_milk"
    ].isna().sum(),
)

display(
    final_cleaned_data.loc[
        reproduction_fill_mask | dim_fill_mask,
        [
            "source_row",
            "days_in_milk",
            "days_in_milk_confidence",
            "days_in_milk_was_imputed",
            "reproduction_status",
            "reproduction_status_confidence",
            "reproduction_status_was_imputed",
        ],
    ].head(10)
)

Complete dataset rows: 4688514

COMBINED IMPUTATION RESULTS
Reproduction-status values filled: 939749
Days-in-milk values filled: 22417
Reproduction status still missing: 0
Days in milk still missing: 917334


,source_row,days_in_milk,days_in_milk_confidence,days_in_milk_was_imputed,reproduction_status,reproduction_status_confidence,reproduction_status_was_imputed
2,3,NaN,0.247628,False,bred,0.323464,True
4,5,NaN,0.251335,False,pregnant,0.422883,True
5,6,NaN,0.306678,False,pregnant,0.343145,True
10,11,NaN,0.264102,False,pregnant,0.381782,True
12,13,NaN,0.308222,False,pregnant,0.473542,True
24,25,NaN,0.264823,False,pregnant,0.357774,True
26,27,NaN,0.243204,False,pregnant,0.571092,True
28,29,NaN,0.254487,False,pregnant,0.444099,True
29,30,NaN,0.349132,False,pregnant,0.487285,True
31,32,NaN,0.286288,False,bred,0.322194,True


In [66]:
# Part 34: Save the final cleaned dataset

from pathlib import Path

if "output_directory" not in globals():
    if Path.cwd().name == "notebooks":
        output_directory = Path("../data/processed")
    else:
        output_directory = Path("data/processed")

output_directory.mkdir(parents=True, exist_ok=True)

# Remove redundant prediction columns while retaining audit information
columns_to_remove = [
    "days_in_milk_predicted",
    "days_in_milk_prediction_accepted",
    "reproduction_status_predicted",
]

final_export = final_cleaned_data.drop(
    columns=[
        column
        for column in columns_to_remove
        if column in final_cleaned_data.columns
    ]
).copy()

final_output_path = (
    output_directory / "dairy_data_imputed_final.csv"
)

print("Saving", len(final_export), "rows...")
print("This may take several minutes because the dataset is large.")

final_export.to_csv(
    final_output_path,
    index=False,
)

print("\nFINAL DATASET SAVED")
print("Rows:", len(final_export))
print("Columns:", len(final_export.columns))
print("Saved to:", final_output_path.resolve())

print("\nFINAL MISSING VALUES")
print(
    final_export[
        [
            "days_in_milk",
            "reproduction_status",
        ]
    ].isna().sum()
)

Saving 4688514 rows...
This may take several minutes because the dataset is large.

FINAL DATASET SAVED
Rows: 4688514
Columns: 20
Saved to: /Users/finmaroney/Desktop/project-1-dairy-missing-data/data/processed/dairy_data_imputed_final.csv

FINAL MISSING VALUES
days_in_milk           917334
reproduction_status         0
dtype: int64


In [67]:
# Part 35: Identify lactation-number prediction objects

print("LACTATION MODEL OBJECTS")

for name, value in list(globals().items()):
    lower_name = name.lower()

    if (
        "lactation" in lower_name
        or "weekly" in lower_name
    ):
        if hasattr(value, "predict"):
            print(
                name,
                "| type:", type(value).__name__,
                "| features:",
                getattr(value, "n_features_in_", "unknown"),
                "| classes:",
                getattr(value, "classes_", "not applicable"),
            )

print("\nLACTATION TABLES AND ARRAYS")

for name, value in list(globals().items()):
    lower_name = name.lower()

    if (
        "lactation" in lower_name
        or "weekly" in lower_name
    ) and hasattr(value, "shape"):
        # Focus on useful prediction and lookup objects
        if any(
            word in lower_name
            for word in [
                "lookup",
                "accepted",
                "prediction",
                "match",
                "reference",
                "train",
                "test",
            ]
        ):
            print(f"{name}: shape={value.shape}")

            if hasattr(value, "columns"):
                print(
                    "  columns:",
                    value.columns.tolist(),
                )

LACTATION MODEL OBJECTS
lactation_model | type: Pipeline | features: 10 | classes: [ 1  2  3  4  5  6  7  8  9 10 11 12]
ordinal_lactation_model | type: Pipeline | features: 10 | classes: not applicable

LACTATION TABLES AND ARRAYS
X_lactation_train: shape=(2624890, 10)
  columns: ['avg_milk_flow', 'flow_30_60_session', 'yield_first_2_min_session', 'milk_yield', 'duration_session_sec', 'event_year', 'event_month', 'event_day_of_year', 'event_day_of_week', 'milking']
y_lactation_train: shape=(2624890,)
lactation_validation_predictions: shape=(562172,)
raw_lactation_predictions: shape=(562172,)
weekly_train: shape=(1000000, 7)
  columns: ['event_week', 'milking', 'milk_yield_bin', 'duration_bin', 'flow_bin', 'first_2_min_bin', 'lactation_number']
weekly_lookup: shape=(586464, 10)
  columns: ['event_week', 'milking', 'milk_yield_bin', 'duration_bin', 'flow_bin', 'first_2_min_bin', 'predicted_lactation', 'class_count', 'neighbor_records', 'confidence']
weekly_matches: shape=(100000, 10)
  

In [68]:
# Part 36: Inspect the weekly lactation lookup structure

tables_to_check = {
    "full_weekly_train": full_weekly_train,
    "lactation_lookup": lactation_lookup,
    "weekly_test": weekly_test,
}

for table_name, table in tables_to_check.items():
    print(f"\n{table_name.upper()}")
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())
    print("\nData types:")
    print(table.dtypes)
    print("\nFirst 3 rows:")
    display(table.head(3))


print("\nPOSSIBLE BIN-DEFINITION VARIABLES")

for name, value in list(globals().items()):
    lower_name = name.lower()

    if "bin" in lower_name:
        try:
            if hasattr(value, "__len__") and len(value) <= 100:
                print(f"{name}: {value}")
        except Exception:
            pass


print("\nPOSSIBLE FEATURE-BUILDING FUNCTIONS")

for name, value in list(globals().items()):
    lower_name = name.lower()

    if callable(value) and any(
        word in lower_name
        for word in ["weekly", "lactation", "bin"]
    ):
        print(name)


FULL_WEEKLY_TRAIN
Shape: (2624890, 7)
Columns: ['event_week', 'milking', 'milk_yield_bin', 'duration_bin', 'flow_bin', 'first_2_min_bin', 'lactation_number']

Data types:
event_week           string
milking              string
milk_yield_bin      float64
duration_bin        float64
flow_bin            float64
first_2_min_bin     float64
lactation_number    float64
dtype: object

First 3 rows:


,event_week,milking,milk_yield_bin,duration_bin,flow_bin,first_2_min_bin,lactation_number
1,2021-07-12/2021-07-18,2,15.0,270.0,1.50,5.5,1.0
6,2019-12-09/2019-12-15,1,12.0,210.0,4.50,8.0,1.0
7,2020-05-11/2020-05-17,2,9.0,150.0,4.25,7.5,5.0



LACTATION_LOOKUP
Shape: (586464, 9)
Columns: ['event_week', 'milking', 'milk_yield_bin', 'duration_bin', 'flow_bin', 'first_2_min_bin', 'predicted_lactation', 'lactation_confidence', 'total_count']

Data types:
event_week               string
milking                  string
milk_yield_bin          float64
duration_bin            float64
flow_bin                float64
first_2_min_bin         float64
predicted_lactation     float64
lactation_confidence    float64
total_count               int64
dtype: object

First 3 rows:


,event_week,milking,milk_yield_bin,duration_bin,flow_bin,first_2_min_bin,predicted_lactation,lactation_confidence,total_count
46707,2020-05-25/2020-05-31,2,13.0,240.0,3.5,6.5,1.0,0.880000,25
125671,2020-11-09/2020-11-15,3,13.0,210.0,4.0,7.5,1.0,0.857143,21
627,2020-06-22/2020-06-28,1,11.0,210.0,3.5,6.5,1.0,0.894737,19



WEEKLY_TEST
Shape: (561701, 7)
Columns: ['event_week', 'milking', 'milk_yield_bin', 'duration_bin', 'flow_bin', 'first_2_min_bin', 'lactation_number']

Data types:
event_week           string
milking              string
milk_yield_bin      float64
duration_bin        float64
flow_bin            float64
first_2_min_bin     float64
lactation_number    float64
dtype: object

First 3 rows:


,event_week,milking,milk_yield_bin,duration_bin,flow_bin,first_2_min_bin,lactation_number
3,2019-12-02/2019-12-08,3,15.0,240.0,4.00,8.5,1.0
8,2021-03-22/2021-03-28,2,15.0,210.0,5.25,9.5,4.0
16,2021-06-21/2021-06-27,1,9.0,180.0,4.00,7.0,2.0



POSSIBLE BIN-DEFINITION VARIABLES

POSSIBLE FEATURE-BUILDING FUNCTIONS
create_neighbor_bins
create_weekly_bins


In [69]:
# Part 37: Apply weekly lactation-number predictions

import numpy as np
import pandas as pd

selected_lactation_confidence = 0.55

lactation_key_columns = [
    "event_week",
    "milking",
    "milk_yield_bin",
    "duration_bin",
    "flow_bin",
    "first_2_min_bin",
]

# Rows that still need a lactation number
missing_lactation = final_cleaned_data.loc[
    pd.to_numeric(
        final_cleaned_data["lactation_number"],
        errors="coerce",
    ).isna()
].copy()

print(
    "Missing lactation-number rows:",
    len(missing_lactation),
)


# Reproduce the same weekly features used to build the lookup
event_dates = pd.to_datetime(
    missing_lactation["event_date"],
    errors="coerce",
)

missing_lactation["event_week"] = (
    event_dates.dt.to_period("W").astype("string")
)

missing_lactation["milking"] = (
    missing_lactation["milking"].astype("string")
)

milk_yield = pd.to_numeric(
    missing_lactation["milk_yield"],
    errors="coerce",
)

duration = pd.to_numeric(
    missing_lactation["duration_session_sec"],
    errors="coerce",
)

average_flow = pd.to_numeric(
    missing_lactation["avg_milk_flow"],
    errors="coerce",
)

first_two_minutes = pd.to_numeric(
    missing_lactation["yield_first_2_min_session"],
    errors="coerce",
)

# Same bin widths used in full_weekly_train
missing_lactation["milk_yield_bin"] = (
    milk_yield.round(0)
)

missing_lactation["duration_bin"] = (
    (duration / 10).round() * 10
)

missing_lactation["flow_bin"] = (
    (average_flow * 4).round() / 4
)

missing_lactation["first_2_min_bin"] = (
    (first_two_minutes * 2).round() / 2
)


# Ensure one lookup result per signature
clean_lactation_lookup = (
    lactation_lookup
    .sort_values(
        ["lactation_confidence", "total_count"]
    )
    .drop_duplicates(
        subset=lactation_key_columns,
        keep="last",
    )
)

# Match missing records to the lookup
lactation_predictions = missing_lactation[
    ["source_row", "lactation_number"]
    + lactation_key_columns
].merge(
    clean_lactation_lookup,
    on=lactation_key_columns,
    how="left",
    validate="many_to_one",
)

lactation_predictions[
    "lactation_prediction_accepted"
] = (
    lactation_predictions[
        "predicted_lactation"
    ].notna()
    & (
        lactation_predictions[
            "lactation_confidence"
        ] >= selected_lactation_confidence
    )
)

lactation_predictions[
    "lactation_number_predicted"
] = np.where(
    lactation_predictions[
        "lactation_prediction_accepted"
    ],
    lactation_predictions["predicted_lactation"],
    np.nan,
)


# Add predictions to the complete dataset
lactation_result_lookup = (
    lactation_predictions
    .drop_duplicates("source_row")
    .set_index("source_row")
)

final_cleaned_data[
    "lactation_number_predicted"
] = final_cleaned_data["source_row"].map(
    lactation_result_lookup[
        "lactation_number_predicted"
    ]
)

final_cleaned_data[
    "lactation_number_confidence"
] = final_cleaned_data["source_row"].map(
    lactation_result_lookup[
        "lactation_confidence"
    ]
)

final_cleaned_data[
    "lactation_number_prediction_accepted"
] = (
    final_cleaned_data["source_row"]
    .map(
        lactation_result_lookup[
            "lactation_prediction_accepted"
        ]
    )
    .fillna(False)
    .astype(bool)
)

lactation_fill_mask = (
    pd.to_numeric(
        final_cleaned_data["lactation_number"],
        errors="coerce",
    ).isna()
    & final_cleaned_data[
        "lactation_number_prediction_accepted"
    ]
    & final_cleaned_data[
        "lactation_number_predicted"
    ].notna()
)

final_cleaned_data[
    "lactation_number_was_imputed"
] = lactation_fill_mask

final_cleaned_data["lactation_number"] = pd.to_numeric(
    final_cleaned_data["lactation_number"],
    errors="coerce",
)

final_cleaned_data.loc[
    lactation_fill_mask,
    "lactation_number",
] = final_cleaned_data.loc[
    lactation_fill_mask,
    "lactation_number_predicted",
]


# Results
accepted_lactation_count = int(
    lactation_fill_mask.sum()
)

print("\nLACTATION IMPUTATION RESULTS")
print(
    "Selected confidence:",
    selected_lactation_confidence,
)
print(
    "Accepted predictions:",
    accepted_lactation_count,
)
print(
    "Coverage:",
    round(
        accepted_lactation_count
        / len(missing_lactation)
        * 100,
        2,
    ),
    "%",
)
print(
    "Lactation numbers still missing:",
    final_cleaned_data[
        "lactation_number"
    ].isna().sum(),
)

display(
    final_cleaned_data.loc[
        lactation_fill_mask,
        [
            "source_row",
            "lactation_number",
            "lactation_number_confidence",
            "lactation_number_was_imputed",
        ],
    ].head(10)
)

Missing lactation-number rows: 939749

LACTATION IMPUTATION RESULTS
Selected confidence: 0.55
Accepted predictions: 85236
Coverage: 9.07 %
Lactation numbers still missing: 854513


,source_row,lactation_number,lactation_number_confidence,lactation_number_was_imputed
5,6,6.0,1.000000,True
34,35,2.0,1.000000,True
71,72,1.0,0.666667,True
137,138,6.0,1.000000,True
284,285,5.0,1.000000,True
315,316,5.0,1.000000,True
330,331,2.0,0.666667,True
388,389,2.0,1.000000,True
421,422,2.0,1.000000,True
433,434,1.0,1.000000,True


In [70]:
# Part 38: Find the broader weekly lactation objects

print("BROAD LACTATION OBJECTS")

for name, value in list(globals().items()):
    lower_name = name.lower()

    if any(
        word in lower_name
        for word in ["broad", "threshold"]
    ):
        if hasattr(value, "shape"):
            print(f"\n{name}: shape={value.shape}")

            if hasattr(value, "columns"):
                print(
                    "Columns:",
                    value.columns.tolist(),
                )
                display(value.head(3))
        else:
            try:
                print(f"{name}: {value}")
            except Exception:
                pass

BROAD LACTATION OBJECTS

possible_thresholds: shape=(0, 9)
Columns: ['true_cow_id', 'predicted_cow_id', 'best_distance', 'second_distance', 'correct', 'confidence', 'accepted_count', 'cumulative_accuracy', 'coverage_percent']


,true_cow_id,predicted_cow_id,best_distance,second_distance,correct,confidence,accepted_count,cumulative_accuracy,coverage_percent


cow_id_confidence_threshold: None
threshold: 0.7
threshold_results: [{'minimum_confidence': 0.5, 'accepted_predictions': 188469, 'coverage_percent': 33.53, 'exact_accuracy': np.float64(0.4797)}, {'minimum_confidence': 0.55, 'accepted_predictions': 136740, 'coverage_percent': 24.32, 'exact_accuracy': np.float64(0.5117)}, {'minimum_confidence': 0.6, 'accepted_predictions': 119219, 'coverage_percent': 21.21, 'exact_accuracy': np.float64(0.5136)}, {'minimum_confidence': 0.65, 'accepted_predictions': 92605, 'coverage_percent': 16.47, 'exact_accuracy': np.float64(0.5187)}, {'minimum_confidence': 0.7, 'accepted_predictions': 54252, 'coverage_percent': 9.65, 'exact_accuracy': np.float64(0.5708)}]

broad_test_accepted: shape=(145173, 10)
Columns: ['event_week', 'milking', 'milk_yield_bin', 'duration_bin', 'flow_bin', 'first_2_min_bin', 'lactation_number', 'predicted_lactation', 'neighbor_records', 'confidence']


,event_week,milking,milk_yield_bin,duration_bin,flow_bin,first_2_min_bin,lactation_number,predicted_lactation,neighbor_records,confidence
5,2020-02-24/2020-03-01,2,12.0,240.0,3.00,6.0,2.0,1.0,16.0,0.937500
17,2019-12-16/2019-12-22,2,12.0,270.0,2.50,5.0,1.0,1.0,14.0,0.714286
18,2019-12-02/2019-12-08,1,19.0,330.0,3.75,7.0,4.0,3.0,3.0,0.666667



broad_test_accuracy: shape=()
broad_test_coverage: 25.845245068105626

eligible_thresholds: shape=(6, 6)
Columns: ['minimum_confidence', 'accepted_predictions', 'coverage_percent', 'stage_accuracy', 'MAE_days', 'within_30_days']


,minimum_confidence,accepted_predictions,coverage_percent,stage_accuracy,MAE_days,within_30_days
5,0.45,2390,2.39,0.4979,47.58,0.5079
6,0.46,1917,1.92,0.5065,48.13,0.5159
7,0.47,1586,1.59,0.5101,49.27,0.5195


selected_dim_threshold: 0.45
selected_repro_threshold: 0.45


In [71]:
# Part 39: Add broader weekly lactation predictions

import numpy as np
import pandas as pd

broad_lactation_threshold = 0.55

lactation_key_columns = [
    "event_week",
    "milking",
    "milk_yield_bin",
    "duration_bin",
    "flow_bin",
    "first_2_min_bin",
]

print("Building final lactation lookup from all known records...")

# Together these contain all 3,748,763 known records
all_known_weekly = pd.concat(
    [
        weekly_reference[
            lactation_key_columns + ["lactation_number"]
        ],
        weekly_test[
            lactation_key_columns + ["lactation_number"]
        ],
    ],
    ignore_index=True,
).dropna(
    subset=lactation_key_columns + ["lactation_number"]
)

# Count each lactation value within each weekly signature
broad_signature_counts = (
    all_known_weekly
    .groupby(
        lactation_key_columns + ["lactation_number"],
        observed=True,
    )
    .size()
    .rename("neighbor_records")
    .reset_index()
)

# Count all reference records for each signature
broad_signature_totals = (
    broad_signature_counts
    .groupby(
        lactation_key_columns,
        observed=True,
    )["neighbor_records"]
    .sum()
    .rename("total_records")
    .reset_index()
)

# Select the most common lactation for each signature
broad_lactation_lookup = (
    broad_signature_counts
    .sort_values(
        lactation_key_columns
        + ["neighbor_records", "lactation_number"]
    )
    .drop_duplicates(
        subset=lactation_key_columns,
        keep="last",
    )
    .rename(
        columns={
            "lactation_number": "predicted_lactation"
        }
    )
    .merge(
        broad_signature_totals,
        on=lactation_key_columns,
        how="left",
        validate="one_to_one",
    )
)

broad_lactation_lookup["confidence"] = (
    broad_lactation_lookup["neighbor_records"]
    / broad_lactation_lookup["total_records"]
)

print(
    "Final lookup signatures:",
    len(broad_lactation_lookup),
)


# ---------------------------------------------------------
# Create weekly features for only the still-missing rows
# ---------------------------------------------------------

remaining_lactation = final_cleaned_data.loc[
    pd.to_numeric(
        final_cleaned_data["lactation_number"],
        errors="coerce",
    ).isna()
].copy()

event_dates = pd.to_datetime(
    remaining_lactation["event_date"],
    errors="coerce",
)

remaining_lactation["event_week"] = (
    event_dates.dt.to_period("W").astype("string")
)

remaining_lactation["milking"] = (
    remaining_lactation["milking"].astype("string")
)

milk_yield = pd.to_numeric(
    remaining_lactation["milk_yield"],
    errors="coerce",
)

duration = pd.to_numeric(
    remaining_lactation["duration_session_sec"],
    errors="coerce",
)

average_flow = pd.to_numeric(
    remaining_lactation["avg_milk_flow"],
    errors="coerce",
)

first_two_minutes = pd.to_numeric(
    remaining_lactation["yield_first_2_min_session"],
    errors="coerce",
)

remaining_lactation["milk_yield_bin"] = (
    milk_yield.round(0)
)

remaining_lactation["duration_bin"] = (
    (duration / 10).round() * 10
)

remaining_lactation["flow_bin"] = (
    (average_flow * 4).round() / 4
)

remaining_lactation["first_2_min_bin"] = (
    (first_two_minutes * 2).round() / 2
)


# ---------------------------------------------------------
# Match remaining rows to the broader lookup
# ---------------------------------------------------------

broad_lactation_predictions = (
    remaining_lactation[
        ["source_row"] + lactation_key_columns
    ]
    .merge(
        broad_lactation_lookup,
        on=lactation_key_columns,
        how="left",
        validate="many_to_one",
    )
)

broad_lactation_predictions[
    "accepted"
] = (
    broad_lactation_predictions[
        "predicted_lactation"
    ].notna()
    & (
        broad_lactation_predictions["confidence"]
        >= broad_lactation_threshold
    )
)

accepted_broad_predictions = (
    broad_lactation_predictions.loc[
        broad_lactation_predictions["accepted"]
    ]
    .drop_duplicates("source_row")
    .set_index("source_row")
)

broad_prediction_map = final_cleaned_data[
    "source_row"
].map(
    accepted_broad_predictions[
        "predicted_lactation"
    ]
)

broad_confidence_map = final_cleaned_data[
    "source_row"
].map(
    accepted_broad_predictions["confidence"]
)

broad_fill_mask = (
    final_cleaned_data["lactation_number"].isna()
    & broad_prediction_map.notna()
)


# Track which method supplied each value
final_cleaned_data[
    "lactation_imputation_method"
] = np.where(
    final_cleaned_data[
        "lactation_number_was_imputed"
    ],
    "strict_weekly",
    "original",
)

final_cleaned_data.loc[
    final_cleaned_data["lactation_number"].isna(),
    "lactation_imputation_method",
] = "unfilled"

# Insert only the additional accepted predictions
final_cleaned_data.loc[
    broad_fill_mask,
    "lactation_number",
] = broad_prediction_map.loc[broad_fill_mask]

final_cleaned_data.loc[
    broad_fill_mask,
    "lactation_number_predicted",
] = broad_prediction_map.loc[broad_fill_mask]

final_cleaned_data.loc[
    broad_fill_mask,
    "lactation_number_confidence",
] = broad_confidence_map.loc[broad_fill_mask]

final_cleaned_data.loc[
    broad_fill_mask,
    "lactation_number_prediction_accepted",
] = True

final_cleaned_data.loc[
    broad_fill_mask,
    "lactation_number_was_imputed",
] = True

final_cleaned_data.loc[
    broad_fill_mask,
    "lactation_imputation_method",
] = "broad_weekly"


# ---------------------------------------------------------
# Final lactation summary
# ---------------------------------------------------------

strict_count = int(
    (
        final_cleaned_data[
            "lactation_imputation_method"
        ] == "strict_weekly"
    ).sum()
)

broad_count = int(broad_fill_mask.sum())
total_lactation_filled = strict_count + broad_count

print("\nFINAL LACTATION RESULTS")
print("Strict predictions retained:", strict_count)
print("Additional broad predictions:", broad_count)
print(
    "Total lactation predictions:",
    total_lactation_filled,
)
print(
    "Total coverage:",
    round(
        total_lactation_filled / 939749 * 100,
        2,
    ),
    "%",
)
print(
    "Lactation numbers still missing:",
    final_cleaned_data[
        "lactation_number"
    ].isna().sum(),
)

Building final lactation lookup from all known records...
Final lookup signatures: 1291550

FINAL LACTATION RESULTS
Strict predictions retained: 85236
Additional broad predictions: 64305
Total lactation predictions: 149541
Total coverage: 15.91 %
Lactation numbers still missing: 790208


In [72]:
# Part 40: Save the updated final dataset

from pathlib import Path

if "output_directory" not in globals():
    if Path.cwd().name == "notebooks":
        output_directory = Path("../data/processed")
    else:
        output_directory = Path("data/processed")

output_directory.mkdir(parents=True, exist_ok=True)

# Remove redundant temporary prediction columns
temporary_columns = [
    "days_in_milk_predicted",
    "days_in_milk_prediction_accepted",
    "reproduction_status_predicted",
    "lactation_number_predicted",
    "lactation_number_prediction_accepted",
]

final_export = final_cleaned_data.drop(
    columns=[
        column
        for column in temporary_columns
        if column in final_cleaned_data.columns
    ]
).copy()

# Use a new filename so the earlier export remains recoverable
final_output_path_v2 = (
    output_directory
    / "dairy_data_imputed_final_v2.csv"
)

print("Saving updated dataset...")
print("Rows:", len(final_export))

final_export.to_csv(
    final_output_path_v2,
    index=False,
)

print("\nUPDATED FINAL DATASET SAVED")
print("Rows:", len(final_export))
print("Columns:", len(final_export.columns))
print("Path:", final_output_path_v2.resolve())

# Show remaining missing values in every affected field
important_columns = [
    column
    for column in [
        "cow_id",
        "avg_milk_flow",
        "lactation_number",
        "days_in_milk",
        "reproduction_status",
    ]
    if column in final_export.columns
]

print("\nREMAINING MISSING VALUES")
print(
    final_export[
        important_columns
    ].isna().sum()
)

Saving updated dataset...
Rows: 4688514

UPDATED FINAL DATASET SAVED
Rows: 4688514
Columns: 23
Path: /Users/finmaroney/Desktop/project-1-dairy-missing-data/data/processed/dairy_data_imputed_final_v2.csv

REMAINING MISSING VALUES
cow_id                 939749
avg_milk_flow             100
lactation_number       790208
days_in_milk           917334
reproduction_status         0
dtype: int64
